In [1]:
import itertools
import math

import cirq
import cirq_google
import numpy as np

# Google Quantum AI Challenge Problem

Your code may import from cirq, cirq_google, their dependencies, and python's standard library. Other imports are not allowed.

**Please upload your solutions to [this form](https://docs.google.com/forms/d/e/1FAIpQLScw93lwRb3OihfaHgpobs8TO6nXijwvkRVYjWPpNg7n92FjIA/viewform?usp=publish-editor).** See the form for further instructions.

Consider an $N$-qubit, weight $n$ Dicke state, defined by
$$|D^{(N)}_n\rangle = \begin{pmatrix} N \\ n \end{pmatrix}^{-1/2} \sum_k \mathcal{P}_k \left( |0\rangle^{\otimes(N-n)} \otimes |1\rangle^{\otimes n}\right),$$
where the sum is over all distinct permutations $\mathcal{P}_k$. For $n=1$, this is the $W$ state, given by
$$|W\rangle = \frac{1}{\sqrt N} \left( |100\ldots 0 \rangle + |010\ldots 0 \rangle + \ldots + |00\ldots 01 \rangle \right). $$

Your task is to complete the following function

In [2]:
"""Self-contained Dicke submission combining three validated constructions.

Selection: ours for product states and N<=3; Richard for 4<=N<=8;
Aanha for N>=9. Each branch uses the supplied calibration. If the selected
branch cannot build a valid circuit, our hub-walk construction is a fallback.
Generated by build_combined.py; edit the source modules and rebuild instead
of editing the embedded implementations by hand.
"""

import cirq

# ours source SHA256: c29737e660cb2daeeefc2b08d73fcbb50c2974aaef07d70fb644744ec42fa817
def _load_ours():
    __name__ = '_embedded_ours'
    """Exact Dicke-state preparation for the DuQuantum Google challenge.

    The split-and-cyclic-shift construction follows Bärtschi and Eidenbenz,
    "Deterministic Preparation of Dicke States" (arXiv:1904.07358).

    Improvements over the original baseline (each one tested in improved/results):
      1. Hand-written CNOT decompositions of the controlled and doubly-controlled
         Ry rotations, instead of Cirq's generic decomposition.
      2. "Hub walk" layout: the circuit is emitted straight onto grid qubits, with
         SWAPs merged into neighbouring gates and fresh |0> qubits taken from any
         free neighbour, instead of a generic router on a single line.
      3. Several placements are compiled and the one with the highest estimated
         error-free probability (from the calibration data) is kept, within a time
         budget. Failed candidates are skipped instead of crashing.
      4. Dynamical decoupling pulses in idle single-qubit slots.
    The original line-routing construction remains as a fallback.
    """

    import argparse
    import math
    import time

    import cirq
    import cirq_google
    import networkx as nx
    import numpy as np


    def _rate(value: tuple[float, float]) -> float:
        """Use one reported standard uncertainty as a conservative placement cost."""
        return float(value[0]) + float(value[1])


    def _pick_path(
        length: int,
        weight: int,
        cz_pauli_error_rate: dict,
        sq_pauli_error_rate: dict,
        zero_errors: dict,
        one_errors: dict,
    ) -> list[cirq.GridQubit]:
        """Find a low-error simple path in the supplied working-coupler graph."""
        return _candidate_paths(
            length,
            weight,
            cz_pauli_error_rate,
            sq_pauli_error_rate,
            zero_errors,
            one_errors,
        )[0]


    def _candidate_paths(
        length: int,
        weight: int,
        cz_pauli_error_rate: dict,
        sq_pauli_error_rate: dict,
        zero_errors: dict,
        one_errors: dict,
    ) -> list[list[cirq.GridQubit]]:
        """Return beam-search placements ordered by a simple calibration cost."""
        available = set(sq_pauli_error_rate) & set(zero_errors) & set(one_errors)
        neighbors = {q: set() for q in available}
        edge_cost = {}
        for (a, b), rate in cz_pauli_error_rate.items():
            if a in available and b in available:
                neighbors[a].add(b)
                neighbors[b].add(a)
                edge_cost[frozenset((a, b))] = _rate(rate)

        if len(available) < length:
            raise NotImplementedError("Too few working qubits")

        density = weight / length
        node_cost = {
            q: 0.5 * _rate(sq_pauli_error_rate[q])
            + (1 - density) * _rate(zero_errors[q])
            + density * _rate(one_errors[q])
            for q in available
        }
        # Both end extensions are tried. The beam bounds construction time, while
        # keeping several paths avoids committing to a locally attractive dead end.
        beam = [(node_cost[q], (q,)) for q in available]
        beam.sort(key=lambda item: (item[0], item[1]))
        beam = beam[:256]
        for _ in range(1, length):
            candidates = {}
            for score, path in beam:
                used = set(path)
                for end, prepend in ((path[0], True), (path[-1], False)):
                    for q in neighbors[end] - used:
                        new_path = (q,) + path if prepend else path + (q,)
                        new_score = score + node_cost[q] + edge_cost[frozenset((end, q))]
                        previous = candidates.get(new_path)
                        if previous is None or new_score < previous:
                            candidates[new_path] = new_score
            if not candidates:
                raise NotImplementedError("No connected path of requested size")
            beam = sorted(
                ((score, path) for path, score in candidates.items()),
                key=lambda item: (item[0], item[1]),
            )[:256]
        # A path and its reverse have the same graph and placement score.
        distinct = []
        seen = set()
        for _, path in beam:
            canonical = min(path, path[::-1])
            if canonical not in seen:
                distinct.append(list(path))
                seen.add(canonical)
        return distinct


    def _controlled_ry(angle: float, control, target) -> list[cirq.Operation]:
        """Controlled-Ry with two CNOTs (instead of Cirq's generic decomposition)."""
        return [
            cirq.ry(angle / 2)(target),
            cirq.CNOT(control, target),
            cirq.ry(-angle / 2)(target),
            cirq.CNOT(control, target),
        ]


    def _doubly_controlled_ry(angle: float, control1, control2, target) -> list[cirq.Operation]:
        """Doubly-controlled Ry as a uniformly-controlled rotation: four CNOTs.

        Only Ry rotations and CNOTs appear, so the gate stays real and needs no
        Toffoli. Angles alternate +-angle/4, which leaves the target untouched unless
        both controls are 1.
        """
        quarter = angle / 4
        return [
            cirq.ry(quarter)(target),
            cirq.CNOT(control1, target),
            cirq.ry(-quarter)(target),
            cirq.CNOT(control2, target),
            cirq.ry(quarter)(target),
            cirq.CNOT(control1, target),
            cirq.ry(-quarter)(target),
            cirq.CNOT(control2, target),
        ]


    def _logical_dicke_circuit(num_qubits: int, num_excitations: int) -> cirq.Circuit:
        """Prepare a Dicke state on a line before routing and gate compilation."""
        q = cirq.LineQubit.range(num_qubits)
        k = min(num_excitations, num_qubits - num_excitations)
        if k == 0:
            return cirq.Circuit(cirq.X.on_each(*q)) if num_excitations else cirq.Circuit()
        circuit = cirq.Circuit(cirq.X(q[i]) for i in range(num_qubits - k, num_qubits))

        # U_{N,k} = U_{N-1,k} SCS_{N,k}, recursively. Each SCS consists of one
        # two-qubit rotation and controlled three-qubit rotations.
        for m in range(num_qubits, 1, -1):
            a, b = q[m - 2], q[m - 1]
            angle = 2 * math.acos(math.sqrt(1 / m))
            circuit.append(
                [cirq.CNOT(a, b), *_controlled_ry(angle, b, a), cirq.CNOT(a, b)]
            )
            for ell in range(2, min(k, m - 1) + 1):
                a, b, c = q[m - ell - 1], q[m - ell], q[m - 1]
                angle = 2 * math.acos(math.sqrt(ell / m))
                circuit.append(
                    [cirq.CNOT(a, c), *_doubly_controlled_ry(angle, b, c, a), cirq.CNOT(a, c)]
                )

        if num_excitations > num_qubits / 2:
            circuit.append(cirq.X.on_each(*q))
        return circuit


    # ---------------------------------------------------------------------------
    # Fix 2: layout-aware "hub walk" emission.
    #
    # In SCS_m the top qubit (the hub) interacts with every window qubit in turn.
    # Instead of handing this to a generic router, the hub walks down the window
    # path: after each block it swaps with the qubit it just used. The SWAP acts on
    # the same pair as the block's last two-qubit gates, so the gate optimizer
    # merges it in at almost no cost. Untouched qubits are all |0>, so the "fresh"
    # qubit each SCS needs is taken lazily from any free grid neighbour of the
    # hub, and the finished hub is parked in that neighbour's place.
    # ---------------------------------------------------------------------------


    def _block_i(a, hub, angle) -> list[cirq.Operation]:
        """Two-qubit SCS rotation (Bärtschi-Eidenbenz gate (i))."""
        return [cirq.CNOT(a, hub), *_controlled_ry(angle, hub, a), cirq.CNOT(a, hub)]


    def _block_ii_hub_middle(a, b, hub, angle) -> list[cirq.Operation]:
        """Three-qubit SCS rotation (gate (ii)) written so the hub is the target.

        Equivalent to CNOT(a,hub) CCRy_{b,hub->a}(angle) CNOT(a,hub), but every
        two-qubit gate touches the hub, so the hub can sit between a and b on a
        path. The block ends with two gates on (a, hub), which lets a following
        SWAP(a, hub) merge into them.
        """
        return [
            cirq.CNOT(hub, a),
            *_doubly_controlled_ry(-angle, b, a, hub),
            cirq.CNOT(hub, a),
        ]


    class _Layout:
        """Calibration-derived graph and costs for the working qubits."""

        def __init__(self, num_qubits, num_excitations, cz, sq, z0, z1):
            self.available = set(sq) & set(z0) & set(z1)
            self.neighbors = {q: set() for q in self.available}
            self.edge_cost = {}
            for (a, b), rate in cz.items():
                if a in self.available and b in self.available:
                    self.neighbors[a].add(b)
                    self.neighbors[b].add(a)
                    self.edge_cost[frozenset((a, b))] = _rate(rate)
            density = num_excitations / num_qubits
            self.node_cost = {
                q: 0.5 * _rate(sq[q]) + (1 - density) * _rate(z0[q]) + density * _rate(z1[q])
                for q in self.available
            }

        def edge(self, a, b):
            return self.edge_cost[frozenset((a, b))]


    class _HubWalk:
        """Emit the Bärtschi-Eidenbenz circuit directly onto physical qubits."""

        def __init__(self, layout: _Layout, path: list):
            self.layout = layout
            self.path = list(path)  # physical window positions, bottom -> top
            self.free = set(layout.available) - set(path)
            self.finished = set()
            self.ops: list[cirq.Operation] = []

        def _recruit(self, anchor):
            """Return a free (|0>) neighbour of `anchor`, freeing one if needed."""
            layout = self.layout
            options = layout.neighbors[anchor] & self.free
            if options:
                choice = min(options, key=lambda q: (layout.edge(anchor, q) + layout.node_cost[q], q))
                self.free.discard(choice)
                return choice
            # No free neighbour: push a finished qubit outward. Breadth-first search
            # through finished qubits to the nearest free qubit, then shift the
            # finished states one step out along that chain. Each shift moves a state
            # into a |0> qubit, which needs only two CNOTs.
            parent = {anchor: None}
            frontier = [anchor]
            found = None
            while frontier and found is None:
                nxt = []
                for q in sorted(frontier):
                    for nb in sorted(layout.neighbors[q]):
                        if nb in parent:
                            continue
                        if q != anchor and q not in self.finished:
                            continue
                        if nb in self.free:
                            parent[nb] = q
                            found = nb
                            break
                        if nb in self.finished:
                            parent[nb] = q
                            nxt.append(nb)
                    if found is not None:
                        break
                frontier = nxt
            if found is None:
                raise NotImplementedError("No free qubit reachable for recruitment")
            zero = found
            holder = parent[zero]
            self.free.discard(zero)
            while holder is not anchor:
                # Move the finished state from `holder` into |0> qubit `zero`.
                self.ops += [cirq.CNOT(holder, zero), cirq.CNOT(zero, holder)]
                self.finished.add(zero)
                self.finished.discard(holder)
                zero, holder = holder, parent[holder]
            return zero

        def build(self, num_qubits: int, k: int) -> tuple[list, set]:
            path = self.path
            ops = self.ops
            ops.append(cirq.X.on_each(*path))  # the k top qubits start in |1>
            # Phase A: SCS_m for m = N .. k+1 each recruit one fresh qubit.
            for m in range(num_qubits, k, -1):
                if k == 1:
                    hub = path[0]
                    fresh = self._recruit(hub)
                    ops += _block_i(fresh, hub, 2 * math.acos(math.sqrt(1 / m)))
                    self.finished.add(hub)
                    path[0] = fresh
                    continue
                j = k - 1
                ops += _block_i(path[j - 1], path[j], 2 * math.acos(math.sqrt(1 / m)))
                ops.append(cirq.SWAP(path[j - 1], path[j]))
                j -= 1
                for ell in range(2, k):
                    angle = 2 * math.acos(math.sqrt(ell / m))
                    ops += _block_ii_hub_middle(path[j - 1], path[j + 1], path[j], angle)
                    ops.append(cirq.SWAP(path[j - 1], path[j]))
                    j -= 1
                # Last block: the hub (now at the bottom) recruits a fresh qubit.
                fresh = self._recruit(path[0])
                angle = 2 * math.acos(math.sqrt(k / m))
                ops += _block_ii_hub_middle(fresh, path[1], path[0], angle)
                ops.append(cirq.SWAP(fresh, path[0]))
                self.finished.add(fresh)  # the finished hub now lives here
            # Phase B: SCS_m for m = k .. 2 use only window qubits. Each finished hub
            # is swapped to the bottom end, so the remaining window stays a path.
            top = len(path) - 1
            for m in range(k, 1, -1):
                j = top
                ops += _block_i(path[j - 1], path[j], 2 * math.acos(math.sqrt(1 / m)))
                if m - 1 >= 2:
                    ops.append(cirq.SWAP(path[j - 1], path[j]))
                    j -= 1
                for ell in range(2, m):
                    angle = 2 * math.acos(math.sqrt(ell / m))
                    ops += _block_ii_hub_middle(path[j - 1], path[j + 1], path[j], angle)
                    ops.append(cirq.SWAP(path[j - 1], path[j]))
                    j -= 1
            qubits = set(path) | self.finished
            return ops, qubits


    def _window_path(layout: _Layout, start, length: int):
        """Greedy cheapest simple path of `length` qubits beginning at `start`."""
        path = [start]
        while len(path) < length:
            options = layout.neighbors[path[-1]] - set(path)
            if not options:
                return None
            path.append(min(options, key=lambda q: (layout.edge(path[-1], q) + layout.node_cost[q], q)))
        return path


    def _start_candidates(layout: _Layout, k: int) -> list:
        """Rank window placements: cheap window path, cheap free neighbours at the bottom."""
        scored = []
        for start in layout.available:
            path = _window_path(layout, start, k)
            if path is None:
                continue
            cost = sum(layout.node_cost[q] for q in path)
            cost += sum(layout.edge(a, b) for a, b in zip(path, path[1:]))
            spare = sorted(
                layout.edge(start, q) + layout.node_cost[q]
                for q in layout.neighbors[start] - set(path)
            )
            if not spare:
                continue
            cost += sum(spare[:3]) / len(spare[:3])
            scored.append((cost, start, path))
        scored.sort(key=lambda item: (item[0], item[1]))
        return [path for _, _, path in scored]


    def _hub_walk_circuit(num_qubits, num_excitations, layout, path):
        k = min(num_excitations, num_qubits - num_excitations)
        walker = _HubWalk(layout, path)
        ops, qubits = walker.build(num_qubits, k)
        circuit = cirq.Circuit(ops)
        if num_excitations > num_qubits / 2:
            circuit.append(cirq.X.on_each(*sorted(qubits)))
        gateset = cirq.CZTargetGateset(allow_partial_czs=False)
        circuit = cirq.optimize_for_target_gateset(circuit, gateset=gateset)
        return circuit, qubits


    def _with_dynamical_decoupling(circuit: cirq.Circuit, cz_pauli_error_rate: dict) -> cirq.Circuit:
        """Fix 4: echo pulses on idle qubits.

        For every stretch where a qubit waits (between two of its gates, or after
        its last gate until the end), two X pulses go at about 1/4 and 3/4 of the
        wait. X X is the identity, so the state is unchanged, but slow phase drift
        accumulated before the first pulse is undone after it. Pulses go only into
        moments that hold single-qubit gates, so none runs alongside a CZ.

        (Cirq's add_dynamical_decoupling does the same job but took over 10 s on
        large circuits, too close to the one-minute limit.)
        If anything goes wrong the circuit is returned unchanged.
        """
        if not USE_DYNAMICAL_DECOUPLING:
            return circuit
        try:
            result = _echo_idle_gaps(circuit)
            _check_circuit(result, cz_pauli_error_rate)
            return result
        except Exception:
            return circuit


    def _echo_idle_gaps(circuit: cirq.Circuit) -> cirq.Circuit:
        moments = [list(m.operations) for m in circuit]
        uses = {}
        for i, ops in enumerate(moments):
            for op in ops:
                for q in op.qubits:
                    uses.setdefault(q, []).append(i)
        single_only = [bool(ops) and all(len(op.qubits) == 1 for op in ops) for ops in moments]
        x_pulse = cirq.PhasedXZGate(x_exponent=1, z_exponent=0, axis_phase_exponent=0)
        end = len(moments)
        for q in sorted(uses):
            times = uses[q]
            for begin, finish in list(zip(times, times[1:])) + [(times[-1], end)]:
                slots = [i for i in range(begin + 1, finish) if single_only[i]]
                if len(slots) < 2:
                    continue
                wait = finish - begin
                first = min(slots, key=lambda i: (abs(i - (begin + wait / 4)), i))
                later = [i for i in slots if i > first]
                if not later:
                    continue
                second = min(later, key=lambda i: (abs(i - (begin + 3 * wait / 4)), i))
                moments[first].append(x_pulse.on(q))
                moments[second].append(x_pulse.on(q))
        return cirq.Circuit(cirq.Moment(ops) for ops in moments)


    def _check_circuit(circuit: cirq.Circuit, cz_pauli_error_rate: dict) -> None:
        """Raise NotImplementedError unless the circuit obeys every challenge rule."""
        if len(circuit) > 1750:
            raise NotImplementedError("Compiled circuit exceeds the 1750-moment limit")
        allowed_edges = {frozenset(pair) for pair in cz_pauli_error_rate}
        for operation in circuit.all_operations():
            gate = operation.gate
            if isinstance(gate, cirq.CZPowGate) and gate.exponent == 1:
                if frozenset(operation.qubits) not in allowed_edges:
                    raise NotImplementedError("Used an unavailable CZ coupler")
            elif not isinstance(gate, cirq.PhasedXZGate):
                raise NotImplementedError(f"Unexpected compiled operation: {operation}")
        if hasattr(cirq_google, "Willow105") and circuit.all_qubits() <= set(
            cirq_google.Willow105.metadata.qubit_set
        ):
            cirq_google.Willow105.validate_circuit(circuit)


    def _product_state_x_bias_score(qubits, zero_errors, one_errors) -> float:
        """Expected X-correlator penalty from asymmetric independent readout."""
        biases = [one_errors[q][0] - zero_errors[q][0] for q in qubits]
        symmetric = [1.0] + [0.0] * len(biases)
        for count, bias in enumerate(biases, start=1):
            for size in range(count, 0, -1):
                symmetric[size] += bias * symmetric[size - 1]
        return sum(
            (symmetric[size] / math.comb(len(biases), size)) ** 2
            for size in range(1, len(biases) + 1)
        )


    def create_dicke_state(
        num_qubits: int,
        num_excitations: int,
        cz_pauli_error_rate: dict[
            tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]
        ],
        sq_pauli_error_rate: dict[cirq.GridQubit, tuple[float, float]],
        zero_errors: dict[cirq.GridQubit, tuple[float, float]],
        one_errors: dict[cirq.GridQubit, tuple[float, float]],
    ) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
        """Return an exact Dicke-state circuit using only permitted Willow gates."""
        if not (num_qubits >= 1 and 0 <= num_excitations <= num_qubits):
            raise ValueError("Require N >= 1 and 0 <= n <= N")

        k = min(num_excitations, num_qubits - num_excitations)
        layout = _Layout(
            num_qubits, num_excitations, cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors
        )
        rates = (cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors)
        if k == 0:
            # The Z chi-square term is zero for weights 0 and N because there is
            # only one bitstring in either target subspace. Their ideal nonempty X
            # correlators are zero; asymmetric readout creates the leading bias.
            if len(layout.available) < num_qubits:
                raise NotImplementedError("Too few working qubits")
            readout_mean = {
                q: (zero_errors[q][0] + one_errors[q][0]) / 2
                for q in layout.available
            }
            ceiling = max(0.05, 2 * float(np.median(list(readout_mean.values()))))
            candidates = [q for q in layout.available if readout_mean[q] <= ceiling]
            if len(candidates) < num_qubits:
                candidates = list(layout.available)
            low_bias = set(sorted(
                candidates,
                key=lambda q: (
                    abs(one_errors[q][0] - zero_errors[q][0]),
                    readout_mean[q],
                    sq_pauli_error_rate[q][0],
                    q,
                ),
            )[:num_qubits])
            low_z_error = set(sorted(
                candidates, key=lambda q: (layout.node_cost[q], q)
            )[:num_qubits])
            qubits = min(
                (low_bias, low_z_error),
                key=lambda choice: (
                    _product_state_x_bias_score(choice, zero_errors, one_errors),
                    tuple(sorted(choice)),
                ),
            )
            score = _product_state_x_bias_score(qubits, zero_errors, one_errors)
            # Swap qubits when opposite-sign readout biases can cancel.
            for _ in range(min(num_qubits, 12)):
                best_score = score
                best_swap = None
                for outgoing in sorted(qubits):
                    for incoming in sorted(set(candidates) - qubits):
                        choice = (qubits - {outgoing}) | {incoming}
                        candidate_score = _product_state_x_bias_score(
                            choice, zero_errors, one_errors
                        )
                        if candidate_score < best_score - 1e-15:
                            best_score = candidate_score
                            best_swap = (outgoing, incoming)
                if best_swap is None:
                    break
                qubits.remove(best_swap[0])
                qubits.add(best_swap[1])
                score = best_score
            if len(qubits) < num_qubits:
                raise NotImplementedError("Too few working qubits")
            ops = cirq.X.on_each(*sorted(qubits)) if num_excitations else []
            circuit = cirq.optimize_for_target_gateset(
                cirq.Circuit(ops), gateset=cirq.CZTargetGateset(allow_partial_czs=False)
            )
            return circuit, qubits

        # Fix 3: compile several placements and keep the one with the highest
        # estimated chance of running error-free. A time budget keeps us well under
        # the one-minute limit.
        start = time.monotonic()
        deadline = start + _TIME_BUDGET_S
        best = None
        too_long = False
        slowest = 0.0
        for path in _start_candidates(layout, k)[:_MAX_CANDIDATES]:
            now = time.monotonic()
            # Stop when another candidate would likely overrun the budget.
            if best is not None and now + slowest > deadline:
                break
            if best is None and now - start > _HARD_STOP_S:
                break
            try:
                circuit, qubits = _hub_walk_circuit(num_qubits, num_excitations, layout, path)
                if len(circuit) > 1750:
                    # Other placements give the same depth: no point trying them.
                    too_long = True
                    break
                _check_circuit(circuit, cz_pauli_error_rate)
            except Exception:  # a failed candidate must never crash the submission
                continue
            finally:
                slowest = max(slowest, time.monotonic() - now)
            success = _estimated_success(circuit, qubits, num_excitations, rates)
            if best is None or success > best[0]:
                best = (success, circuit, qubits)
        if best is not None:
            return _with_dynamical_decoupling(best[1], cz_pauli_error_rate), best[2]
        if too_long:
            raise NotImplementedError("Circuit for this (N, n) exceeds the 1750-moment limit")

        # Fallback: generic routing along a single line (the original baseline).
        # It always needs more gates than the hub walk, so it only runs for small
        # cases where the hub walk failed for some other reason.
        for path in _candidate_paths(num_qubits, num_excitations, *rates)[:3]:
            if time.monotonic() - start > _FALLBACK_CUTOFF_S:
                break
            try:
                circuit = _compile_for_path(path, num_qubits, num_excitations, cz_pauli_error_rate)
                return _with_dynamical_decoupling(circuit, cz_pauli_error_rate), set(path)
            except Exception:
                continue
        raise NotImplementedError("No placement produced a valid circuit")


    _TIME_BUDGET_S = 15.0  # stop trying new placements after this
    _HARD_STOP_S = 30.0  # give up if no placement has worked by now
    _FALLBACK_CUTOFF_S = 20.0  # do not start the slow fallback router after this
    _MAX_CANDIDATES = 12
    # Echo pulses (fix 4). Our simulations cannot show whether they help on the real
    # chip; the challenge notebook says dynamical decoupling is important, so they
    # are on by default. Set to False to submit without them.
    USE_DYNAMICAL_DECOUPLING = True


    def _estimated_success(circuit, qubits, num_excitations, rates) -> float:
        """Probability that no gate or readout error occurs (from calibration data)."""
        cz, sq, z0, z1 = rates
        cz_by_pair = {frozenset(pair): value[0] for pair, value in cz.items()}
        log_p = 0.0
        for operation in circuit.all_operations():
            if len(operation.qubits) == 2:
                log_p += math.log1p(-cz_by_pair[frozenset(operation.qubits)])
            else:
                log_p += math.log1p(-sq[operation.qubits[0]][0])
        density = num_excitations / len(qubits)
        for q in qubits:
            log_p += math.log1p(-(density * z1[q][0] + (1 - density) * z0[q][0]))
        return math.exp(log_p)


    def _compile_for_path(
        path: list[cirq.GridQubit],
        num_qubits: int,
        num_excitations: int,
        cz_pauli_error_rate: dict,
    ) -> cirq.Circuit:
        """Compile the exact construction on a chosen connected physical path."""
        logical = _logical_dicke_circuit(num_qubits, num_excitations)
        gateset = cirq.CZTargetGateset(allow_partial_czs=False)
        compiled = cirq.optimize_for_target_gateset(logical, gateset=gateset)
        graph = nx.Graph(zip(path, path[1:]))
        graph.add_nodes_from(path)
        routed, _, _ = cirq.RouteCQC(graph).route_circuit(compiled)
        circuit = cirq.optimize_for_target_gateset(routed, gateset=gateset)
        _check_circuit(circuit, cz_pauli_error_rate)
        return circuit


    def _demo() -> None:
        """Run a local ideal-state check with illustrative, not live, error rates."""
        parser = argparse.ArgumentParser(description="Try the Dicke-state baseline locally")
        parser.add_argument("N", type=int, help="number of qubits")
        parser.add_argument("n", type=int, help="number of excitations")
        args = parser.parse_args()

        if not (1 <= args.N <= 12 and 0 <= args.n <= args.N):
            parser.error("demo supports 1 <= N <= 12 and 0 <= n <= N")

        if hasattr(cirq_google, "Willow105"):
            device = cirq_google.Willow105
            layout = "Willow105 static layout"
        else:
            # Public cirq-google 1.7.0 includes the Willow-pink device record.
            device = cirq_google.SimulatedProcessorWithLocalDeviceRecord(
                "willow_pink"
            ).get_device()
            layout = "Willow-pink reference layout"
        qubits = set(device.metadata.qubit_set)
        pairs = {tuple(pair) for pair in device.metadata.qubit_pairs}

        cz_rates = {pair: (0.0025, 0.0005) for pair in pairs}
        sq_rates = {q: (0.0004, 0.0002) for q in qubits}
        zero_rates = {q: (0.005, 0.0002) for q in qubits}
        one_rates = {q: (0.01, 0.0003) for q in qubits}
        circuit, target = create_dicke_state(
            args.N, args.n, cz_rates, sq_rates, zero_rates, one_rates
        )
        actual = cirq.Simulator(dtype=np.complex128).simulate(
            circuit, qubit_order=sorted(target)
        ).final_state_vector
        ideal = np.array(
            [
                1 / math.sqrt(math.comb(args.N, args.n))
                if basis.bit_count() == args.n
                else 0
                for basis in range(1 << args.N)
            ]
        )
        fidelity = abs(np.vdot(ideal, actual)) ** 2
        cz_count = sum(len(op.qubits) == 2 for op in circuit.all_operations())
        print(f"Layout: {layout} (illustrative error rates)")
        print(f"Dicke state: N={args.N}, n={args.n}")
        print(f"Moments: {len(circuit)}, CZ gates: {cz_count}")
        print(f"Ideal-state fidelity: {fidelity:.12f}")


    if __name__ == "__main__":
        _demo()
    return create_dicke_state

# richard source SHA256: c78a8a5afbeec9ae168fc628a9ddce13d96b635310b5e571e675e7242a1d2044
def _load_richard():
    __name__ = '_embedded_richard'
    """Dicke-state preparation on Google Willow (DuQuantum x Google Quantum AI, standard track).

    Only cirq, cirq_google, their dependencies (numpy, networkx) and the standard library are
    imported, as the challenge requires.

    Construction (baseline): Baertschi & Eidenbenz, "Deterministic Preparation of Dicke
    States" (arXiv:1904.07358). We build the logical circuit, place it on a low-error path of
    calibrated qubits, route it with cirq.RouteCQC, and compile to CZ + PhasedXZ. The Dicke
    state is permutation-symmetric, so wherever routing leaves the logical qubits is a valid
    answer; we just return that set.
    """

    import math
    import time

    import cirq
    import networkx as nx
    import numpy as np


    # Only attempt the second routing variant if the first finished within this many seconds,
    # so the 60 s build limit holds with a wide margin on slower machines.
    _SECOND_TRY_BUDGET_S = 10.0


    # ---- calibration graph ----------------------------------------------------------------


    def _robust(v) -> float:
        """Reported error rate plus one standard uncertainty, used for every placement decision.

        Picking the lowest reported rates favours qubits that only look good by chance (their
        true error regresses toward the median). Ranking by value + 1 sigma hedges that; with
        the notebook's uniform uncertainties it shifts all costs equally and changes nothing.
        Accepts (value, sigma) or a bare value.
        """
        try:
            return float(v[0]) + max(float(v[1]), 0.0)
        except (TypeError, IndexError):
            return float(v)


    def _usable_graph(cz_pauli_error_rate: dict, sq_pauli_error_rate: dict) -> nx.Graph:
        """Calibrated couplers whose endpoints both have single-qubit calibrations."""
        g = nx.Graph()
        for pair, v in cz_pauli_error_rate.items():
            a, b = tuple(pair)  # keys are tuple(frozenset): order is arbitrary
            if a in sq_pauli_error_rate and b in sq_pauli_error_rate:
                g.add_edge(a, b, err=_robust(v))
        return g


    def _qubit_cost(q, sq_pauli_error_rate, zero_errors, one_errors, p_one: float, n_sq: float = 1) -> float:
        """Expected per-qubit error: readout weighted by how often the qubit reads 1, plus
        `n_sq` single-qubit gates."""
        e0 = _robust(zero_errors.get(q, (0.5, 0)))
        e1 = _robust(one_errors.get(q, (0.5, 0)))
        return (1 - p_one) * e0 + p_one * e1 + n_sq * _robust(sq_pauli_error_rate[q])


    def _best_path(g: nx.Graph, length: int, node_cost: dict, beam: int = 256,
                   edge_weight: float = 1.0, keep: int = 1):
        """Beam search for a simple path of `length` qubits minimizing
        edge_weight * (CZ error) + node cost. Returns the best path, or the best `keep`."""
        frontier = sorted(((node_cost[q], [q]) for q in g.nodes), key=lambda t: t[0])[:beam]
        for _ in range(length - 1):
            grown = []
            seen = set()
            for cost, path in frontier:
                used = set(path)
                for nb in g.neighbors(path[-1]):
                    if nb in used:
                        continue
                    key = (frozenset(used | {nb}), nb)
                    if key in seen:
                        continue
                    seen.add(key)
                    step = edge_weight * g.edges[path[-1], nb]["err"] + node_cost[nb]
                    grown.append((cost + step, path + [nb]))
            if not grown:
                return None
            grown.sort(key=lambda t: t[0])
            frontier = grown[:beam]
        return frontier[0][1] if keep == 1 else frontier[:keep]


    def _connected_patch(g: nx.Graph, size: int, node_cost: dict) -> list | None:
        """Greedy connected set of `size` qubits, grown from the cheapest qubit outward."""
        for comp in sorted(nx.connected_components(g), key=len, reverse=True):
            if len(comp) < size:
                return None
            start = min(comp, key=node_cost.get)
            chosen, frontier = [start], set(g.neighbors(start))
            while len(chosen) < size:
                nxt = min(frontier, key=lambda q: node_cost[q])
                chosen.append(nxt)
                frontier |= set(g.neighbors(nxt))
                frontier -= set(chosen)
            return chosen
        return None


    # ---- W state (n = 1): broadcast tree of amplitude splits ------------------------------

    # Idle-error model for layout decisions. The calibration data has no T1/T2, so we use
    # the Willow spec-sheet T1 (68 us) and an assumed T_phi (100 us). One tree round is
    # 2 CZ layers (42 ns) + 2 single-qubit layers (25 ns). A W state carries exactly one
    # excitation in every branch, so its idle loss per round is independent of N.
    _ROUND_NS = 2 * 42 + 2 * 25
    _IDLE_RATE_PER_NS = 1 / 68e3 + 1 / 100e3


    def _readout_lambda(qubits: list, zero_errors, one_errors, n: int, f: float) -> float:
        """Chi^2 noncentrality from readout alone: a valid string x survives readout with
        relative weight ~ exp(-sum_{i in x} (e1_i - e0_i)), so its spread over random
        n-subsets is n (N-n)/(N-1) var(e1 - e0)."""
        N = len(qubits)
        if N < 2:
            return 0.0
        d = np.array([one_errors.get(q, (0.5, 0))[0] - zero_errors.get(q, (0.5, 0))[0] for q in qubits])
        return 1e4 * f * n * (N - n) / (N - 1) * float(np.var(d))


    def _grow_tree(g: nx.Graph, root, N: int, node_cost: dict):
        """Each round, every qubit holding amplitude may split it onto one fresh neighbour.
        Greedy: take the cheapest (edge + new-qubit) splits first. Returns (rounds, cost)
        where rounds is a list of [(parent, child), ...]."""
        nodes, used, rounds, cost = [root], {root}, [], node_cost[root]
        while len(nodes) < N:
            candidates = sorted(
                (2 * g.edges[p, c]["err"] + node_cost[c], p, c)
                for p in nodes for c in g.neighbors(p) if c not in used
            )
            busy, splits = set(), []
            for c_cost, p, c in candidates:
                if p in busy or c in busy:
                    continue
                splits.append((p, c))
                busy |= {p, c}
                used.add(c)
                cost += c_cost
                if len(nodes) + len(splits) == N:
                    break
            if not splits:
                return None, math.inf
            nodes += [c for _, c in splits]
            rounds.append(splits)
        cost -= g.edges[rounds[0][0]]["err"] if rounds else 0  # the first split is 1 CZ
        return rounds, cost


    def _cheap_connected_sets(g: nx.Graph, N: int, node_cost: dict, cz_per_edge: float, keep: int):
        """Beam search over connected qubit sets; cost = node costs + cheapest attaching edge."""
        beam = max(16, 4000 // N)
        frontier = sorted(((node_cost[q], frozenset([q])) for q in g.nodes), key=lambda t: t[0])[:beam]
        for _ in range(N - 1):
            grown = {}
            for cost, S in frontier:
                for v in {u for s in S for u in g.neighbors(s)} - S:
                    e = min(g.edges[v, u]["err"] for u in g.neighbors(v) if u in S)
                    c2 = cost + node_cost[v] + cz_per_edge * e
                    S2 = S | {v}
                    if c2 < grown.get(S2, math.inf):
                        grown[S2] = c2
            if not grown:
                return []
            frontier = sorted(((c, S) for S, c in grown.items()), key=lambda t: t[0])[:beam]
        return [S for _, S in frontier[:keep]]


    def _best_w_tree(g, N, node_cost, zero_errors, one_errors):
        """Pick a cheap qubit set first, then the best broadcast tree inside it."""
        best, best_cost = None, math.inf
        for S in _cheap_connected_sets(g, N, node_cost, cz_per_edge=2, keep=8):
            sub = g.subgraph(S)
            roots = sorted(S, key=lambda q: max(nx.single_source_shortest_path_length(sub, q).values()))
            for root in roots[:12]:
                rounds, cost = _grow_tree(sub, root, N, node_cost)
                if rounds is None:
                    continue
                cost += len(rounds) * _ROUND_NS * _IDLE_RATE_PER_NS
                lam = _readout_lambda(list(S), zero_errors, one_errors, 1, math.exp(-cost))
                cost += math.log1p(lam / (N - 1))
                if cost < best_cost:
                    best, best_cost = (root, rounds), cost
        return best


    def _w_tree_circuit(root, rounds: list) -> cirq.Circuit:
        """Amplitude starts on the root; a split hands a child the share of its subtree."""
        children = {}
        for r in rounds:
            for p, c in r:
                children.setdefault(p, []).append(c)

        size = {}

        def subtree(v):
            size[v] = 1 + sum(subtree(c) for c in children.get(v, []))
            return size[v]

        remaining = {root: subtree(root)}  # amplitude^2 (in units of 1/N) still held by v
        moments = []
        for i, r in enumerate(rounds):
            ops = []
            for p, c in r:
                theta = math.asin(math.sqrt(size[c] / remaining[p]))
                remaining[p] -= size[c]
                remaining[c] = size[c]
                if i == 0:  # product input: Ry, CNOT, X gives cos|10> + sin|01> with 1 CZ
                    ops += [cirq.ry(2 * theta).on(c), cirq.CNOT(c, p), cirq.X(p)]
                else:  # cirq.givens(-t): |10> -> cos t |10> + sin t |01>
                    ops.append(cirq.givens(-theta).on(p, c))
            moments.append(cirq.Circuit(ops))
        return sum(moments, cirq.Circuit())


    # ---- tiny W states (K <= 3): choose qubits by predicted expected score --------------------
    #
    # For K <= 3 the judged score 1/(z + x) is so heavy-tailed that its expectation is set by
    # the X-term floor x and the imbalance lambda, not by CZ count. With at most 3 qubits we can
    # afford to simulate every placement exactly (gate depolarizing + readout, from the
    # calibration dicts) and pick the one with the highest predicted E[score].

    _PAULIS = [np.array(m, dtype=complex) for m in ([[0, 1], [1, 0]], [[0, -1j], [1j, 0]], [[1, 0], [0, -1]])]


    def _embed(mats: dict, N: int) -> np.ndarray:
        """Kronecker product placing 2x2 `mats[i]` on logical qubit i (qubit 0 most significant)."""
        out = np.array([[1.0 + 0j]])
        for i in range(N):
            out = np.kron(out, mats.get(i, np.eye(2)))
        return out


    def _tiny_program(N: int, rounds: list, flip: bool):
        """Compile one tree shape on LineQubits once; return it plus a noisy-simulation program."""
        lq = cirq.LineQubit.range(N)
        logical = _w_tree_circuit(lq[rounds[0][0][0]], [[(lq[p], lq[c]) for p, c in r] for r in rounds])
        if flip:
            logical.append(cirq.X.on_each(*lq))
        compiled = _to_cz_phxz(logical)
        steps = []
        for op in compiled.all_operations():
            idx = tuple(q.x for q in op.qubits)
            u = cirq.Circuit(op).unitary(qubit_order=lq)
            if op.gate == cirq.CZ:
                paulis = [_embed({idx[0]: a, idx[1]: b}, N) for a in [np.eye(2)] + _PAULIS
                          for b in [np.eye(2)] + _PAULIS][1:]
                steps.append((u, "cz", idx, paulis))
            elif not _is_virtual_z(op):
                steps.append((u, "sq", idx, [_embed({idx[0]: p}, N) for p in _PAULIS]))
            else:
                steps.append((u, None, idx, []))
        return compiled, lq, steps


    def _expected_score(K: int, f: float, lam: float, x: float, exact: bool) -> float:
        """E[1/(z + x)] with z = chi^2/(K f), chi^2 ~ noncentral chi^2(K-1, lam).

        exact=False uses closed forms for lam = 0 (fast ranking); exact=True integrates."""
        from scipy import integrate, special, stats

        a = K * f * x  # E[score] = K f E[1/(chi^2 + a)]
        if not exact:
            if K == 2:
                return K * f * math.sqrt(math.pi / (2 * a)) * special.erfcx(math.sqrt(a / 2))
            return K * f * 0.5 * math.exp(a / 2) * special.exp1(a / 2)
        dist = stats.ncx2(K - 1, lam) if lam > 1e-12 else stats.chi2(K - 1)
        t_max = math.sqrt(dist.isf(1e-14))
        val, _ = integrate.quad(lambda t: dist.pdf(t * t) * 2 * t / (t * t + a), 0, t_max,
                                points=[min(math.sqrt(a), t_max / 2)], limit=500)
        return K * f * val


    def _predict_tiny(steps, phys, n, cz_err, sq_err, zero_errors, one_errors, exact, shots=10_000):
        N = len(phys)
        rho = np.zeros((2**N, 2**N), dtype=complex)
        rho[0, 0] = 1
        for u, kind, idx, paulis in steps:
            rho = u @ rho @ u.conj().T
            if kind is None:
                continue
            p = cz_err[frozenset(phys[i] for i in idx)] if kind == "cz" else sq_err[phys[idx[0]]]
            rho = (1 - p) * rho + (p / len(paulis)) * sum(P @ rho @ P for P in paulis)
        h = _embed({i: np.array([[1, 1], [1, -1]]) / math.sqrt(2) for i in range(N)}, N)
        probs = []
        for r in (rho, h @ rho @ h):
            pr = np.real(np.diag(r)).reshape((2,) * N)
            for j, q in enumerate(phys):  # readout confusion, [read, true]
                e0, e1 = _robust(zero_errors.get(q, (0.5, 0))), _robust(one_errors.get(q, (0.5, 0)))
                conf = np.array([[1 - e0, e1], [e0, 1 - e1]])
                pr = np.moveaxis(np.tensordot(conf, pr, axes=([1], [j])), 0, j)
            probs.append(pr.reshape(-1))
        pz, px = probs
        weight = np.array([bin(i).count("1") for i in range(2**N)])
        pv = pz[weight == n]
        f = float(pv.sum())
        lam = shots * f * float(np.mean((pv / pv.mean() - 1) ** 2))
        qw = np.bincount(weight, weights=px, minlength=N + 1)
        x = 0.0
        for k in range(1, N + 1):  # mean X term, including its shot-noise floor
            v = np.array([sum((-1) ** j * math.comb(w, j) * math.comb(N - w, k - j) for j in range(k + 1))
                          for w in range(N + 1)]) / math.comb(N, k)
            m = float(qw @ v)
            var = float(qw @ v**2) - m * m
            ideal = (math.comb(k, k // 2) * math.comb(N - k, n - k // 2) / math.comb(N, n)
                     if k % 2 == 0 and 0 <= n - k // 2 <= N - k else 0.0)
            x += ((m - ideal) ** 2 + var / shots) / (ideal**2 if ideal else 1.0)
        return _expected_score(math.comb(N, n), f, lam, x, exact)


    def _tiny_candidates(g: nx.Graph, N: int):
        """(logical rounds, physical qubits by logical index) for every placement of N <= 3."""
        if N == 2:
            for a, b in g.edges:
                yield [[(0, 1)]], [a, b]
                yield [[(0, 1)]], [b, a]
            return
        for mid in g.nodes:  # N == 3: every path a - mid - c
            nbrs = sorted(g.neighbors(mid))
            for i, a in enumerate(nbrs):
                for c in nbrs[i + 1:]:
                    yield [[(0, 1)], [(0, 2)]], [mid, a, c]  # star from the middle, either order
                    yield [[(0, 1)], [(0, 2)]], [mid, c, a]
                    yield [[(0, 1)], [(1, 2)]], [a, mid, c]  # chain from either end
                    yield [[(0, 1)], [(1, 2)]], [c, mid, a]


    def _best_tiny_w(g, N, n, flip, cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors):
        cz_err = {frozenset(p): _robust(v) for p, v in cz_pauli_error_rate.items()}
        sq_err = {q: _robust(v) for q, v in sq_pauli_error_rate.items()}
        programs, scored = {}, []
        for rounds, phys in _tiny_candidates(g, N):
            key = str(rounds)
            if key not in programs:
                programs[key] = _tiny_program(N, rounds, flip)
            e = _predict_tiny(programs[key][2], phys, n, cz_err, sq_err, zero_errors, one_errors, exact=False)
            scored.append((e, key, phys))
        if not scored:
            return None
        scored.sort(key=lambda t: -t[0])
        best = max(scored[:10], key=lambda t: _predict_tiny(
            programs[t[1]][2], t[2], n, cz_err, sq_err, zero_errors, one_errors, exact=True))
        compiled, lq, _ = programs[best[1]]
        mapping = dict(zip(lq, best[2]))
        return compiled.transform_qubits(lambda q: mapping[q]), set(best[2])


    # ---- Baertschi-Eidenbenz logical circuit ------------------------------------------------


    def _ccry(phi: float, c1, c2, t) -> list:
        """Doubly-controlled Ry(phi) with 4 CNOTs: X Ry(a) X = Ry(-a) makes the Ry's cancel
        unless both controls are 1, in which case they add to Ry(4a) = Ry(phi)."""
        a = phi / 4
        return [
            cirq.ry(a).on(t), cirq.CNOT(c1, t),
            cirq.ry(-a).on(t), cirq.CNOT(c2, t),
            cirq.ry(a).on(t), cirq.CNOT(c1, t),
            cirq.ry(-a).on(t), cirq.CNOT(c2, t),
        ]


    def _scs(q: list, m: int, k: int) -> list:
        """Split & Cyclic Shift SCS_{m,k} on the 1-indexed logical qubits q[1..m] (paper 2.2)."""
        ops = []
        # gate (i) on (m-1, m)
        a, b = q[m - 2], q[m - 1]
        ops += [cirq.CNOT(a, b), cirq.ry(2 * math.acos(math.sqrt(1 / m))).on(a).controlled_by(b), cirq.CNOT(a, b)]
        # gates (ii)_l on (m-l, m-l+1, m)
        for l in range(2, k + 1):
            t, c = q[m - l - 1], q[m - l]
            ops += [cirq.CNOT(t, b)]
            ops += _ccry(2 * math.acos(math.sqrt(l / m)), b, c, t)
            ops += [cirq.CNOT(t, b)]
        return ops


    def _bartschi_eidenbenz(N: int, k: int) -> tuple[cirq.Circuit, list]:
        """U_{N,k} |0^{N-k} 1^k> = |D^N_k> on LineQubits."""
        q = cirq.LineQubit.range(N)
        ops = [cirq.X(q[i]) for i in range(N - k, N)]
        for m in range(N, k, -1):
            ops += _scs(q, m, k)
        for m in range(k, 1, -1):
            ops += _scs(q, m, m - 1)
        return cirq.Circuit(ops), q


    # ---- small cases: numerically synthesized exact circuits ----------------------------------
    #
    # For small (N, k) we embed exact Ry + CZ circuits found offline by experiments/synth.py
    # (gradient optimization to infidelity < 1e-13, then greedy CZ pruning). They depend only
    # on a layout shape (grid coordinates), never on calibration data; the calibration-dependent
    # part, where to put the shape, is decided below at build time. Synthesis itself takes
    # minutes per case, far over the 60 s build budget, which is why the result is cached here.

    _GRID_SYMMETRIES = (
        lambda r, c: (r, c), lambda r, c: (r, -c), lambda r, c: (-r, c), lambda r, c: (-r, -c),
        lambda r, c: (c, r), lambda r, c: (c, -r), lambda r, c: (-c, r), lambda r, c: (-c, -r),
    )


    def _place_synth(g, entry, n, sq_pauli_error_rate, zero_errors, one_errors):
        """Best placement of a synthesized shape on calibrated qubits and couplers."""
        coords = entry["coords"]
        N = len(coords)
        th = np.array(entry["params"]).reshape(-1, N)
        ry_count = (np.abs(np.sin(th / 2)) > 1e-9).sum(axis=0)  # non-trivial rotations per qubit
        uses = {}
        for layer in entry["cz_layers"]:
            for e in layer:
                uses[tuple(e)] = uses.get(tuple(e), 0) + 1
        usable = set(g.nodes)
        best, best_cost = None, math.inf
        for sym in _GRID_SYMMETRIES:
            pts = [sym(r, c) for r, c in coords]
            r0, c0 = pts[0]
            for anchor in usable:
                qs = [cirq.GridQubit(anchor.row + r - r0, anchor.col + c - c0) for r, c in pts]
                if not all(q in usable for q in qs) or not all(g.has_edge(qs[i], qs[j]) for i, j in uses):
                    continue
                cost = sum(u * g.edges[qs[i], qs[j]]["err"] for (i, j), u in uses.items())
                cost += sum(
                    _qubit_cost(q, sq_pauli_error_rate, zero_errors, one_errors, n / N, n_sq=ry_count[i])
                    for i, q in enumerate(qs)
                )
                lam = _readout_lambda(qs, zero_errors, one_errors, n, math.exp(-cost))
                cost += math.log1p(lam / max(math.comb(N, n) - 1, 1))
                if cost < best_cost:
                    best, best_cost = qs, cost
        return best


    def _synth_circuit(entry, qs) -> cirq.Circuit:
        N = len(qs)
        th = np.array(entry["params"]).reshape(-1, N)
        moments = []
        for l in range(len(th)):
            moments.append(cirq.Moment(cirq.ry(th[l, i]).on(qs[i]) for i in range(N)))
            if l < len(entry["cz_layers"]):
                moments.append(cirq.Moment(cirq.CZ(qs[i], qs[j]) for i, j in entry["cz_layers"][l]))
        return cirq.Circuit(moments)


    # BEGIN SYNTH TABLE (generated by experiments/synth.py --export; do not edit)
    _SYNTH = {
        (4, 2): dict(  # 5 CZ, ladder, seed 0, robust pick 18
            coords=[(0, 0), (1, 0), (0, 1), (1, 1)],
            cz_layers=[[(2, 3)], [(0, 2), (1, 3)], [(0, 1), (2, 3)]],
            params=[1.011641720900928, 2.2789188858281637, -1.3076572937407227, 2.3795057930799706, 3.70074725852797, -0.7081225567239268, -1.6779636626810785, -0.8638085259288678, 1.7818695622763414e-09, 5.417358508585396e-10, 1.5707963256719608, -1.570796329932613, 1.232338693447e-09, -1.5707963271329526, -3.1415926551997733, 4.712388979190364],
        ),
        (5, 2): dict(  # 10 CZ, path, seed 0, robust pick 21
            coords=[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)],
            cz_layers=[[(0, 1), (2, 3)], [(1, 2)], [(2, 3)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1)]],
            params=[-1.4986414490951898, 0.6030549441059361, -1.694517047389452, 0.46750258492084046, -0.8868353061110757, 0.9118787801784288, 1.27621343166358, 1.0501511920838598, 0.038332717492985925, 4.127597365561725, -4.15047590148269, -0.37135563950150685, 0.9117382703191945, -1.0233121926372912, 4.207942601792098, -4.17328935685691, -0.5839610094147459, -2.6179938817221795, -0.46364762627110256, 0.4052769752256716, -2.8563973746921527, -0.6154797200760713, 7.31177774405034e-10, -1.5707963198525992, 2.468324523175826, 0.5451618742927558, -0.7853981653743284, 1.5707963246198553, -0.8492082787191461, 1.9359403955115857, 0.2402362793754006, 1.5707963323593293, -1.152017897434777, -3.531489246951883, 1.0362492938856247, -3.426946963739398e-09, 1.5707963252379087, 2.7228142396304356, -0.3316914655960666, -0.7281252343604439],
        ),
        (6, 2): dict(  # 18 CZ, path, seed 0, robust pick 19
            coords=[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5)],
            cz_layers=[[(0, 1), (2, 3)], [(1, 2), (3, 4)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)], [(3, 4)], [(0, 1), (2, 3)]],
            params=[1.273401929989079, -2.6629953412693688, -1.1939606465323103, 4.176766894705059, -1.7829991268996401, -1.0753845379217315, 0.3912099294907602, 1.2573895570428093, -0.9141970221022485, -0.7894541023186473, 0.2122028170155825, -2.988012893451692, -0.24130623722819666, -1.959929601761315, -0.45178167614039333, 1.2833688120374722, -1.2012224064022992e-07, -0.6489914531278403, 2.8314469771168516, 3.6471817311943973, 1.039931446636215, 0.2874275436518202, -4.092375402160591, -0.37280878090051606, 2.6832296672128617, 2.3571525770817834, -2.5534368387727473, -0.8210874460269298, 1.5707962714355483, 2.315713695205924, -0.17159733097131105, 2.3571527864420037, 1.5707963170140482, 2.820087997027582, -0.6679354206263197, 0.2915723909618664, 0.3459405490564516, -3.1886386324184737, -5.083238801751641e-06, -0.32502165430141255, -1.5707963269518088, 0.9194467459990205, 0.9131317817015843, 0.9289964056245835, 1.918517887380368, 2.4560572830361436e-08, 1.563735176654994, -1.4608111378352324, 0.48332093592281383, -2.452746759761478, 1.2250008480283063, -0.0019275463714309416, -2.6380347126710237, 3.2230017455792863, -1.296261642185438e-08, 1.5707963165289673, 4.712394058952568, -1.570796364013428, 1.067238622003698, 2.9501983416037136],
        ),
        (6, 3): dict(  # 15 CZ, ladder, seed 0, robust pick 11
            coords=[(0, 0), (1, 0), (0, 1), (1, 1), (0, 2), (1, 2)],
            cz_layers=[[(0, 1), (2, 3), (4, 5)], [(0, 2), (1, 3)], [(3, 5)], [(2, 3)], [(0, 2), (1, 3)], [(2, 4), (3, 5)], [(0, 1), (2, 3)], [(0, 2), (1, 3)]],
            params=[-1.6833327905971032, -3.9131196713627574, 4.341168673558379, -1.0678724970825328, 4.712388982885306, -4.712388975292479, 0.43163538346296787, 1.1809666058926551, 1.6346909844915862, 1.5114228726759555, 4.21962563962444, 1.3749485264472727, 1.547568542350625, -1.9420186765959466, -1.6806071783873842, 0.5235987841617172, 5.524967593479281, 0.22694055202490632, 5.791065438109848, 2.094040365347798, -2.0764651884616643, -3.141592647509924, 3.2689490957552283, 1.1696749274549658, 3.656940298418632, 1.418774641266783, -0.6154796992737697, 1.57079632908446, 3.906002231953504, -1.6094251340194683, 5.911045383838438, -0.09474504892640277, 3.9269907883821156, -4.712388982345858, 3.531900420964907, -2.701842427932343, 2.728334403932701, 1.665541375198194, -4.712388976756994, 3.584454232746003e-08, 2.747662515563989, -1.0099745059033434, 1.5707963223033918, -6.283185308515168, 6.283185304954134, 4.712388982499544, 1.3652406440829166, -1.9717065658794286, 1.570796316203271, -1.570796310237535, -8.492225472465768e-09, 6.283185295438324, 2.1702821826282004, -1.7307079337981268],
        ),
        (7, 2): dict(  # 20 CZ, path, seed 0, robust pick 4
            coords=[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6)],
            cz_layers=[[(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4), (5, 6)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (5, 6)], [(0, 1), (2, 3)], [(3, 4)], [(4, 5)], [(5, 6)]],
            params=[2.967492661493879, 0.8168458820013998, -2.003723848363806, 4.312730799132894, -0.9107507716748718, 1.4940346153324664, -0.6120009416648662, 2.8753747264975495, -3.1770972222804117, 0.3192704873557996, 1.879184760054684, -0.8494593213126443, -3.001719359170047, 2.534520093205146, -1.1304784574292615, 0.6154796831604005, -1.5126116121812235e-08, 4.100791818808925, -0.558419780783166, 0.9633819976315638, -0.014033797866221822, 4.210026020199953, 3.0821935805591766e-08, 2.180479957818448, 2.617993858138423, 2.617993857490405, -4.069420820163932, -2.5239650524795563, 2.0731592505063188, 1.5707962635903796, -2.526113022899635, -1.0471976366568048, 5.309838121475653e-08, 0.9553165610879187, -3.1336973621667483, 0.2117599888347925, -0.4560721879121676, 4.096909329643039, -0.6838504514887151, 1.8056762124565968, 0.6154796857319087, 2.1783807331212452, -0.3989773148753558, 2.3561944916708253, 1.5707963069052584, -2.45774225340186, 2.127827760458616, -2.2869103661449515, -0.41622352641856536, 0.849645597348783, -3.163548500474424, -2.306106083585746, 1.5707963243299767, 2.349681269715352, -3.21690816902943, 1.3569837256186912, -1.5875069698559126, -2.8644218675817177, 2.332200464511252, 1.7381127737396762, 1.570796341743864, -3.1355613212616857, 0.7262571333561135, -0.2573963970453183, -2.3705718732797414, 2.091469814931959, 2.289657494222605, -1.0684090142361817, -1.5707963285657756, 2.2599735129214156, 2.5660541740450897, -2.5970320611391657, 2.594824777677355, -2.4569739585348316, 2.6392053295406135, 1.570796301362198, -3.1415926732597668],
        ),
        (7, 3): dict(  # 25 CZ, path, seed 0, robust pick 13
            coords=[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6)],
            cz_layers=[[(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4), (5, 6)], [(2, 3), (4, 5)], [(1, 2), (3, 4), (5, 6)], [(0, 1)], [(1, 2), (3, 4), (5, 6)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(0, 1), (4, 5)]],
            params=[-0.3942314280927663, 1.4092262234909678, 2.5385912449389814, 0.8702059606520871, 0.6487688377703466, 1.2737463439107866, 1.9828598153297126, -1.8043094634563046, -0.10145935997514405, -0.6391434117148288, 2.6334610467336432, 1.3674001167779066, 1.1492499702137444, -2.8333352848561235, 1.0140502909364142, 4.851352548047241, 2.551639092085599, 2.6870542599761844, -1.531892087279405, -3.9327395798784464, -2.100602228503966, 3.497022069033054, -1.5533503001156914e-08, 1.976904916524363, -2.3547256087627524, 1.5707963305705859, 2.328667685775536, -1.7613112860988238, 3.613401877140848, 2.1091378833128034, -1.2401026586424566, 3.1415926927756277, 0.9553166020516288, 1.5707963234284212, -2.828336223425421, 2.8344549587996832, 1.0324547646279119, 0.5752675381520481, 5.144270899290797e-08, -1.5707963544151784, -0.9165071171804843, -0.31325641108912194, 1.6661751043135566, -1.078930635817034, 1.7318282249116903, -3.841211615494095, 0.04461094346511278, -1.617337574616888, -2.7181960383259276, -2.7614149038680127, 1.5707963588427158, 2.468978769044101, -4.012770026905119, 0.29589799861053134, -0.7388568891100873, -0.42339662833402436, -1.2399857777587897, -3.926990826891757, 3.259904301411565e-08, -6.521426334902808e-08, 1.1478933643403952, -2.7228828964553564, -1.8588125380322826, -2.7171955269594563, 3.92699081588842, -4.712388984421516, 3.42347480976285e-08, -1.1478933901648036, 0.04119851838543039, -1.585650124864282, 1.146399240285612, -1.5707963280630206, -0.5593017029595194, -0.12587204601869884, -1.911305274073867, -2.9472117032301988, 0.4453862890888238, 3.1415926642838246, -1.570796328806121, -1.0114946294955436, 1.6966683619492111, 4.88968465336644e-09, -6.283185325678817, 1.4282800879394413],
        ),
        (8, 2): dict(  # 26 CZ, path, seed 0, robust pick 15
            coords=[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7)],
            cz_layers=[[(2, 3), (4, 5), (6, 7)], [(1, 2), (3, 4), (5, 6)], [(2, 3), (6, 7)], [(1, 2)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4), (5, 6)], [(0, 1), (4, 5), (6, 7)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (5, 6)], [(0, 1), (4, 5), (6, 7)]],
            params=[-1.747091435851154, 1.3650438966653893, -1.4963156882850905, 0.7243754809984432, -1.5707964486317998, 1.5707967144379076, 1.5707963250708792, 1.5707963140450014, -2.8727113975652174, -2.9687515255419883, -3.2156355428303773, -0.7711331145531166, -2.667826536462889, 2.829960688705391, 1.0133799837386552, 2.7585709201560156, -0.10280586172410847, 0.4803497430455318, -0.724607804230786, -2.4797469148653417, -1.6916262713862011, 1.7446967904483326, 0.6154797610523011, 0.9404390249538708, 0.18525438473856937, -2.018234645804437, 1.4945634073743364, 0.787941171661171, 0.5796932975028568, -1.0630548480984767, -2.4407821249660384, 2.080665086948523, 0.7802819949728967, 2.0943950804617053, 2.29867877120356, -1.2515886812207433, 0.32653479788033807, 2.4599506594817933, -2.213504370338546, -0.5281477605201444, 0.009788489860277406, -2.186276016330939, -1.4274485566246515, -0.4241943790539373, 5.759586419007105, -4.712389079908457, -1.6288987943988045, 2.841017326072248, -2.1960645371412717, -2.526112972386863, 1.7452773306202503, 0.733461854349365, -1.0471977353031687, 0.6154794753837496, -0.6154796829156034, 1.8896505216829342, -2.022360551046738, 0.19123498417378482, -1.2747959688866757, -3.875054534689223, 1.5707962700262, -0.4100195899810998, 0.08178840926145188, -1.0258666611839367, -1.9091759058927387, 0.17872120720750784, -0.4704812243526994, 1.5707963262955833, 2.8034974131303967, 1.3653360795356821, -2.7248405599107683, -1.544119701620624, -1.9101773492711813, -2.7261507342033724, 1.5707963201747963, -0.8164117758785704, 1.2709388392898626, 1.520251745024208e-10, -0.49854054289071703, 2.7226794918235093, 0.3439266395111363, -1.5707963238353995, -2.573461711934341, 0.12875433355091292, -1.2709389551799208, -1.8290929670694593e-08, -2.3561946328910452, 2.2035002657418508, -4.195444697288995e-08, 1.5707962892950067, 1.0026653601802733, 2.2584537394976825, -1.2327011127206535, 1.5707963040481836, 1.5707962805424485, 4.712389256241399],
        ),
        (8, 3): dict(  # 43 CZ, path, seed 0, robust pick 2
            coords=[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7)],
            cz_layers=[[(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4), (5, 6)], [(2, 3), (4, 5)], [(1, 2), (3, 4), (5, 6)], [(0, 1), (2, 3), (4, 5), (6, 7)], [(1, 2), (5, 6)], [(2, 3), (4, 5), (6, 7)], [(1, 2), (3, 4), (5, 6)], [(0, 1), (2, 3), (4, 5), (6, 7)], [(1, 2), (3, 4), (5, 6)], [(0, 1), (2, 3), (4, 5)], [(3, 4), (5, 6)], [(0, 1), (4, 5), (6, 7)], [(1, 2), (3, 4), (5, 6)], [(0, 1), (2, 3)]],
            params=[-1.4528572997252809, 1.6939772098043697, 1.9262873874942834, -1.6925572510654292, 2.2125195123530297, 1.4432215887314443, -1.4546771756226995, 1.6341048544231525, -0.22651675507784388, -0.8108278044615322, 1.882978597136051, 0.8622657596163803, 2.296580987075838, -1.0773803660419103, 0.4366878315036302, -0.7575944030329999, 2.9667375171411225, -2.4690583830172494, -1.191201287384057e-08, -0.2591889038072575, 1.2569096870390104, 2.3654749055468707, -2.7762951741351856, -1.6480755433945689, 0.08389752218718916, -3.814126949681976, 1.5707963316686613, -1.1622069848356023, -2.3378124104335596, 2.370638864232938, -1.805638217590833, 0.8854396014517993, -2.0568682131118896, -3.1415927980820175, -0.5309868090034835, -1.1774428735778373, 0.2383793019006771, -3.83260654748873, -4.749938585314228, -1.052870527690528, 2.3103783279462937, 3.1415926330798967, 3.1098524976377657, -0.6347288264336309, -3.157371375154515, -1.7585448968094595, -3.1415928577697394, 2.7307813312823, -0.5031433283414898, -1.497816298145078, -3.141592672027004, 2.205525147436584, 1.5793073544037086, -0.2982929562772877, -1.4824047400423204, -0.7563870533550071, -0.6355191015137842, -3.4981962094023373, 1.7667980694982486, -1.4173921378014023, -3.0957319945588955, 3.1597759335896063, -2.048984377808955, 1.0031490085872614, -1.0413054952292278, -1.5707963174182402, 2.0562182661662722e-08, 0.9786326943754086, -1.5707963297288574, -1.0469757669665862e-07, 0.9734640662055081, -0.8798256738799023, -0.10686526163011476, 5.6671417383888585, -0.6452230793010015, 1.5707962327636695, -6.044537776914961, -1.570796324001922, -4.164366844936398, -0.58233274050579, 0.1068651749811387, -1.7141262774601425e-08, 0.7853982202936503, -1.0312183659906449, 6.284715306014242, -2.0980660678725027e-09, -0.3524146891174211, 1.634253862883702, 2.260592449436984, -0.08776307929396919, -0.8964735655653115, 3.141592748724993, -2.931345223488456e-07, -3.1415930127282183, 1.923211013060196, 1.2475423281385833, 2.500695544972951, 1.2229806023138918, 2.8806816284308, 1.5012009982465802, 3.200036885598149e-07, -1.750033994459817e-08, 2.9042166349316685, 1.8680521647452333, 2.747291158251342, -1.5707963330615746, -1.9842079900968084, -1.5012006897811816, 1.446448263527206, 3.3180684294729814, -0.6511152489333101, 3.0692170038525233, -1.176494724639539, -1.5707963259754543, -8.538205518028015e-08, -3.141592695255856, 4.742387529607343, -0.27562881331612815, 0.6188011336038999, -2.5616890029049535, 1.570796285504707, -1.5707963351730927, -4.712388915728165, 0.20422051000965505, 1.5423276884559203, -1.295167479952093, -2.1895974341752096, 2.634064663997086],
        ),
        (8, 4): dict(  # 29 CZ, path, seed 0, robust pick 14
            coords=[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7)],
            cz_layers=[[(3, 4)], [(2, 3), (4, 5), (6, 7)], [(1, 2), (3, 4), (5, 6)], [(2, 3)], [(1, 2), (3, 4)], [(0, 1), (6, 7)], [(1, 2), (3, 4), (5, 6)], [(0, 1), (2, 3), (4, 5), (6, 7)], [(1, 2), (3, 4), (5, 6)], [(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5), (6, 7)]],
            params=[1.547198989998157, -2.192123647098601, 0.06931157035242458, -2.127957126320808, 2.2833469410568434, -2.160483728229487, -2.7860358403476355, 1.7852074422747406, -0.8574739597764105, 0.6732315697380621, -1.6577635220233697, -1.4207147786755805, 3.8473144649058844, -1.7857426185475427, 1.21523948413521, -3.3560037816655597, -0.004198826183240813, -3.1934968805335133, -1.4355442890070365, -2.237433838909259, -0.9223453434117104, 2.336959039600489, 1.634173992579475, -1.5525024825750688, -1.7170495692696344, -1.3141585233950228, -1.5707963180564533, -0.8506827351245877, -0.348160400452928, 3.309528269803486, 0.39064960802909826, 1.9141753132392516, 2.343156708961155, -3.3982304470059996, 1.5707963164646408, -3.1415926408592867, -0.23984220891096922, 2.079142535865502, -3.019238460414366, 1.4026127068746377, 0.692700772820962, -3.1415926318648713, 0.4711905796767751, -1.3604062178756855, -2.4911019318437515, -1.9141460693156949, 0.4423128178197566, -1.2121834937704705, -1.3811631605860242, -1.5707963082774687, 1.0996057789909608, -3.351982773047053, 0.8196319107198208, 1.237864244954955, 1.570796343955774, 2.7996051877212733, 0.42584652912954774, 0.6895745041167302, -3.141592646249463, -1.5707963083947118, -4.712388980666592, -1.3110794700647877e-08, 0.7853981674979944, -1.228808816525811, -2.0845470123503893, 2.3561944790627387, -1.5747251394914847e-08, -0.785398166751132, -4.712388978720002, -1.5707963266774692, 3.926990816494695, 1.60698247705685, 3.655343348116879, 1.5707963276661134, 4.5388134591135784e-09, 1.570796329583965, 4.460134156418878, -0.31265938414584604, -2.3409508365436533, -0.001007751407567284, -1.1380244189286717, 0.24497881352567927, -1.5707963125445377, -0.9762019528890976, 1.5280363069761873, 0.7580899024028405, -2.0994244961237554, 2.1363876711803926, 1.1380244321383224, -1.5707963197573402, 1.0691935657763076, -1.0096401934073185, -1.5280362997714414, -3.899682555550743, -0.27201364791671406, -2.171566084946386, -1.5707963148500337, 1.2687064319744044e-08, -1.0691935839103623, -1.1557505083119262, 4.460134159231326, -1.258136945539457, 1.5707963262061964, -7.49252850561493e-09],
        ),
    }
    # END SYNTH TABLE


    # ---- larger cases: B-E from synthesized nearest-neighbour SCS blocks ----------------------
    #
    # SCS_{m,k} acts on the k+1 consecutive line qubits m-k .. m, and its input is always a
    # sorted string |0^{w-l} 1^l> on that window. experiments/scs_synth.py finds exact Ry + CZ
    # circuits for these small isometries on a path, so the whole B-E circuit becomes
    # nearest-neighbour on a line: no routing. Keys are (m, k, all_ones); all_ones marks the
    # second-group blocks SCS_{m,m-1}, m <= K, whose window can hold |1^m>.


    def _scs_blocks(N: int, K: int) -> list[tuple[int, int, bool]]:
        return [(m, K, False) for m in range(N, K, -1)] + [(m, m - 1, True) for m in range(K, 1, -1)]


    def _scs_cz_count(N: int, K: int) -> int | None:
        keys = _scs_blocks(N, K)
        if not all(key in _SCS for key in keys):
            return None
        return sum(sum(map(len, _SCS[key][0])) for key in keys)


    def _scs_circuit(N: int, K: int, path: list) -> cirq.Circuit:
        """U_{N,K} |0^{N-K} 1^K> = |D^N_K> on `path` (path[0] is B-E qubit 1).

        Operations are appended in order with cirq's EARLIEST insertion, so consecutive
        blocks overlap in time wherever their qubits allow (the paper's "pushed stairs").
        """
        ops = [cirq.X(q) for q in path[N - K:]]
        for m, k, all_ones in _scs_blocks(N, K):
            layers, params = _SCS[(m, k, all_ones)]
            window = path[m - k - 1:m]
            th = np.array(params).reshape(-1, k + 1)
            for l in range(len(th)):
                ops += [cirq.ry(th[l, i]).on(window[i]) for i in range(k + 1)]
                if l < len(layers):
                    ops += [cirq.CZ(window[i], window[j]) for i, j in layers[l]]
        return cirq.Circuit(ops)


    # BEGIN SCS TABLE (generated by experiments/scs_synth.py --export; do not edit)
    _SCS = {  # (m, k, all_ones): (cz_layers, ry_params)
        (2, 1, False): ([[(0, 1)], [(0, 1)]], [-1.015321455826436, 0.9012076585199794, 0.7853981635004875, -2.1584696466585975, -0.7290898508023025, 1.5707963268232388]),
        (2, 1, True): ([[(0, 1)], [(0, 1)]], [1.5707963299106977, -1.5707963279692112, 0.7853981644905792, 0.7853981648548624, -1.5707963324863208, 1.5707963356010937]),
        (3, 1, False): ([[(0, 1)], [(0, 1)]], [-1.1407597698521028, 0.8649861674793102, 0.9553166183850991, -2.025741846810719, -0.7639646880798547, 1.570796325631784]),
        (3, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [2.5261129440759578, -3.1415926534463203, 1.6814398415163123, 1.8673550539072388, 1.5707963269906793, -3.252236168399352, -2.482834760860102, 0.6154797037939795, 2.6934577818389718, 1.4461898972389164, 1.570796322171429, -1.7381411647522957, 1.6954027561629759, -1.5707963268805634, 1.5707963267467884]),
        (3, 2, True): ([[(0, 1)], [(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.5707964203929847, 1.5707963520373425, -0.7875640413941856, 2.526112939928778, 0.6154797147869595, 3.442482558804403, 1.0254108620531002, -1.570796367896398, 2.0574703112574455, 0.5453853558120263, -1.8785361722138427, 2.530702774328467, -3.5450922942447773, -0.3077398612692013, -0.9599064343508606, 0.4034996247001735, -0.3077398564289521, -2.2855112763759493, 0.7539365622784483, 1.8785361808190313, -2.426877804746498, 2.3876561191795918, 3.1415927900973726, -1.5707963312735973]),
        (4, 1, False): ([[(0, 1)], [(0, 1)]], [-1.2067991147336417, 0.8501571332801536, 1.0471975503230786, -1.955939742194762, -0.7784186580454592, 1.5707963250791273]),
        (4, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [2.3561944902018714, -3.141592653201629, 1.681439841579331, 1.7823958256956658, 1.570796327062821, -3.252236168336336, -2.567793989071676, 0.5235987756088417, 2.7393982488821935, 1.4461898972013847, 1.5707963267701406, -1.6922006977090764, 1.6954027561254483, -1.5707963268858744, 1.5707963268049172]),
        (4, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [0.5235987756403431, 1.5241303672196429e-10, -1.5707963268101248, 1.570796326790054, 1.5284495953370514, 1.5707963266253244, 2.6179938771048707, -0.3973794920781327, -2.05204837083282, 2.795357843898402, -1.5707963267929774, 1.4445770418768198, 2.7795191979441385, 1.5243320518143768, 1.570796326754649, -3.956499180968387, 0.3620734557846727, -3.188056931174852, 3.1415926535875043, 1.9930037750916123, -2.9766722307666393, -2.0099596830663353, 1.570796328158474, 1.052883329402526, 2.9766722311973584, -1.3226001086956851e-08, 1.5707963268001304, 0.12521391232496118, -3.2577292833979974, -1.0203875102664337e-08, -1.570796325149596, -2.0735106129714755, 0.11613662974040719, -1.570796326802337, 1.5707963267949383, -2.6388783674678624]),
        (4, 3, True): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(2, 3)]], [1.5707963062814767, -1.5707962713225228, 1.5915914333336953e-08, 1.570796227073341, -1.7720651735309385, 0.7853981867001699, -1.5707963511249923, -0.22775788406567657, -0.5841292782624524, 1.0154709729356895e-08, -0.7853981708967535, -0.4267415368013041, 1.6797599165278145, 2.3561945140333775, -3.14159262901271, 0.06740666596816665, -1.6797598588283682, 1.5707962893558238, 0.5235986287973314, 2.157888978693121, -3.1291894908699924, 0.30773987999020186, -1.047197590649111, -1.1319250404546353, -1.5831994422047118, -0.3077398688417215, -3.1415926888392445, -3.5804639918184415, -2.351240053941933, -0.3077398722401282, -3.1415926669134913, 4.650550553507826, -5.502741620140648, 0.30773989386597483, 0.7853981174131546, 0.06183841104645399, 3.307166940539704, 9.271642357478542e-10, 0.7853981113726165, 1.8846489422097708, -0.5529118002335035, -8.825476369880942e-09, -2.356194441066823, -1.8846490607070039, 5.099726468331428, -3.141592663128518, -2.3561944279409666, 7.026400268227146e-09]),
        (5, 1, False): ([[(0, 1)], [(0, 1)]], [-1.2493722484973575, 0.8420415032801906, 1.1071487289052535, -1.9109407348473444, -0.7863771580214348, 1.5707963302956998]),
        (5, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [2.2555155297783154, -3.1415926528393414, 1.6814398416230167, 1.7320563455920457, 1.5707963272402872, -3.2522361682926526, -2.618133469175297, 0.463647609041793, 2.7693738320971537, 1.4461898970645548, 1.5707963266562575, -1.6622251144941151, 1.6954027559886222, -1.570796326979636, 1.5707963268268799]),
        (5, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [0.684719200760806, 3.89382212395982e-10, -1.5707963267475615, 1.5707963267336866, 1.4478893830798967, 1.570796326692386, 2.6779450484263787, -0.3674039087853684, -2.1326085830899757, 2.7449228296430337, -1.5707963267538714, 1.4745526251695888, 2.7795191977243827, 1.5244275926819684, 1.5707963212754228, -3.981668924602958, 0.36207345556490583, -3.1879613890938696, 3.1415926535184586, 1.96783403145703, -2.976672230610278, -2.060203622833494, 1.5707963287255762, 1.0277135905950752, 2.9766722313537137, -4.066488119839899e-09, 1.5707963267903209, 0.1000441735175128, -3.257729283173108, -6.295575621867655e-10, -1.5707963293194807, -2.073510612800667, 0.11613662996529607, -1.5707963266985356, 1.5707963269296419, -2.6388783672970573]),
        (5, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.936893910683001, 1.871831455622161e-08, 3.141592591392152, 1.5707963960678542, 3.1415925774019144, -0.11735621215646262, 3.141592673859249, -2.1394113976540326e-06, -3.1415903419986133, -2.791479456665574, 2.6120048372655607, 1.5707963096362427, -1.5707941247056312, -4.712391388645436, -0.8137608055784339, 0.06594020748485975, 2.2555155295958573, 3.8263118571164636, -1.5707957936378947, -0.5661986805647138, 0.4147308149040186, 1.570796327049326, -3.821641166415872e-08, 2.3561944909274692, 3.2441437257665067, 2.784515549756841, 1.5707956562641578, -3.5846312164310614, 0.443038561075173, 3.0353930185057965, 3.2290247643657475, 3.14159330223582, -4.71238856560492, 1.570796774112821, 0.1061996412574048, -0.14508591862222386, -3.1415926610720684, -1.5707959949354213, 1.5707959844508377, 2.7060146909274128, -2.6880665512405644, -6.283185244722265, 1.2284367264929819, -1.913155929314181, -1.1352183259637156, 2.771796463164189, 3.141592662190583, -0.7853981644060074, 1.599243135582578e-08, 0.8481806271369727, -3.376244876129918, -7.573690193787372e-09, 3.141592523789559, 1.570796327020095, 3.8642083539270997, 0.1509223963730345, -3.1415926077121346, -1.570796403085478, -1.5707963651568324, 7.853981267970707]),
        (5, 4, True): ([[(0, 1)], [(1, 2), (3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(3, 4)]], [1.5707962405969675, 1.5707962905123638, -1.9986545615303268, 1.882695571581244, -2.150853264573379, -1.3630175444522235, -2.677945032234645, 0.8741077143157976, -0.3118992649631603, -2.5615353750516645, 0.8993699254217412, 3.141592661669857, -3.607108004568963, 1.2151378045825205, -0.12295362875472635, 1.4035755263265186, 3.1415927490942335, -0.2192038701614214, 1.4628072402230956, -2.554991419649251, -0.04684644596051661, -2.174502111967305, 2.6953431446289735, 1.570796120146085, -2.295859340028196, -1.9942050409288767, 3.7452984615716796, -2.6021054572634482e-08, 0.5277874645278107, -3.4695003705082152, -0.9333202521755063, 2.013834971859119, 1.5707962329015617, 2.170766630763149, 1.107362284280402, -0.2843893751065307, 0.44303855516324897, 3.1415925356617485, -2.6985540883302965, -3.195983908742667, 0.2843893956898251, -0.4430385404753229, -1.5707960835613908, -2.356194485470851, -1.2561865755063177, 1.8097006292329474, 3.5846311861267632, -1.5707962869997616, 2.776322687234782, 0.7458858235239523, -1.1113611822858258, 1.3080336495753961, -1.5707963186640008, -1.2055262213991538, -3.0191443300641976, -0.9142762245197629, -1.3080336385638183, -2.799233049705837, -1.2284367313976283, -1.1829439056399105, -1.3475879118655933, 1.927622343388024, -2.356194488014053, -3.141592656399593, -0.7631716757874621, -0.2812515271040594, -1.0164348406804926, -3.0157594101766727, -4.712388980018044, 2.33396800250067, 1.8447761792520496, 2.230405148108358, -1.6966296201713749, -1.5707963309089443, 4.712388843133264]),
        (6, 1, False): ([[(0, 1)], [(0, 1)]], [-1.2797755554426096, 0.8369137344221296, 1.1502620026722152, -1.8788000046902575, -0.7914245858992704, 1.5707963332316701]),
        (6, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [2.186276036096864, -3.1415926501151947, 1.6814398431344826, 1.697436600970363, 1.5707963268663996, -3.252236166781186, -2.6527532137969825, 0.4205343368891023, 2.79093046674452, 1.4461898987749984, 1.57079632171125, -1.6406684798467481, 1.6954027576990653, -1.5707963256518889, 1.570796328167465]),
        (6, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [0.7853981589339939, -3.549500306381916e-10, -1.5707963266138816, 1.5707963268520218, 1.3975499050013058, 1.5707963277404189, 2.7210583204638326, -0.3458472821383155, -2.1829480611685605, 2.7102644641988918, -1.5707963268088607, 1.4961092518166574, 2.7795191968060693, 1.5244662001349762, 1.5707963257110988, -3.998978796558535, 0.36207345464661134, -3.187922798068281, 3.141592653580437, 1.9505241595014517, -2.976672227870911, -2.0947847693487045, 1.570796346078658, 1.0104037115111997, 2.976672234093094, -1.6588549673737222e-09, 1.5707963267020681, 0.08273429443362254, -3.257729282577532, 1.4514719038134356e-08, -1.570796324836202, -2.0735106124243807, 0.11613663056086608, -1.5707963265947398, 1.5707963268300777, -2.638878366920778]),
        (6, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.860977860665075, 4.211887568202972e-09, 3.1415925994090874, 1.570796374161921, 3.1415925776238254, -0.19327226217438234, 3.1415926593528214, -2.131394467864578e-06, -3.1415903390651003, -2.7699228206087043, 2.5360887867704607, 1.570796349028954, -1.5707941166887, -4.712391385711917, -0.7922041695215678, -0.009975843010240588, 2.3561944886041624, 3.926990817642552, -1.5707956968073693, -0.5446420455053238, 0.4147308157164884, 1.5707963267844822, -3.8721587761431525e-08, 2.3561945037224032, 3.265700360825894, 2.7845155505693104, 1.5707956545162036, -3.61925096606253, 0.477658300054291, 3.035393022424044, 3.2290247651782216, 3.141593300487861, -4.712388457029337, 1.570796847538605, 0.10619964517565275, -0.14508591780975338, -3.141592659337703, -1.5707959270603578, 1.5707959215145864, 2.7060146967600858, -2.68806655042809, -6.283185242987902, 1.26305647846094, -1.878536190479966, -1.1352183201310444, 2.771796463976664, 3.141592663619797, -0.785398167669143, 2.0055353870508316e-08, 0.848180627677561, -3.376244875317446, -6.144477727820846e-09, 3.1415924895261513, 1.5707963263981555, 3.864208354467685, 0.15092239718550374, -3.1415926197304955, -1.5707964300630364, -1.5707963642802332, 7.853981183269579]),
        (6, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)]], [0.42053433258321876, -1.549842151719446e-07, 1.4069652055640711e-07, -4.814425782158794e-07, -1.5707961561581192, 1.570790722550021, -2.1454701325680245, -1.5707955705745127, 1.5707964812134094, 2.067995541065594, 0.23169267349743675, 2.1391579961605434, 1.7249357989312353, 2.186275445323593, -0.11717951096989422, 1.4702009657337565, -2.9527510298225863, 1.4229689420984386, -2.2027260695511885, -1.5707957726301593, -2.408933453134081, 1.1741930365228812, 1.5707954330869682, -2.461089457053677, 2.127905062808956, 0.6362712967333183, 0.5821750039480882, -3.1404325708211096, -2.0484546398535888, -2.251304676921941, -2.0345087876371553, 1.4217632346039295, -0.442161653534664, -1.5719571581091702, -0.4776583127415695, 0.6469591185897509, -1.667821701730063, -1.2635042797529583, -0.14000923504819818, -1.570792161358871, -2.6639343381426914, -0.6469591545480542, -0.8724400198514266, -2.3653261921985655, 0.7853981675227605, -2.3561944649185995, 2.0484546356300783, 1.8552360385534807, -1.6335920843664318, 3.1415920799862747, 3.534291723024262, -5.105088069967489, -3.141592647685746, -1.8552360614075443, 1.0979431865365783, -3.14159205469281, -1.570789511304214, -1.5707952706479762, 3.1415926565033936, 2.058234102260265, -1.8348282752819982, -3.292403744404596, 1.5707976850896372, -1.5707952821403859, 0.4138266023375347, 1.5713217183930421, 2.833499053862503, 0.195996151496813, 2.936540055379419, -2.7488935872297176, 1.1569696989723142, 0.11404910369585425, 0.3080920653761022, -0.19599648934020245, 2.1685480003493236, 2.7108066598343106e-08, 1.570796332022784, -2.172808634567402, 0.32515808135153584, -3.292404013741152, 3.0696863408990336e-06, -1.5707963235490705, 3.1415926592515793, 3.1287609287352875, 3.0097733658169754, 1.7960283654163407, 1.5708002638017422, 1.570796010271263, -1.5707963369046134, -1.5579646289453564, 0.5435464526853294, -1.796028442438568, 3.141596831247581, -3.1415923169523756, -1.5707963423059104, 1.570798642693676]),
        (6, 5, True): ([[(0, 1), (4, 5)], [(1, 2)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(4, 5)], [(3, 4)], [(0, 1), (2, 3), (4, 5)]], [-1.5707967063062631, -1.5707966053324622, 3.5084250869050506, 2.174514996879484, 1.570796416578593, 1.5707929948400583, -3.2321304725218423, 2.356194471144124, 1.2039646207485126, 3.3011564138473246, 0.5350107046829758, -0.7559616733388369, -2.2656566495472283, -3.141592620876377, -2.5261129574939196, 2.3783102821238873, -0.11447635268345013, 3.4770199739511316, 2.116449016504574, -2.526112956192014, 3.1415926713783997, -0.7853981749391284, -1.5707976095503686, -2.861615120789052, -2.1164496684404885, -1.5707966638844573, 2.7488938090961086, -1.963495401656546, -0.45511302034991025, -1.8507737152827088, 1.0636735573579283, 9.774213404797473e-08, 1.5707965937153012, 1.5514493367556417, -1.176648753082684, 3.8577088656033793, 0.5071242981683495, 2.799233010567725, -3.141592507694479, 1.5901432903592503, -2.2952290713254637, 2.4254794148324854, -0.676823189660211, 0.34235962966412387, -2.7488935761256807, 4.712388798444849, -3.619250996113091, -0.3200481872931435, 4.111924830929902, 3.012601329703603, -1.5707965716319867, -1.7768032399906891e-09, -2.6639342962382395, 3.4616407817596326, 2.318439640691369, 3.270583870898994, -1.5707963501098192, 1.570794967995352, -2.356194537811598, -2.33703683514301, 3.671235835350571, 0.34235950394084885, -2.74889361202183, -1.5707929780111125, 1.9761083489767717, 2.435620907081589, -1.7586054687380437, 2.8692998441890154, 0.6218828793787473, 1.5708013131281864, 2.7362804218982, -0.3353872447474936, -0.827287592909352, 2.1820763809269055, -3.3376851305668014, 2.2622084419193995, 0.30773983276575617, 1.8075994239910924, 1.675548169981364, 1.4029888051307544, -0.4257903926571502, 2.7579203924439177, -4.211139133119618e-08, 2.273252018801587, 2.4811397531413544, 3.312772426368298, -3.1415926553194184, -3.141597413636725, 4.712388976721598, -0.7024557017616246, 1.5707965560631183, -2.245069441510587e-07, -3.141592687710977, 1.5707932374034428, -4.712388951043549, -1.5707953552648874]),
        (7, 1, False): ([[(0, 1)], [(0, 1)]], [-1.3028972020112384, 0.8333779884976906, 1.1831996402511757, -1.8543528424609963, -0.7949135493121982, 1.5707963265368166]),
        (7, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [2.1347389682452946, -3.1415926514608477, 1.6814398416349414, 1.671668064646064, 1.5707963280561572, -3.2522361682807266, -2.6785217501212792, 0.38759668673761727, 2.807399293372445, 1.446189896108908, 1.5707963269096916, -1.6241996532188234, 1.695402755032974, -1.570796327655232, 1.570796326742017]),
        (7, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [0.8570719450308016, 2.7224391828359735e-09, -1.5707963266472988, 1.5707963268529699, 1.361713010004807, 1.5707963253282293, 2.7539959774212326, -0.3293784467687446, -2.2187849561650554, 2.684477224580026, -1.570796326724253, 1.5125780871862273, 2.779519198114746, 1.5244849215464118, 1.5707963247076349, -4.01186306499019, 0.3620734559552743, -3.1879040578981135, 3.1415926535883965, 1.9376398910697978, -2.976672231734907, -2.120534579407098, 1.5707963288224978, 0.9975194526334087, 2.9766722302290867, 8.618954545886777e-09, 1.5707963268074507, 0.06985003555581411, -3.257729283065587, 1.3163520811314764e-08, -1.5707963293097542, -2.0735106127505665, 0.11613663007281427, -1.5707963265560467, 1.5707963268517058, -2.638878367246968]),
        (7, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.811855528163934, 6.163306385442014e-11, 3.14159257450944, 1.5707963929540798, 3.141592588272582, -0.24239459467553223, 3.1415926552025617, -2.1562941165334087e-06, -3.1415903524728934, -2.753453995612028, 2.4869664510684637, 1.570796290275634, -1.5707941415883497, -4.712391399119711, -0.7757353445248908, -0.059098178712242364, 2.427868274085418, 3.9986646048125714, -1.5707956482183376, -0.5281732181132602, 0.41473081590479977, 1.5707963242941279, -3.649741880095522e-08, 2.3561944856158545, 3.2821691882179596, 2.784515550757625, 1.5707956729654902, -3.6450194986908633, 0.5034268439302398, 3.0353930165705947, 3.2290247653665314, 3.141593318937147, -4.712388442513509, 1.5707968564171182, 0.1061996393222045, -0.14508591762144218, -3.1415926695969363, -1.5707958149091832, 1.5707958656622094, 2.7060146841431343, -2.6880665502397783, -6.283185253247129, 1.2888250003384225, -1.852767648540013, -1.135218332748002, 2.7717964641649786, 3.1415926638256817, -0.7853981583674773, 1.5366865700216192e-08, 0.8481806290140685, -3.37624487512913, -5.938592686656713e-09, 3.1415924413673095, 1.5707963226780821, 3.8642083558042044, 0.15092239737381608, -3.1415925843946475, -1.570796422349087, -1.5707963569840295, 7.853981116210921]),
        (7, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)]], [2.5776500324580227, 3.141592358382348, -3.1415926359429647, 3.1415926962306853, -1.5708016074384625, 1.57079337465175, -2.4629790476615154, -4.712389021914221, -4.712388995413269, -2.5270113624107724, -1.6034664841135167, 2.813958621000956, -1.2425562300457922, 3.998655116782793, -0.13453469360152306, 3.199519366487829, 1.2158697955088758, 3.0816300186761847, -1.3360714002069587, 1.5708058121143404, 0.8482590789184595, 0.8982881259181561, 1.5708004736565826, -1.5012746529655874, 1.863228749529144, -4.712389051107567, -2.407066087126139, 2.064259771759787, 3.6450195214507763, -0.06951809499040341, 0.7067351753808948, -1.5802353352954774, -2.2641973410745138, -0.4934632559505372, 1.0673695094630073, -0.5437003158632249, -0.4830844791676599, 0.5307047207539994, -1.6119199177820374, 4.712388866887893, 2.074223156788373, 3.685292916942179, -2.281757891985551, -2.0920621638272316, 5.497787171793717, 5.497787104740473, -2.6381657908597305, 2.9008432983581107, -1.610644223476683, -3.141605976736388, -3.4984548205829675, 2.7130567012456592, -6.960607324793232e-08, 0.24075027695944218, -3.3457009163203426, -3.141605975212197, -1.5707918161921472, -1.5707959466254569, -3.141592643371282, 1.5361883234962592, -2.9019069169087413, 0.06614416044605188, 4.71238910035252, -1.5707967177641613, 2.4915012065489424, 1.6028159576049121, -1.6794534324136754, -3.141587867399735, -0.17406386302494634, -0.3568622218244778, -0.9207048928984117, 2.325228257842731, -1.4621405417248492, -3.141588029467776, 2.8871205227797017, -3.395858663399311e-09, -1.5707963325968475, 2.3897471894789963, 1.2421356907490322, 0.06613476150728637, 3.1415932155412136, -4.7123889845219615, -2.9010666534545572e-08, -3.187996632652778, -1.6050850471074223, -1.4377258729314182e-08, 1.570791050615118, 1.5707963139130154, -1.5707963056964873, -1.5243923441400924, -5.9558134170279216, 3.1415926804682517, 3.14159301349042, -3.141592624649799, 4.712389052680858, -1.5707959320249427]),
        (8, 1, False): ([[(0, 1)], [(0, 1)]], [-1.3212505296537456, 0.8307917205497494, 1.2094292020806356, -1.834944378108838, -0.7974701857608345, 1.570796326248331]),
        (8, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [2.094395104259356, -3.141592649861804, 1.681439841912144, 1.651496132365669, 1.5707963286018456, -3.2522361680035243, -2.698693682401674, 0.36136712550263966, 2.8205140742161325, 1.446189895601535, 1.5707963245891852, -1.6110848723751359, 1.6954027545256016, -1.5707963280095594, 1.5707963268875684]),
        (8, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [0.9117382921717447, -3.4590898673866783e-09, -1.5707963254926276, 1.5707963277411525, 1.3343798368042403, 1.570796328167911, 2.780225529207832, -0.31626366651715226, -2.2461181293656516, 2.6642948836947005, -1.5707963268852754, 1.5256928674378079, 2.779519197422447, 1.5244953199408244, 1.5707963257852704, -4.021949029203483, 0.36207345526296175, -3.1878936610324193, 3.141592654283685, 1.9275539268565236, -2.976672228653015, -2.140696110717492, 1.5707963242351763, 0.9874334829468264, 2.9766722333110045, -1.3596703063668512e-09, 1.5707963263792186, 0.05976406586926228, -3.2577292833597182, 6.573232268447581e-10, -1.5707963259387991, -2.0735106110430643, 0.11613662977868619, -1.570796326178581, 1.5707963279977222, -2.638878365539461]),
        (8, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.776018631153869, 3.690623860378068e-09, 3.1415925756828003, 1.57079637144368, 3.141592576198967, -0.27823149168559996, 3.1415926588315477, -2.1551207550799243e-06, -3.141590363026949, -2.740339211484763, 2.451129557828329, 1.570796288711113, -1.5707941404149877, -4.71239140967377, -0.7626205603976312, -0.09493507195237647, 2.4825346271262325, 4.05333094696149, -1.5707955163808243, -0.5150584386049629, 0.41473081454951904, 1.5707963343962257, -4.186835705317384e-08, 2.356194515911853, 3.2952839677262653, 2.784515549402349, 1.5707956690846572, -3.6651914238785848, 0.5235987697558312, 3.03539302063172, 3.229024764011251, 3.1415933150563093, -4.71238828654429, 1.5707970011211807, 0.10619964338333165, -0.145085918976726, -3.1415926660175977, -1.5707956821144438, 1.570795772048599, 2.706014691461905, -2.6880665515950604, -6.283185249667801, 1.3089969609356658, -1.832595748826142, -1.1352183254292174, 2.771796462809704, 3.1415926628709707, -0.7853981829456529, 1.993641733286267e-08, 0.8481806236680415, -3.3762448764844124, -6.893303962651722e-09, 3.141592391666859, 1.5707963371139528, 3.864208350458181, 0.1509223960185396, -3.1415925801256535, -1.5707964440972924, -1.5707963579492075, 7.853980971137368]),
        (8, 5, False): ([[(1, 2), (3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(3, 4)], [(2, 3)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (4, 5)]], [2.17210049984681, -1.5707963364933308, -1.5707963279495292, -1.570796326960163, -1.5707963251959494, 2.9891268607565937, -1.1249029131683794, 3.1975562055844735, -0.7853981677463828, 3.665191434395755, 1.0471975313056234, 0.15246580556114217, -0.8673479208749676, -2.4121580387916666, -5.811712208208417e-09, 1.5707963276646373, 3.1415926612725715, 0.2802310934116177, -0.32913148368282347, -2.596475567763119, 2.6857235021441954, 1.5707963322813416, 0.45586913372208465, -3.2411401837974907, -1.2116083596789355, 1.0256792568019226, 2.02666547627613, 1.7956949718277733e-09, -1.5707963312260727, 1.4520121352772053, -0.209906162975556, 3.1415926630251354, -0.6590580376220412, 1.5707963315646591, 1.570796309469623, 1.6895805360266756, 2.279013758504131, 1.7233459178144073, 3.1415926561278518, 1.5707963305774373, -0.4558691341932658, 1.6836114325399996, 1.2094043388736786, -1.4182467333580668, -3.141592657580788, 1.5707963438644246, -0.3196351889655169, 3.3259527783085643, 1.863412139832413, -4.232090008661287, 1.5707963365261095, 0.7853981624576323, 2.4278839139375754, -0.21281311992334284, 2.502151405410028, -0.7853981717750586, 5.180513682399239e-09, 0.7853981647339503, 1.033343924858469, 3.651581212374006, -2.9749135476725024, 0.7853981704224134, -3.1415926655821638, 2.103439575741413e-09, -1.570796325693223, -1.4527085744731505, -1.737475443182789, -0.7853981705003886, -1.5707963384955619, -1.7013223526147541e-09, -1.5707963344238567, -0.11808774591175483, 2.783635536755788, -6.283185317392107, 9.635321641201464e-09, 4.5568216027994415e-09, -2.9609090914433533, -0.09440169254800422, 1.9287534271087525, 1.570796325214528, 0.47904972253073963, 8.221620022005144e-09, -1.7514798905183246, 4.011028577013559, 3.141592645634192, 4.712388968124107, -2.0498460470669797, 3.141592639141935, 7.85398163128895, -4.7123889702423245]),
        (9, 1, False): ([[(0, 1)], [(0, 1)]], [-1.3362794243636635, 0.8288174423076456, 1.2309594130292647, -1.8190493547162818, -0.799424515677198, 1.570796323664979]),
        (9, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [2.0616790053138483, -3.141592649283687, 1.681439841345574, 1.6351380831473525, 1.5707963296697243, -3.252236168570093, -2.7150517316199916, 0.33983690960082674, 2.83127918196056, 1.4461898933571045, 1.5707963268522276, -1.6003197646307106, 1.6954027522811723, -1.5707963294870269, 1.5707963262823494]),
        (9, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [0.9553166179696087, 3.61035443818762e-09, -1.5707963267903775, 1.5707963268467324, 1.3125906745652252, 1.570796324693049, 2.801755748788916, -0.3054985580292448, -2.267907291604686, 2.6479305328320812, -1.570796326741588, 1.5364579759256973, 2.7795191988298003, 1.5245016275719345, 1.5707963267503806, -4.030128055632309, 0.3620734566703084, -3.187887351038198, 3.14159265357671, 1.919374900427727, -2.9766722319240935, -2.157047851278967, 1.5707963254324602, 0.979254462128551, 2.976672230039933, -6.629591856691456e-10, 1.5707963268209466, 0.05158504505102767, -3.2577292835114413, -1.5727959484110952e-09, -1.5707963313876194, -2.073510613031172, 0.11613662962697612, -1.5707963266061158, 1.5707963267403529, -2.638878367527567]),
        (9, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.74818337958148, -7.846484335997964e-09, 3.14159257351821, 1.5707963874759805, 3.1415925900599455, -0.30606674325798866, 3.1415926472944333, -2.1572853473664027e-06, -3.1415903656509196, -2.729574107152622, 2.4232943064768806, 1.570796309986381, -1.570794142579581, -4.712391412297742, -0.7518554560654868, -0.12277032330382265, 2.526112944905079, 4.09690927171735, -1.57079541284025, -0.5042933310093146, 0.4147308173037127, 1.5707963269553222, -4.0806164721492375e-08, 2.356194479630151, 3.3060490753219094, 2.784515552156541, 1.5707956681373696, -3.681549473110016, 0.5399568289766049, 3.0353930174280976, 3.2290247667654444, 3.141593314109025, -4.712388207915789, 1.5707970619279716, 0.1061996401797174, -0.1450859162225332, -3.141592665982087, -1.5707956091500899, 1.5707957101394785, 2.706014686816974, -2.688066548840865, -6.283185249632289, 1.3253549772926136, -1.8162376553008115, -1.1352183300741554, 2.7717964655638987, 3.141592660940218, -0.7853981530830609, 2.0263743452160047e-08, 0.8481806268917357, -3.376244873730219, -8.824059727071894e-09, 3.1415923726661092, 1.5707963265579465, 3.8642083536818777, 0.15092239877273333, -3.141592573444156, -1.570796455263786, -1.5707963549874027, 7.853980889490922]),
        (9, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)]], [2.4118649965778336, 3.141592527633889, -3.141592314257473, 3.1415914198854327, -1.5707975794145332, 1.5708304049124027, -2.5458715640286282, -4.712389280785085, -4.712389391061723, -2.527011238643, -1.5795865942168315, 2.8378385020835935, -1.3254487464118265, 3.8713212865926105, -0.07086254744606545, 3.1995194902563875, 1.2397496854056163, 3.1055098997604986, -1.3360708931155507, 1.5707953549842297, 0.911931225073754, 0.8982882496841029, 1.5708088113001757, -1.5012924298971444, 1.8632292566199655, -4.71238904053475, -2.407066598618577, 2.0642596368911916, 3.6815494507062803, -0.06953587192096365, 0.7067356824713793, -1.580235324723085, -2.2641978525665474, -0.4934633908199167, 1.0308394735936415, -0.5437002481138385, -0.4830839720780924, 0.5307047313269174, -1.6119204292740978, 4.712378573995498, 2.110753182958813, 3.6852929846924307, -2.2817573848962938, -2.0920621532541883, 5.497787156660723, 5.497787133045668, -2.6016358461258933, 2.900842933395718, -1.6106437163860154, -3.1415912671086317, -3.449332508164067, 2.663934352969156, 1.6110306873416106e-07, 0.24074991199529377, -3.3278519862231386, -3.141591259124314, -1.5707850774494052, -1.5708067503388197, -3.1415927967087955, 1.5361887135144643, -2.884057986811396, 0.06799862392313141, 4.7123989738164855, -1.5707865582618203, 2.491501256319552, 1.6028163476242177, -1.6794519093003928, -3.1415930655946256, -0.19862502579365587, -0.30773985085211925, -0.92070484312596, 2.3252286478605204, -1.46213901860937, -3.1415932276623706, 2.8625593600088286, 1.3991002254326521e-07, -1.570796323408394, 2.3897475794993137, 1.2302364147798217, 0.06799960350213356, 3.1415826691271653, -4.712388990486183, -8.968597662164484e-09, -3.187996635535275, -1.6169843230772178, -1.4403238705076012e-07, 1.57078580985941, 1.5707957743091292, -1.5707963262877211, -1.5243923470232699, -5.967712692995975, 3.141592550813532, 3.1415928223235925, -3.14159213204553, 4.7123890133742234, -1.5708083449977233]),
        (10, 1, False): ([[(0, 1)], [(0, 1)]], [-1.3488808670096775, 0.8272607687864508, 1.249045770565431, -1.805720169749727, -0.8009671180962932, 1.5707963256669961]),
        (10, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [2.034443936180929, -3.141592645232987, 1.6814398425253279, 1.6215205495800442, 1.5707963303722394, -3.252236167390339, -2.7286692651873, 0.32175055462911134, 2.840322358803394, 1.4461898933555006, 1.5707963245934387, -1.591276587787879, 1.6954027522795703, -1.570796329450161, 1.5707963270345593]),
        (10, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [0.9911565868025957, -3.9146245040192376e-09, -1.5707963267909832, 1.5707963267200848, 1.2946706897205136, 1.5707963281607231, 2.8198421044904936, -0.2964553786281865, -2.2858272764493845, 2.634308941497357, -1.5707963267475427, 1.5455011553267735, 2.7795191982433325, 1.5245057754052933, 1.5707963086482921, -4.036936868265358, 0.36207345608382957, -3.187883207162176, 3.1415926536197647, 1.9125660877947135, -2.976672230305594, -2.1706613318739243, 1.5707963151321802, 0.9724457391106534, 2.9766722316584757, -9.810677525911164e-09, 1.5707963269019043, 0.04477632203309971, -3.2577292842820684, -6.156217724481243e-08, -1.5707963825972495, -2.07351061315182, 0.11613662885634245, -1.5707963270577288, 1.5707963271786347, -2.6388783676481977]),
        (10, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.725679150421451, -8.433165907531339e-09, 3.141592571347952, 1.570796384502187, 3.141592570080566, -0.32857097241802574, 3.1415926467077577, -2.1594556071324817e-06, -3.1415903897870883, -2.720530944400553, 2.40079007785746, 1.5707963036329597, -1.5707941447498428, -4.712391436433911, -0.7428122933134167, -0.14527455192324376, 2.5619528981393223, 4.1327492375026225, -1.570795258620601, -0.49525016220060986, 0.41473081499976927, 1.5707963366207034, -4.457418631705929e-08, 2.3561945046458366, 3.315092244130627, 2.7845155498525997, 1.5707956685504603, -3.695167032757759, 0.5535743368837128, 3.0353930209161213, 3.2290247644615047, 3.141593314522116, -4.712388044827314, 1.5707972283685192, 0.10619964366773368, -0.14508591852647676, -3.1415926653008492, -1.5707954502516588, 1.5707956116044257, 2.706014689394058, -2.688066551144811, -6.28318524895105, 1.3389725363834535, -1.802620167471479, -1.1352183274970655, 2.7717964632599497, 3.1415926609823797, -0.7853981567504279, 2.49543834813478e-08, 0.8481806306466274, -3.3762448760341632, -8.78189023303846e-09, 3.141592318547921, 1.5707963417455821, 3.864208357436766, 0.15092239646879113, -3.141592564518023, -1.5707964763928095, -1.5707963627489616, 7.853980727222223]),
        (10, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)]], [2.3561944878861287, 3.1415923947093494, -3.141592184336502, 3.1415908314305363, -1.5707956578134699, 1.570855929270299, -2.573706817899682, -4.712389367739198, -4.712389561832005, -2.5270112036214067, -1.5705434196123782, 2.846881674581769, -1.3532840002823923, 3.8263083056623106, -0.04835832753869209, 3.19951952527812, 1.2487928600104643, 3.114553072258967, -1.3360715519986894, 1.5707998845272089, 0.9344354449806324, 0.8982882847057544, 1.5708142543213712, -1.5013048288961097, 1.8632285977378102, -4.712389084507961, -2.4070666844168005, 2.064259536254355, 3.6951670022960545, -0.0695482709197509, 0.7067350235883831, -1.5802353686965398, -2.2641979383651205, -0.4934634914559264, 1.017221946339472, -0.5437002031914812, -0.4830846309605834, 0.5307046873532834, -1.6119205150727725, 4.712371911481876, 2.1243707351009586, 3.685293029614024, -2.2817580437786624, -2.0920621972272606, 5.497787158158455, 5.497787129666409, -2.5880183156910457, 2.9008427320922774, -1.6106443752690833, -3.141597693739356, -3.4314125133559825, 2.6460143681778256, 2.4292484769698076e-07, 0.24074971069141587, -3.325869435410485, -3.1415976949244047, -1.5707786293536443, -1.5708133845515035, -3.141592933705808, 1.5361888796922292, -2.8820754359989125, 0.06819865413462754, 4.712405843950528, -1.5707805189409199, 2.4915012933328176, 1.6028165138026813, -1.6794538228545066, -3.141590802657933, -0.20758501739049517, -0.28981988286075494, -0.920704806112774, 2.325228814038373, -1.462140932163623, -3.141590964725394, 2.853599368411741, 2.5172829242651904e-07, -1.5707963232023252, 2.3897477456764116, 1.2289147339409427, 0.06819511863569508, 3.1415757210151196, -4.71238900204379, -3.0850341634745203e-08, -3.187996633479284, -1.6183060039153576, -1.8693170139833828e-07, 1.5707798878712402, 1.5707955471989565, -1.5707963294848848, -1.524392344967457, -5.969034373834367, 3.1415925079140052, 3.1415927299369977, -3.1415919445253695, 4.712389010724299, -1.5708157598196497]),
        (11, 1, False): ([[(0, 1)], [(0, 1)]], [-1.359645604477575, 0.8260018033669561, 1.2645189584098586, -1.7943326514547893, -0.8022157877797601, 1.5707963266832448]),
        (11, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [2.011306988958161, -3.1415926318810947, 1.6814398459649167, 1.6099520768357325, 1.5707963297197503, -3.252236163950751, -2.74023773793161, 0.30627736934417904, 2.848058951171418, 1.4461899016315984, 1.5707963214559186, -1.5835399954198532, 1.6954027605556699, -1.570796326608598, 1.5707963309386355]),
        (11, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.021329081381551, 2.8290014786839912e-09, -1.570796326701034, 1.5707963268597185, 1.2795844409178376, 1.5707963255610202, 2.835315278396422, -0.28871878935973433, -2.3009135252520214, 2.6227376093244152, -1.5707963267203822, 1.5532377445952181, 2.7795191980509846, 1.5245085568885963, 1.570796319904736, -4.042721066966055, 0.3620734558914699, -3.187880430291581, 3.1415926535778778, 1.9067818890939905, -2.9766722287797878, -2.182226958184388, 1.5707963277694967, 0.966661463897588, 2.9766722331842748, -6.974423698891025e-09, 1.5707963268131797, 0.03899204682003281, -3.257729283023431, -1.454295564333775e-08, -1.5707963336267938, -2.073510612697557, 0.11613663011497027, -1.5707963264988842, 1.5707963269056788, -2.6388783671939318]),
        (11, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.706961974618802, -3.894353077411057e-09, 3.141592570107976, 1.5707964261881553, 3.141592579883285, -0.3472881482206599, 3.141592651246574, -2.160695581871293e-06, -3.1415903903700544, -2.712794339308213, 2.3820729011763384, 1.570796300183105, -1.5707941459898127, -4.712391437016875, -0.7350756882210764, -0.16399172860436426, 2.592125413312606, 4.16292173603521, -1.5707951503509257, -0.48751355997914725, 0.41473081650836857, 1.570796330677729, -4.37338650655677e-08, 2.356194487781915, 3.3228288463520856, 2.784515551361203, 1.5707956684433286, -3.7067354825827823, 0.5651428400304933, 3.035393016001225, 3.2290247659701143, 3.1415933144149846, -4.7123879718121575, 1.5707972855154007, 0.10619963875284197, -0.14508591701787857, -3.1415926652462183, -1.5707953713967948, 1.5707955613071873, 2.7060146892837738, -2.6880665496362095, -6.283185248896408, 1.3505409961829453, -1.7910516591388006, -1.1352183276073415, 2.7717964647685522, 3.1415926580521236, -0.7853981688222343, 2.6268186312393338e-08, 0.8481806285098648, -3.376244874525564, -1.1712150765003647e-08, 3.1415923008440814, 1.5707963221931605, 3.8642083553000033, 0.1509223979773943, -3.1415925630840813, -1.5707964810932735, -1.5707963661223339, 7.853980651896173]),
        (11, 5, False): ([[(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)]], [-0.8309155496707723, -3.14159264874557, -1.5707962961909459, -0.13530641728850568, 3.1415926356329806, 1.570796254313455, -0.18291428316265804, -4.220135125911035, -3.1415926888872403, 1.4354906998883485, 0.2568792369404802, -0.3322273734259396, -2.543220312302015, -3.453359766464939, 0.20524631769888457, 2.756319188257261, -3.8236828649142947, -1.2385688591523958, -3.4555610920938706, -0.611744389426383, 1.953213809215219, 1.461749851429764, -1.2248375732985128, -3.3768362088416826, -0.10148922991601593, -0.1684707603822693, -2.737345181306518, -2.362126237863659, 1.9344055907584006, -2.906349104074167, 1.0190449841999305, 5.169862598575636, -0.785398170340539, 3.141592667216413, -3.8431178460690174, -0.9584083307090625, -1.434502922160717, 3.1415924215863753, 1.8455299417374504, 0.903952987525953, -1.652238624207045, 1.6973422594889263, -1.2268013168343643, 2.3873663121744843e-07, 2.318894812611401, 1.5707963237934905, 1.4893537898834939, -0.7287306465782484, 0.8113433075911567, 1.5707962829031021, 2.3188947188010256, -1.5707963896493211, -3.0891009640404264e-08, -0.7956014254523863, 1.8487927043213566, -4.712388949235682, 5.223053540680872, -3.8084360015541283, -1.5707963483902028, -1.9412474521777408, 0.542609627895333, 3.6651912696034286, -1.5707963055545504, 1.036083732061563e-09, -1.5707963318400449, -3.5565396905729716, -3.485902633983327, 0.52359882807049, -2.496131145073723, 1.5707963147683972, 3.1415926597060366, 1.664067793545077, -2.0470923564702495, 0.5235988690017701, 1.758147110476161, -1.5707963154110427, 1.5707963166823404, 3.0483211996893274, 1.767390442007569, 1.5707963163823224, 1.8210384961049617, 2.4881318755367976e-08, 3.9269908843238626, -2.652841905130788, 1.3742022082206091, 3.027035184719727e-08, 4.12981878882625, -1.5952832997666153e-08, 2.35619455956882, 1.0820455894174656]),
        (12, 1, False): ([[(0, 1)], [(0, 1)]], [-1.3689807695775276, 0.8249626574879323, 1.2779535573016527, -1.7844566110680178, -0.803247374204487, 1.57079632786389]),
        (12, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.9913306575342296, -3.141592637694843, 1.6814398423111587, 1.5999639099265046, 1.5707963336038868, -3.2522361676045097, -2.7502259048408386, 0.29284276322944336, 2.8547762507800094, 1.4461898872978378, 1.570796320027087, -1.576822695811264, 1.6954027462219092, -1.5707963273065848, 1.5707963274798569]),
        (12, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.0471975482817304, -2.287737980266337e-09, -1.57079632749725, 1.5707963245730612, 1.2666502061733889, 1.5707963258621218, 2.848749878728925, -0.2820014888088393, -2.3138477599964773, 2.6127474227496013, -1.5707963287353992, 1.5599550451461555, 2.7795191969331148, 1.5245105618264185, 1.5707963297285719, -4.047715136598987, 0.36207345477355757, -3.1878784151126136, 3.141592652116615, 1.9017878194610576, -2.9766722303379365, -2.1922130799254633, 1.5707963230135562, 0.9616673732896535, 2.976672231626051, -6.09719643252126e-09, 1.5707963221037264, 0.033997956212165864, -3.257729283038919, -3.96343984307873e-10, -1.570796328951715, -2.073510612722336, 0.11613663009946576, -1.570796325693153, 1.5707963314181423, -2.6388783672187195]),
        (12, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.6910594063830375, -5.477513659996493e-09, 3.1415925621600835, 1.5707964581286302, 3.1415925806378073, -0.36319071645643675, 3.1415926496634135, -2.168643472707877e-06, -3.141590404999487, -2.7060770349859897, 2.366170333061409, 1.570796291775319, -1.5707941539377037, -4.712391451646304, -0.7283583838988495, -0.17989429671929097, 2.617993879104451, 4.188790203595653, -1.570794985724501, -0.48079626099277095, 0.4147308169209157, 1.5707963269355776, -4.4788989185899815e-08, 2.356194487787165, 3.329546145338457, 2.784515551773751, 1.5707956700978873, -3.716723648884822, 0.5751309945286572, 3.0353930130226243, 3.2290247663826666, 3.141593316069539, -4.712387847151195, 1.5707973998850349, 0.10619963577424377, -0.14508591660533052, -3.1415926668049154, -1.5707952331658044, 1.5707954619736897, 2.7060146867837336, -2.6880665492236564, -6.2831852504551104, 1.360529156912404, -1.7810634933033391, -1.1352183301073728, 2.771796465181097, 3.1415926618499728, -0.7853981612787734, 2.7316325796385587e-08, 0.8481806225666289, -3.3762448741130195, -7.914304497039984e-09, 3.141592261897922, 1.5707963353376264, 3.8642083493567707, 0.1509223983899404, -3.1415925438821675, -1.570796493092498, -1.5707963686525677, 7.853980519151064]),
        (12, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (4, 5)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (4, 5)], [(1, 2), (3, 4)]], [0.7297276214303231, -3.1415926964056817, -3.141592632581924, 3.1415926117983703, 1.570796327461837, -0.42053432729754386, 1.7009731833665447, -1.5707962861131808, 1.5707962766462642, 3.1415926071048728, -1.5707963291865261, -1.433333789931961, 2.2400443828197227, -3.7570723317117296, -2.1862760334646048, -4.712388896964051, -4.4195462020865595, 3.0041301347034466, -1.2599525707329402, 3.1415926288471367, 1.5707962938751716, -1.047197511083736, -4.291854641037643, -1.1549806730327914, -1.9513373712140687, 3.1415926447921314, 2.094395204780384, -3.555639915270491e-09, -1.2897614211554367, -3.557408291730901, 7.853981287151208, -1.047197505852707, -2.0943951436485446, -3.1415926493431687, 0.9113047361716651, 1.9634954457812552, 2.7184697893999066, 3.1415926308293924, -1.5707963108230623, -2.0152041803806497e-09, 0.0069053907271297505, -5.204826738562552, 3.819585396394412, -0.2548698833796674, 1.5707963254163886, 3.1415926573644097, 3.0087806612632573, -5.356333430194041, 1.8956843836963504, -1.570796029483958, -1.5707963517066186, -0.7853981962945406, -8.288766909104626e-09, -1.8006537253777737, -1.8956850565441323, -4.712388973414433, -1.5707963337775053, 1.5707963515915586, -1.658846373024138, 0.22985739158528712, -1.0924144692307394, -2.8867227745320707, -3.1415926610168197, 3.1415926385393296, -4.419546201483893, 1.8294570075813898, -2.304048062529405, 3.5211155275067313e-09, -2.356194590400482, 8.462844655062569e-09, 0.8271131774344465, 1.3121356606969596, -2.7411970990424175, 1.5707963776837681, -5.497787036678776, 4.218195840076371e-09, 1.5707963067214656, -1.214362712964122, -1.9711921266310461, -1.5707963051271727, -1.5707960999523884, -1.2897614050855222, -1.65884637654572, -0.3564336319084016]),
        (13, 1, False): ([[(0, 1)], [(0, 1)]], [-1.3771773061262595, 0.8240902780048295, 1.2897614253721392, -1.7757845273852257, -0.8041139037868821, 1.5707963268513274]),
        (13, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.9738534009759419, -3.141592645880443, 1.6814398413672589, 1.5912252801620637, 1.5707963321724288, -3.252236168548411, -2.758964534605281, 0.2810349104231624, 2.8606801845771246, 1.4461898869799221, 1.5707963256711486, -1.57091876201415, 1.6954027459039962, -1.570796331870021, 1.5707963254705688]),
        (13, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.0697033044903945, 3.792591564937326e-09, -1.5707963265962894, 1.570796327328663, 1.2553973271529548, 1.570796325844533, 2.8605577819858334, -0.2760975496301068, -2.3251006390169096, 2.6040103408385735, -1.5707963271121428, 1.5658589843248776, 2.7795191967633657, 1.5245090042691591, 1.5707963351478016, -4.052084443859599, 0.36207345460385765, -3.187879967700934, 3.141592653891526, 1.8974185122003535, -2.976672238666549, -2.200953235344855, 1.5707963221747816, 0.957298049221968, 2.9766722232973817, -2.5021470692013266e-09, 1.570796326373183, 0.029628632144397263, -3.2577292818607644, 5.154887481780438e-09, -1.5707963309927835, -2.0735106122319857, 0.11613663127765565, -1.5707963263471192, 1.5707963260917452, -2.6388783667283744]),
        (13, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.6773208530339145, 1.2682982890411549e-08, 3.1415925646608476, 1.5707965118983673, 3.141592598210804, -0.37692926980555963, 3.141592667823922, -2.1661427091417303e-06, -3.1415904113774507, -2.700173102005489, 2.3524317799656966, 1.5707962739141528, -1.5707941514369395, -4.71239145802426, -0.7224544509183468, -0.19363284981500714, 2.6404996402862353, 4.211295966015764, -1.5707948595391765, -0.47489232542529136, 0.4147308131846311, 1.5707963275998482, -4.293070280792139e-08, 2.3561945015245973, 3.3354500809059333, 2.7845155480374575, 1.5707956613927203, -3.7254622680859493, 0.5838696139815004, 3.0353930061630097, 3.2290247626463784, 3.141593307364384, -4.712387750824551, 1.5707974777465377, 0.10619962891463054, -0.14508592034161777, -3.1415926620313672, -1.5707951295440614, 1.5707954080764288, 2.706014674857204, -2.688066552959937, -6.283185245681551, 1.3692677883639373, -1.7723248653460841, -1.1352183420339006, 2.7717964614448083, 3.141592661782931, -0.7853981518441354, 2.5817397056172697e-08, 0.8481806255821893, -3.3762448778493126, -7.981357733262099e-09, 3.1415922427953533, 1.5707963265214877, 3.864208352372326, 0.15092239465365573, -3.1415925563301137, -1.5707965118627445, -1.5707963610978273, 7.853980420793716]),
        (13, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)]], [2.239760391705169, 3.1415927593448285, -3.141592805419368, 3.14159277100287, -1.5707951475483455, 1.57077976911738, -2.6319238658359128, -4.7123889627969415, -4.712388876878978, -2.5270113950202484, -1.5501855921837522, 2.8672395058805678, -1.411501048218879, 3.7295966051524188, -2.3243603189864482e-08, 3.199519333879617, 1.2691506874403538, 3.1349109035589464, -1.3360709911263475, 1.5707949815069335, 0.9827937492771167, 0.8982880933070186, 1.5707907786653765, -1.501268547121479, 1.8632291586109782, -4.712388969454212, -2.4070655360505593, 2.064260156962968, 3.725462312856251, -0.06951198914472917, 0.7067355844599894, -1.5802352536418323, -2.264196789999132, -0.49346287074711764, 0.9869267201205586, -0.5437003779910207, -0.48308407008786575, 0.5307048024079918, -1.6119193667066334, 4.712391308186535, 2.154665924926349, 3.685292854812955, -2.2817574829059457, -2.0920620821726525, 5.497787108415857, 5.497787149527884, -2.557722994341794, 2.900842779791413, -1.6106438143966484, -3.141590795252503, -3.392139195860771, 2.606740993884004, 2.261360278630306e-07, 0.2407497583894689, -3.3183598219150072, -3.1415907757947332, -1.5707855192247462, -1.570793410135262, -3.141592794196774, 1.5361888307721436, -2.8745658225033353, 0.06926541292992588, 4.7123868476062825, -1.5708000738245094, 2.491501318158994, 1.60281646488206, -1.6794522795619042, -3.141593227697131, -0.2272216805577288, -0.25054650986297944, -0.9207047812847805, 2.32522876511957, -1.4621393888718903, -3.1415933897646924, 2.8339627052452165, 2.094972572753169e-08, -1.5707963157749616, 2.389747696760189, 1.2239082959058731, 0.06926672499959817, 3.1415956136883563, -4.712388973098454, 4.813492633287407e-09, -3.187996631605237, -1.623312441948872, 1.9035578602163647e-08, 1.57078417983729, 1.5707962003947404, -1.5707963179039361, -1.5243923430933326, -5.974040811868812, 3.1415927138815363, 3.1415934455584265, -3.141592651391824, 4.712389001295918, -1.5707921161240779]),
        (14, 1, False): ([[(0, 1)], [(0, 1)]], [-1.3844494436241623, 0.8233474891201374, 1.3002465657025088, -1.768089972335896, -0.8048520733355694, 1.5707963266214846]),
        (14, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.9583930138198578, -3.1415926445978886, 1.6814398425548254, 1.5834950867006599, 1.5707963300638148, -3.2522361673608464, -2.7666947280666867, 0.2705497653394672, 2.8659227553038273, 1.4461898927569714, 1.5707963269861747, -1.5656761912874475, 1.6954027516810468, -1.5707963299206738, 1.5707963269871645]),
        (14, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.0895209518028461, -2.006126692183694e-09, -1.5707963266770442, 1.570796326625493, 1.2454885072159396, 1.5707963278283348, 2.8710428959482432, -0.2708549850093781, -2.3350094589539663, 2.59628132330302, -1.5707963266763418, 1.5711015489457338, 2.7795191971768345, 1.524507833899254, 1.570796325803807, -4.055949549059083, 0.36207345501723964, -3.1878811442399906, 3.1415926535034697, 1.8935534070008158, -2.976672230991362, -2.208684631782823, 1.5707963274028325, 0.9534329600953768, 2.976672230972561, -2.7678982455346344e-09, 1.5707963269258907, 0.025763543017837526, -3.257729283090427, 2.1150160461803413e-09, -1.570796326297687, -2.0735106127712952, 0.11613663004791681, -1.5707963268610958, 1.570796327203698, -2.638878367267597]),
        (14, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.66529087251781, -3.943714507450022e-09, 3.14159256672785, 1.5707964689960385, 3.1415925917764427, -0.38895925032166845, 3.1415926511972208, -2.1640757007360535e-06, -3.1415903989653318, -2.694930533121339, 2.3404017992110124, 1.5707962971491447, -1.5707941493699318, -4.712391445612145, -0.717211882034194, -0.20566283056968918, 2.6603172798111694, 4.231113606517928, -1.5707949799679415, -0.46964975764083444, 0.41473081670322404, 1.570796326717281, -3.8697312150268826e-08, 2.3561944900529013, 3.340692648690398, 2.7845155515560487, 1.5707956686949613, -3.7331924732711235, 0.5915998205865481, 3.035393011914538, 3.2290247661649696, 3.141593314666618, -4.712387835556736, 1.5707973958307138, 0.1061996346661606, -0.14508591682302244, -3.141592670657187, -1.5707952215304366, 1.5707954723106563, 2.706014685709386, -2.6880665494413423, -6.283185254307362, 1.376997983465929, -1.764594669923955, -1.1352183311817152, 2.771796464963402, 3.141592661602161, -0.7853981631164108, 2.5993310453960164e-08, 0.8481806258289745, -3.3762448743307174, -8.162135578127076e-09, 3.1415922798617175, 1.5707963284131068, 3.864208352619115, 0.1509223981722466, -3.1415925616887788, -1.5707964871557438, -1.5707963684466115, 7.853980506536657]),
        (14, 5, False): ([[(0, 1), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3), (4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(3, 4)], [(1, 2)], [(4, 5)]], [0.9302740077210688, 4.703700437865649e-09, -0.5907423771192686, -1.7839224002444989, 2.0228260556167323, 3.14159264287724, -2.1131118899490238, 4.712388977936777, 2.1615386749859042, 0.21312603125973364, -1.570796376038179, -0.38639412981006227, 1.1828378816808467, 3.7055352892334414, -4.1484463321537595, 0.7516326708135612, 1.9583930781105625, -2.7551985158886403, -0.12309696474647114, -4.712388972558349, -4.4351715485291345e-09, 1.1601662051175606e-07, -1.570796273000144, -2.567214575353038, 2.8082676640256694, 1.0236079366312232, -3.9716949464312377, -0.6614359515747497, -2.555002425746361, 2.4319396891561116, 0.45642200266619165, 0.547188361557319, -1.0260358590388783, 1.7348984322840886, -1.1206977792248605, 0.3445586545477685, -0.8436718267674369, -1.5886155966322343e-09, 1.3939950731446378, -4.712388895368881, -2.020894786232354, -2.7307984024132934, 3.9435847833208695, 1.133686175796428e-07, 0.32508121521296274, -1.5707962645975222, -1.8067667436453987, 1.0863975307962577, -2.721935022650626, -1.0985624875911133e-07, 0.24063769708029006, 0.8423183647351621, -2.9056222442774553, -2.86398927430525, -1.3876066138366603, 0.9688557058838178, 2.356194502226287, -5.343098055434735e-09, -1.5707963399554208, -0.6838323679376742, -0.6974276400498773, -0.8863374532916649, 3.1433502651125638, 1.5707963276571801, -5.5529495398998805e-09, -1.3326436582992733, -2.6560367180836586, -0.08251818094917478, -2.013267478039115, 1.5707963398103895, -1.5707963066340038, -0.23815266883642383, 1.1790940139469597, 1.9396984751697899, 0.2562998140726224, -3.1415926794348796, 1.4355214412548993, 0.2538754219297182, 1.4072641649127768, 1.201894114540167, 0.873783160060808, 3.14159257793732, 2.9755951552923836, -3.491565667576432, -1.8479920640559189, 3.1415925974059538, 1.0706050869188612, 0.18277215505698174, 3.442865029847501, 0.09609761018634198, 0.48313422492030805, -3.1415925989168323, -1.291327412942891, -0.1827720929053387, 1.868786643614875e-08, 4.712388896674768]),
        (15, 1, False): ([[(0, 1)], [(0, 1)]], [-1.3909589032690937, 0.8227074029892689, 1.3096389159210724, -1.761202002534714, -0.8054884477937535, 1.5707963268353513]),
        (15, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.9445885019389597, -3.141592645002856, 1.6814398435215172, 1.5765928310206765, 1.5707963289152784, -3.252236166394156, -2.773596983746668, 0.261157411072487, 2.87061893144989, 1.4461898954678418, 1.5707963284342483, -1.5609800151413846, 1.6954027543919181, -1.570796328099931, 1.5707963276240389]),
        (15, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.1071487179537243, 1.5455948895814684e-09, -1.5707963272745071, 1.5707963265213027, 1.2366746241008735, 1.5707963263700762, 2.8804352418751145, -0.26615880922548885, -2.343823342069099, 2.5893799890709848, -1.570796326755805, 1.5757977247296207, 2.7795191981697567, 1.5245069136041534, 1.5707963271443501, -4.05940067873059, 0.36207345601028196, -3.1878820672646295, 3.1415926533349072, 1.8901022773293021, -2.9766722309288514, -2.215587814934759, 1.5707963269247323, 0.949981833134305, 2.9766722310350016, 7.456143978844525e-11, 1.5707963268082143, 0.022312416056573723, -3.2577292834312033, 2.9598925862437583e-12, -1.5707963265306093, -2.073510613625593, 0.11613662970712765, -1.5707963270253167, 1.5707963262040725, -2.6388783681218735]),
        (15, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.654639103098011, -3.347747885411105e-10, 3.1415925621946146, 1.5707964507974426, 3.1415925996816814, -0.399611019741456, 3.141592654806148, -2.1686089310652687e-06, -3.141590391448078, -2.6902343576503527, 2.329750029750611, 1.5707962896954417, -1.5707941539031602, -4.712391438094886, -0.7125157065631991, -0.21631460003009037, 2.677945044929563, 4.2487413715482765, -1.570795101965015, -0.46495358276213056, 0.41473081576572857, 1.570796324257339, -3.334685523887552e-08, 2.3561944884162127, 3.345388823569105, 2.7845155506185506, 1.5707956667458336, -3.7400947338489927, 0.5985020738327836, 3.0353930123043886, 3.2290247652274693, 3.1415933127174878, -4.712387931737874, 1.5707973030874165, 0.10619963505601135, -0.1450859177605154, -3.1415926745663616, -1.5707952961965463, 1.5707955425059559, 2.706014682138693, -2.6880665503788426, -6.283185258216529, 1.383900236429805, -1.7576924125441082, -1.1352183347524234, 2.7717964640258987, 3.141592663326701, -0.7853981589601734, 2.1668792874351845e-08, 0.8481806273910713, -3.3762448752682137, -6.437598765796574e-09, 3.141592313887838, 1.570796321310567, 3.8642083541812067, 0.15092239723475506, -3.141592559768154, -1.570796466026187, -1.5707963609996447, 7.853980591663908]),
        (15, 5, False): ([[(4, 5)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3), (4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(3, 4)], [(4, 5)], [(3, 4)], [(4, 5)], [(3, 4)]], [-0.8177065888844087, -2.7015970146788635, 3.1415926311955986, 1.8025695740928373e-08, 1.5707963216967116, -0.3737921744646206, 1.7730232059361184, 5.843189695951172, -0.1652028425191549, -0.398682925885038, -0.6624315399806526, 2.596038531609348, 2.2528935992356742, -4.712389080696102, -1.4055935191365445, 0.3986829063562616, -0.9083647915761278, -0.17732909567572053, 1.8440156722087453, -0.5426391534837175, -2.1134354482577136, -1.5707965423874286, -1.3096389363671874, -0.8479131470466658, 0.643336066221612, 1.5707962913657814, 3.1415926585681553, 0.23182381710478095, -1.8582235129038895, 2.6433943232480197, -2.5357238172284444, -2.870273122820647, 3.3734164503006916, 3.141592652547295, -3.141592628525677, 1.2791422619303905, -0.6283644650047022, 1.2994767585606044, -4.712388902403707, -3.0794055341899853e-09, 3.1415926324157737, 0.9514591358502199, 0.4659877290136949, 3.141592393173447, 1.5707963186644842, 3.141592848963851, 2.4803734297992857, -3.303199390913076, -1.0974737851019365, -3.141592521621844, -3.5156488237420764, -4.4805651722240425, 1.4492992201255292e-08, -2.68541322786826, -3.7661551607254293, -3.1415926469448516, -0.9649163373882843, -5.673179505968342e-09, -0.7853982025595831, -3.5977720543081944, -0.20103105680733058, 0.00566525892362839, 3.458478381919039, -0.7853980841129313, 1.8234765594374132, -3.7009219906980686, 0.44601767585196855, 3.1472595847571725, -1.8876816985093, -1.5707962824646968, 3.14159267239086, -1.0114668859642686, -1.160233675038211, 3.1302606357353047, -3.201647316350281e-07, 3.1415927116663735, -1.8319537160934243, 0.5704084644253993, -1.5911375092817188, 3.1415925664553312, 2.1126305180918283, 4.965069350032525, 0.8187562870752721, -2.8932449417118513, 0.24134652715752006, 3.1415944247997127, -1.145610437996983, 1.3181159251605212, 1.5707963128319231, 0.2850191275037668, -0.2413465065767634, -1.6673346326674416e-06, 2.1745725749395737, 1.8234765617596147, 3.14159279361948, 1.2857772320763805]),
        (16, 1, False): ([[(0, 1)], [(0, 1)]], [-1.396830358471015, 0.8221500918172666, 1.3181160716581939, -1.7549888182555455, -0.8060427320206347, 1.5707963268288248]),
        (16, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.9321634509001153, -3.1415926461652663, 1.6814398434627535, 1.5703803060840011, 1.5707963283240325, -3.252236166452919, -2.7798095086833428, 0.2526802553120026, 2.8748575088452277, 1.446189896443132, 1.570796327764351, -1.5567414377460476, 1.6954027553672055, -1.5707963275172632, 1.5707963276568375]),
        (16, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.1229639300019953, 1.6643459389793535e-09, -1.5707963267602623, 1.5707963270019494, 1.2287670155715356, 1.5707963262610816, 2.888912394141669, -0.26192023893730654, -2.35173095059849, 2.5831681419835917, -1.570796326915663, 1.5800362950179045, 2.779519197733498, 1.5245061335700483, 1.5707963384893602, -4.0625068932182815, 0.3620734555740693, -3.1878828504170795, 3.141592653713425, 1.886996062841625, -2.976672230297107, -2.221801023374271, 1.570796340697235, 0.9468755217725572, 2.976672231666839, 1.677174361219939e-09, 1.5707963266484604, 0.019206104694649815, -3.257729283003, 4.5309786749489524e-08, -1.5707962890177842, -2.073510612781338, 0.11613663013523695, -1.5707963266501128, 1.5707963264994007, -2.638878367277598]),
        (16, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.645118933308322, -1.477318811944053e-09, 3.141592567850351, 1.5707964180554952, 3.141592600267703, -0.4091311895311347, 3.141592653663595, -2.1629531981176094e-06, -3.1415903845126554, -2.685995788211049, 2.3202298594634825, 1.5707963006272674, -1.570794148247429, -4.712391431159456, -0.7082771371238934, -0.2258347703172227, 2.6937602547360853, 4.264556571150523, -1.5707951857654168, -0.4607150028987943, 0.41473081733816386, 1.5707963156556892, -2.97430249611427e-08, 2.356194480612328, 3.3496274034324376, 2.7845155521909764, 1.5707956683524036, -3.7463072653266565, 0.6047146309836249, 3.0353930159738796, 3.229024766799899, 3.1415933143240515, -4.712387973005604, 1.570797259396153, 0.10619963872550103, -0.1450859161880789, -3.141592673504324, -1.5707953462610331, 1.570795587076018, 2.70601468189037, -2.688066548806405, -6.283185257154494, 1.3901127856007132, -1.7514799190388264, -1.1352183350007463, 2.771796465598324, 3.141592652616904, -0.7853982204624363, 2.020315216436058e-08, 0.8481806373459261, -3.376244873695777, -1.7147408064638986e-08, 3.1415923388305855, 1.5707963035523556, 3.8642083641360636, 0.15092239880718625, -3.1415925763605, -1.5707964594768875, -1.5707963592641507, 7.8539806373896]),
        (16, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(1, 2)], [(0, 1), (2, 3), (4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(3, 4)], [(0, 1), (4, 5)]], [1.5707963326882795, 4.712389047095583, -3.1415926452127483, 6.2831853040388985, 1.5707963563564342, -4.783811280204081e-09, 3.5410713498063195, 3.72282411494036, 3.3225742897034602e-09, 4.712388973286948, 1.570796350484416, 7.926473456288015, 0.2829319381530003, 2.9994116002848967, 6.283185308958077, 5.983530568273209, 3.141592521276496, 4.387216918091621, 4.2704728697538865, 2.2365761539337052, 6.283185299519427, -1.7638035390971722, 4.760323472256777, -2.2617195627903883, 1.784299579685881, 1.967161416031111, 1.5707962520778966, 0.4926619421765004, -2.5848125805105697, -0.6271928122363309, 0.12087396706902688, -1.1043679715590051, -2.989151808546806, 1.269876456859845e-08, -2.5368781142092605, -2.369258182661241, -0.5615255646076576, -2.6317147000336902, 1.4183554361192523, 5.8353529057307005, 0.6047145381168522, -0.7723344367093948, -0.8812162636700113, -4.117898901405563, 1.5707963636094449, 0.5614819834295693, -8.45869622585598, 1.1421105688565083, -0.2489285077501784, 8.639379773612129, -1.5707963059384635, -0.5614819516201521, 1.570796318693671, 3.2561191059865884, -1.8521221884109451, -7.068583494626183, 6.283185333530639, 2.5801106881387836, -1.5707962761777956, 2.214336836334175, -6.903165300742154, 3.926990843395746, 17.27875958700634, 7.292499662692995, 7.853981693757997, 3.3062635297736596, -3.811083124350612, -12.566370628485393, 6.283185304980066, 3.141592608319103, -1.5707963376074365, 1.0767442810052712, 4.0943289311463, 10.21017612471036, 7.85398163137902, -4.712388985741156, -4.172779661368957e-09, -2.1809028321306854, 3.759652656652197, 1.4336106026124183e-08, 1.4809381927529568e-08, -4.712389006514364, 1.5707963272550407, 3.7516991442834615, 4.712388941807937, -3.141592661743596, 3.1415926296421754, 2.480595913180179e-08, -1.5707963287764986, 3.141592681320604]),
        (17, 1, False): ([[(0, 1)], [(0, 1)]], [-1.4021616902902747, 0.821660470626402, 1.3258176636612564, -1.7493469367106214, -0.8065298522222192, 1.570796326813298]),
        (17, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.9209021045154797, -3.1415926489371904, 1.6814398422614074, 1.5647496325361754, 1.5707963282076478, -3.2522361676542624, -2.7854401822311674, 0.24497866224428375, 2.8787083054975815, 1.4461898952475947, 1.5707963280335104, -1.5528906410936945, 1.695402754171667, -1.5707963282626252, 1.5707963269764054]),
        (17, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.137258417342416, -6.853522956645567e-10, -1.5707963267730254, 1.5707963266672782, 1.2216197744822295, 1.5707963271590268, 2.896613988894931, -0.25806943520675096, -2.3588781916878214, 2.5775381099058405, -1.5707963267162597, 1.5838870987486124, 2.7795191975892792, 1.5245055945577886, 1.5707963275509673, -4.065322278115665, 0.36207345542962704, -3.187883386898151, 3.1415926535073004, 1.8841806779443488, -2.976672230731083, -2.227432333191875, 1.5707963270603693, 0.9440602338608128, 2.9766722312328207, 3.2091526288604783e-10, 1.5707963268763834, 0.016390816782893666, -3.257729283226966, -4.0620283617439016e-10, -1.5707963264972058, -2.0735106129219436, 0.1161366299112451, -1.5707963268440848, 1.570796326979268, -2.6388783674183194]),
        (17, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.636541872123857, -1.113609857849455e-09, 3.1415925642741063, 1.5707964029988917, 3.1415926092250426, -0.41770825071557743, 3.1415926540272876, -2.1665294514681163e-06, -3.1415903816093738, -2.6821449817711867, 2.3116527986980944, 1.5707962952733754, -1.57079415182368, -4.712391428256173, -0.7044263306840302, -0.2344118310826128, 2.7080547436294924, 4.2788510673174835, -1.570795270045712, -0.45686420536559696, 0.41473081621904995, 1.570796322024211, -2.6477307882876094e-08, 2.3561944727895776, 3.353478200965639, 2.7845155510718658, 1.5707956672142944, -3.751937919551934, 0.6103452821549744, 3.035393014139828, 3.229024765680778, 3.1415933131859273, -4.712388049764459, 1.5707971834987766, 0.10619963689143713, -0.14508591730718973, -3.141592678075722, -1.5707954062008846, 1.5707956404623278, 2.706014681035412, -2.688066549925515, -6.28318526172588, 1.395743426895111, -1.745849211771274, -1.1352183358557009, 2.7717964644792104, 3.141592659829417, -0.785398156149669, 1.66765851059918e-08, 0.848180624762178, -3.376244874814885, -9.934902148709628e-09, 3.1415923649150153, 1.5707963294942968, 3.864208351552305, 0.15092239768807728, -3.141592568200231, -1.5707964436435844, -1.5707963501067888, 7.853980703789778]),
        (17, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)]], [2.143999638388394, 3.141592410220003, -3.1415925743859945, 3.1415927589564143, -1.5707887962887586, 1.5707980911902344, -2.679804243340385, -4.712389232923743, -4.712389334422656, -2.527011372286997, -1.5321574664439317, 2.8852676235339607, -1.4593814257230868, 3.6480345543559127, 0.040778956282478636, 3.1995193566134805, 1.2871788131795088, 3.152939021212232, -1.3360712891677025, 1.5707990691785167, 1.0235727288055578, 0.898288116040663, 1.5707894257444237, -1.5012778759330998, 1.863228860568266, -4.712389066505099, -2.4070664225624134, 2.0642598527400136, 3.7519379910338886, -0.06952131795640772, 0.706735286419023, -1.58023535069276, -2.264197676511219, -0.49346317497005865, 0.9604511120042176, -0.5437002843044406, -0.48308436813009875, 0.5307047053570554, -1.6119202532188104, 4.712388262616848, 2.1811415446187814, 3.685292948499011, -2.2817577809469936, -2.0920621792237672, 5.497787170637552, 5.497787126262995, -2.5312473145763574, 2.9008422322518754, -1.6106441124379491, -3.141596520593061, -3.358361583216086, 2.572963438954181, 9.222543283235522e-08, 0.24074921085148576, -3.323615429609295, -3.1415965141707747, -1.5707917235286744, -1.5707968136195392, -3.1415927674399184, 1.5361892333888305, -2.8798214301971092, 0.06879183016453375, 4.712391101065712, -1.5707964438383737, 2.491501259892125, 1.6028168674992667, -1.6794530229132572, -3.1415912133424753, -0.24411047861752458, -0.21676897163765727, -0.9207048395518445, 2.325229167736953, -1.4621401322234966, -3.141591375410024, 2.8170739071847595, 3.388522825854958e-08, -1.5707963303669203, 2.3897480993757036, 1.2274120557028123, 0.06878908963775511, 3.1415907734797974, -4.712388986983394, -2.8231821314582524e-08, -3.1879966391023613, -1.6198086821530382, -7.40898662106572e-08, 1.5707916134332813, 1.5707963414433381, -1.570796336543584, -1.5243923505909571, -5.970537052073158, 3.1415926207556044, 3.141592595135328, -3.141592736446672, 4.7123889760593425, -1.570796411176397]),
        (18, 1, False): ([[(0, 1)], [(0, 1)]], [-1.4070309611634295, 0.8212269113847128, 1.3328552033657557, -1.7441938193090516, -0.8069613214331393, 1.5707963253554789]),
        (18, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.9106332375224038, -3.1415926499337123, 1.6814398420416712, 1.559615198181439, 1.5707963279500055, -3.252236167873998, -2.790574616585905, 0.23794112468438214, 2.882227075299116, 1.4461898954337506, 1.5707963292861733, -1.5493718712921591, 1.6954027543578263, -1.5707963281149033, 1.5707963268803136]),
        (18, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.150261991323049, 2.9702124526020387e-10, -1.5707963265996079, 1.570796327031829, 1.2151179876861893, 1.5707963265128873, 2.9036515275975097, -0.2545506687983135, -2.365379978483986, 2.5724041337103083, -1.570796326881318, 1.5874058651570953, 2.779519198005869, 1.52450509300303, 1.5707963288757387, -4.06788947316738, 0.36207345584616657, -3.1878838895085333, 3.1415926537773555, 1.881613482892856, -2.9766722311579916, -2.232567226679011, 1.5707963318666318, 0.941492995343581, 2.976672230805947, -2.492982806397426e-09, 1.5707963267320582, 0.013823578265710863, -3.2577292834076963, 1.9490969672714922e-08, -1.5707963071892574, -2.073510612639531, 0.11613662973054499, -1.5707963266161318, 1.5707963269097234, -2.6388783671358396]),
        (18, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.62876089082273, 3.207143727180451e-10, 3.141592563581479, 1.5707963930512265, 3.1415926132452694, -0.4254892320169296, 3.1415926554617153, -2.1672220499150302e-06, -3.141590372524593, -2.678626214663134, 2.303871817248053, 1.570796296726074, -1.5707941525163147, -4.712391419171327, -0.7009075635759452, -0.2421928125325318, 2.72105831832851, 4.291854644996096, -1.5707953418704574, -0.45334543866000415, 0.414730815935805, 1.5707963269479945, -2.370638594891855e-08, 2.356194490316906, 3.3569969676711406, 2.7845155507884134, 1.570795665612129, -3.757072362414367, 0.6154797081994632, 3.0353930153592836, 3.2290247653976105, 3.141593311583848, -4.71238810134235, 1.5707971352574008, 0.10619963811097126, -0.14508591759042166, -3.1415926783367416, -1.570795454516797, 1.5707956783320547, 2.706014679237488, -2.688066550208643, -6.283185261986906, 1.4008778718715453, -1.7407147815004738, -1.135218337653684, 2.771796464195714, 3.141592661751455, -0.7853981631824641, 1.4939738616182514e-08, 0.848180626709212, -3.376244875098048, -8.012930238985273e-09, 3.141592386093584, 1.570796326856873, 3.8642083534992877, 0.15092239740483854, -3.1415925677925705, -1.5707964271594188, -1.5707963464884251, 7.853980754630019]),
        (18, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)]], [2.1259175090880995, 3.1415927660743885, -3.1415928144969154, 3.1415923381729813, -1.5707903990877186, 1.5707881937914514, -2.6888453089212776, -4.712388899126223, -4.7123888247286825, -2.5270118255635077, -1.528638703398775, 2.88878639665368, -1.468422491304016, 3.632475691325994, 0.0485599346329109, 3.199518903337444, 1.2906975762248016, 3.156457794331338, -1.3360712417146272, 1.570795964114473, 1.0313537071561634, 0.8982876627646335, 1.5707891217981789, -1.5012729176453454, 1.863228908021797, -4.7123889714604426, -2.407064900417929, 2.0642603983921557, 3.7570723607054157, -0.06951635966853904, 0.7067353338725699, -1.5802352556488906, -2.264196154366971, -0.49346262931794366, 0.955316605584988, -0.543700382150284, -0.48308432067667895, 0.5307048004009658, -1.6119187310745455, 4.712386737834227, 2.186276036899597, 3.685292850653054, -2.2817577334937336, -2.0920620841796898, 5.497787147888042, 5.497787138774169, -2.5261129475221407, 2.9008424039483076, -1.6106440649845013, -3.141592132099869, -3.3518598296625024, 2.566461666258846, 2.3763253865867652e-07, 0.24074938254755515, -3.323836777942886, -3.1415921385189267, -1.5707699416141656, -1.570796229659382, -3.141593006598407, 1.53618907907868, -2.880042778530744, 0.06877210252509998, 4.712389259583628, -1.5707982071541544, 2.491501367840538, 1.6028167131892337, -1.6794530434657111, -3.1415927520040103, -0.24736136651795773, -0.21026716604704326, -0.9207047316034975, 2.3252290134267137, -1.462140152775915, -3.141592914071555, 2.8138230192840257, 1.9477664212958315e-07, -1.5707963106445366, 2.3897479450653134, 1.2275595986603491, 0.06877245580414194, 3.1415933412511112, -4.712388968126655, -1.157594023235437e-08, -3.1879966323872218, -1.6196611391953892, 3.4313954214484555e-08, 1.5707681223083487, 1.57079640954849, -1.570796315520519, -1.5243923438756317, -5.9703895091152015, 3.1415927291590386, 3.1415940801955187, -3.1415928445660124, 4.712389021086119, -1.570794672084388]),
        (19, 1, False): ([[(0, 1)], [(0, 1)]], [-1.411501297412312, 0.8208403040523914, 1.3393189638948062, -1.7394627188689487, -0.807346163925788, 1.5707963259311697]),
        (19, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.9012189755992324, -3.141592650369, 1.6814398420806824, 1.5549080676576679, 1.5707963276939607, -3.252236167834986, -2.7952817471096765, 0.23147736386476733, 2.8854589553945846, 1.4461898959561432, 1.5707963284807889, -1.5461399911966898, 1.6954027548802209, -1.5707963277694992, 1.5707963269325345]),
        (19, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.162158472215834, -1.407527607884688e-09, -1.570796326220397, 1.570796327272357, 1.2091697476053875, 1.570796326403748, 2.910115289834435, -0.25131878270635666, -2.371328218564624, 2.567697453067902, -1.5707963269268959, 1.590637751248939, 2.77951919896617, 1.5245047450146954, 1.5707963203750903, -4.070243091575727, 0.36207345680662256, -3.1878842374603993, 3.141592654030621, 1.8792598644844911, -2.9766722326001296, -2.2372748043401183, 1.5707963200893813, 0.9391394811666794, 2.9766722293638166, -4.8277851166014905e-09, 1.570796326792092, 0.011470064088889778, -3.257729284408622, -2.776778524083472e-08, -1.5707963475237265, -2.073510612310364, 0.11613662872960843, -1.5707963264637952, 1.570796327538948, -2.6388783668066007]),
        (19, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.62165935945674, 9.766421305982161e-11, 3.1415925672546705, 1.5707963823984248, 3.141592617022536, -0.4325907633829147, 3.1415926552386595, -2.1635488609597886e-06, -3.1415903662888907, -2.6753943221717096, 2.296770286170475, 1.5707963005476053, -1.570794148843123, -4.71239141293561, -0.6976756710845273, -0.24929434361011102, 2.7329547907902962, 4.303751118549044, -1.5707954016906793, -0.4501135539571842, 0.41473081624610453, 1.570796321170412, -2.088826864658271e-08, 2.3561944845031855, 3.3602288523739685, 2.7845155510987176, 1.5707956651397972, -3.7617795189697767, 0.6201868269054217, 3.0353930155455404, 3.229024765707903, 3.1415933111115315, -4.712388146226601, 1.5707970884410838, 0.10619963829722959, -0.14508591728012454, -3.141592678985754, -1.5707954990192856, 1.570795714499143, 2.706014679796496, -2.6880665498983545, -6.283185262635914, 1.4055849994331295, -1.7360076502534938, -1.135218337094687, 2.771796464506007, 3.1415926671642125, -0.7853981672961201, 1.3682106553394079e-08, 0.8481806115717427, -3.3762448747877603, -2.6001726977247663e-09, 3.1415924060789795, 1.5707963513281193, 3.8642083383618204, 0.15092239771513946, -3.1415925771431383, -1.5707964222725763, -1.5707963445022453, 7.853980799780794]),
        (19, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)]], [2.1094597832515576, 3.141592437815424, -3.1415926792773177, 3.141592298723076, -1.5707936253919565, 1.5707814270338227, -2.6970741700829284, -4.712389190324258, -4.712389319545677, -2.5270118978291802, -1.5254068279453676, 2.8920182651316813, -1.4766513524656255, 3.6182705966223017, 0.05566146843969931, 3.1995188310721043, 1.293929451678048, 3.159689662809036, -1.3360711910100977, 1.570798008871361, 1.0384552409626902, 0.8982875904990347, 1.5707908289592991, -1.5012693503157803, 1.8632289587259816, -4.712389059210522, -2.40706401859793, 2.0642606577273583, 3.7617796184099634, -0.0695127923388756, 0.7067353845773434, -1.5802353433990168, -2.2641952725470205, -0.49346236998320825, 0.9506096147141925, -0.5437004182582298, -0.48308426997225773, 0.5307047126509548, -1.611917849254453, 4.7123869754258925, 2.1909830437044553, 3.6852928145447446, -2.281757682789115, -2.0920621719297903, 5.497787064747109, 5.497787221228274, -2.5214056998006362, 2.90084268076123, -1.6106440142799432, -3.141595036464852, -3.3459116166111724, 2.5605133628653727, 3.2438277568665277e-07, 0.2407496593605894, -3.324023704021629, -3.141595027038177, -1.5707575475792912, -1.5707957170492066, -3.141593072142554, 1.5361888764214453, -2.880229704609482, 0.0687577306262717, 4.71239032092117, -1.5707993969459757, 2.4915014420083237, 1.6028165105321623, -1.6794527305326643, -3.1415917426722504, -0.25033545647791927, -0.2043188939847434, -0.9207046574361036, 2.325228810769833, -1.4621398398429377, -3.1415919047398084, 2.8108489293242607, 2.3693223397457294e-07, -1.5707963288881837, 2.3897477424086415, 1.2276842427756085, 0.06875606647549923, 3.1415927288529426, -4.712388970321664, 6.370662451119819e-09, -3.1879966346069852, -1.6195364950796818, -3.9121330302093886e-08, 1.5707550225428748, 1.5707963679669295, -1.5707963367961002, -1.5243923460954358, -5.970264864999724, 3.141592655723905, 3.141594620100111, -3.1415927815588836, 4.712389041311043, -1.5707935739943866]),
        (20, 1, False): ([[(0, 1)], [(0, 1)]], [-1.4156243707886076, 0.8204934126167481, 1.3452829214104252, -1.73509899227247, -0.8076915516906306, 1.5707963265629739]),
        (20, 2, False): ([[(0, 1)], [(1, 2)], [(0, 1)], [(1, 2)]], [1.8925468826055205, -3.1415926511958943, 1.681439841816826, 1.5505720208778462, 1.5707963275728969, -3.25223616809884, -2.7996177938894977, 0.22551340577989756, 2.888440934773306, 1.4461898958648296, 1.5707963292681235, -1.543158011817969, 1.695402754788907, -1.5707963278041848, 1.5707963268054226]),
        (20, 3, False): ([[(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)], [(0, 1), (2, 3)], [(1, 2)]], [1.1730969117431556, -5.05618698019837e-10, -1.5707963267538547, 1.5707963265766975, 1.2037005272290833, 1.5707963270356782, 2.916079247572796, -0.24833680673367978, -2.3767974389412507, 2.5633617115817158, -1.570796326648774, 1.5936197272220027, 2.7795191976860725, 1.524504379123497, 1.5707963271457426, -4.072411082793509, 0.36207345552610776, -3.187884601280673, 3.1415926534383196, 1.8770918732669184, -2.9766722309909652, -2.241611157309419, 1.5707963271512002, 0.9369714270865429, 2.976672230972439, -3.519403485423575e-11, 1.5707963269462875, 0.009302010009136944, -3.2577292832765017, 1.4072769005783467e-09, -1.570796325551871, -2.0735106129269063, 0.11613662986150404, -1.5707963268438485, 1.5707963271583931, -2.6388783674229863]),
        (20, 4, False): ([[(3, 4)], [(0, 1)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)], [(2, 3)], [(1, 2), (3, 4)]], [5.615143356051757, -1.9107838427853744e-09, 3.1415925765534287, 1.570796370462534, 3.141592618206295, -0.4391067667879246, 3.1415926532302114, -2.154250106480041e-06, -3.141590361274036, -2.672412354804256, 2.290254282896003, 1.5707963101085356, -1.5707941395443632, -4.712391407920757, -0.6946937037170811, -0.2558103468845722, 2.743893238649604, 4.314689564998943, -1.5707954581390686, -0.4471315791029756, 0.4147308164497065, 1.5707963271828613, -1.921270740189741e-08, 2.3561944902649032, 3.363210827228179, 2.78451555130231, 1.5707956673122134, -3.7661155407193836, 0.6245228848063047, 3.0353930170522165, 3.229024765911506, 3.1415933132839444, -4.712388178996348, 1.5707970595034528, 0.10619963980390433, -0.14508591707652344, -3.141592680267768, -1.5707955429633507, 1.5707957493769815, 2.7060146805512457, -2.6880665496947627, -6.2831852639179155, 1.4099210493723893, -1.731671603852521, -1.1352183363399382, 2.7717964647095967, 3.1415926617897907, -0.785398163048827, 1.3500474565034411e-08, 0.8481806260562521, -3.3762448745841644, -7.974580570068023e-09, 3.1415924235780692, 1.5707963282525206, 3.8642083528463265, 0.1509223979187401, -3.1415926022040233, -1.5707964215477588, -1.5707963452417653, 7.853980839001154]),
        (20, 5, False): ([[(0, 1), (2, 3), (4, 5)], [(1, 2)], [(0, 1), (4, 5)], [(1, 2), (3, 4)], [(4, 5)], [(3, 4)], [(2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(0, 1), (2, 3)], [(3, 4)], [(0, 1), (2, 3), (4, 5)], [(1, 2), (3, 4)], [(2, 3), (4, 5)]], [2.0943951094654607, 3.1415925532643842, -3.1415927548457048, 3.141592169891693, -1.5707951533478952, 1.5707775707924474, -2.704606508247201, -4.712389063011616, -4.712389025720749, -2.5270122039296568, -1.522424838878677, 2.8950002528630843, -1.4841836906301817, 3.605239194590624, 0.0621774774842041, 3.199518524971668, 1.296911440744896, 3.1626716505405903, -1.3360712805961485, 1.57079739974409, 1.0449712500074602, 0.8982872843983549, 1.5707918452051257, -1.5012673196863027, 1.8632288691401448, -4.712389007951291, -2.407063638660775, 2.064260830966303, 3.7661155558258166, -0.06951076170951609, 0.7067352949914016, -1.5802352921394138, -2.2641948926099675, -0.4934621967445474, 0.946273447797361, -0.5437004636604581, -0.4830843595584481, 0.5307047639105209, -1.6119174693173264, 4.712386179928758, 2.1953191988622973, 3.6852927691422, -2.28175777237506, -2.092062120670162, 5.4977870512636935, 5.497787234677558, -2.517069741129078, 2.90084282513934, -1.610644103865743, -3.141594162525163, -3.340442401618678, 2.555044129957265, 2.7397273562735086e-07, 0.24074980373876703, -3.324181670499133, -3.14159415275473, -1.5707507319716514, -1.5707952709567052, -3.141593156110166, 1.536188774866976, -2.8803876710867997, 0.0687445070370974, 4.712388812771856, -1.5708000012947811, 2.4915014480111943, 1.6028164089774306, -1.67945306365034, -3.1415920506021218, -0.25307006162703277, -0.1988496677248504, -0.9207046514333892, 2.325228709215539, -1.4621401729606118, -3.141592212669618, 2.808114324174802, 3.0294899710136937e-07, -1.5707963249783183, 2.3897476408540843, 1.227789537257534, 0.06874346180540658, 3.14159434490389, -4.712388987055188, 2.7272508672623742e-08, -3.1879966391795413, -1.619431200597835, 5.613310771880372e-09, 1.5707476930260773, 1.5707965558109644, -1.5707963359608785, -1.5243923506680774, -5.970159570518078, 3.1415927004588333, 3.141595119377687, -3.1415929557096884, 4.7123890322108295, -1.5707928669140505]),
    }
    # END SCS TABLE


    # ---- compilation helpers -----------------------------------------------------------------


    def _to_cz_phxz(circuit: cirq.Circuit) -> cirq.Circuit:
        """Compile to cirq.CZ + PhasedXZGate and drop global phases."""
        # CZTargetGateset already merges single-qubit runs into PhasedXZ and drops negligible ops
        c = cirq.optimize_for_target_gateset(circuit, gateset=cirq.CZTargetGateset())
        c = cirq.drop_empty_moments(cirq.Circuit(
            cirq.Moment(op for op in m if not isinstance(op.gate, cirq.GlobalPhaseGate)) for m in c
        ))
        # hardware-friendly layering: every moment is either all CZ or all single-qubit
        if any(any(op.gate == cirq.CZ for op in m) and any(len(op.qubits) == 1 for op in m) for m in c):
            c = cirq.stratified_circuit(c, categories=[lambda op: op.gate == cirq.CZ])
        return c


    # ---- dynamical decoupling --------------------------------------------------------------------

    _X = cirq.PhasedXZGate(x_exponent=1, z_exponent=0, axis_phase_exponent=0)
    _Y = cirq.PhasedXZGate(x_exponent=1, z_exponent=0, axis_phase_exponent=0.5)


    def _is_virtual_z(op: cirq.Operation) -> bool:
        if not isinstance(op.gate, cirq.PhasedXZGate):
            return False
        x = op.gate.x_exponent % 2
        return min(x, 2 - x) < 1e-9


    # Minimum free single-qubit slots in an idle window before we echo it (XY4 needs 4).
    # In simulation, echoing every short window cost up to ~1% of the ceiling share (pulse error
    # with nothing to refocus in sub-0.3 us idles); long windows are where hardware dephasing
    # and ZZ crosstalk accumulate, so only those are protected.
    _DD_MIN_SLOTS = 4


    def _add_dd(circuit: cirq.Circuit) -> cirq.Circuit:
        """Echo long idle windows of qubits that may be in superposition with X-Y-X-Y.

        A qubit is protected from its first non-virtual single-qubit gate (before that it sits
        in |0>, which dephasing cannot touch) to the end of the circuit, including the trailing
        window before measurement. Pulses go only into single-qubit moments, and only windows
        with >= _DD_MIN_SLOTS free slots are echoed. XYXY = -I, so the noiseless state is
        unchanged up to a global phase.
        """
        moments = [list(m) for m in circuit]
        one_q_layer = [all(len(op.qubits) == 1 for op in m) for m in circuit]
        for q in sorted(circuit.all_qubits()):
            busy = [any(q in op.qubits for op in m) for m in circuit]
            active = [i for i, m in enumerate(circuit) for op in m
                      if q in op.qubits and len(op.qubits) == 1 and not _is_virtual_z(op)]
            if not active:
                continue
            i = active[0] + 1
            while i < len(circuit):
                if busy[i]:
                    i += 1
                    continue
                j = i
                while j < len(circuit) and not busy[j]:
                    j += 1
                slots = [t for t in range(i, j) if one_q_layer[t]]
                seq = [_X, _Y, _X, _Y] if len(slots) >= _DD_MIN_SLOTS else []
                for s, gate in enumerate(seq):  # evenly spaced over the window
                    moments[slots[round((s + 0.5) * len(slots) / len(seq) - 0.5)]].append(gate.on(q))
                i = j
        return cirq.Circuit(cirq.Moment(m) for m in moments)


    # ---- entry point ------------------------------------------------------------------------


    def create_dicke_state(
        num_qubits: int,
        num_excitations: int,
        cz_pauli_error_rate: dict[
            tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]
        ],
        sq_pauli_error_rate: dict[cirq.GridQubit, tuple[float, float]],
        zero_errors: dict[cirq.GridQubit, tuple[float, float]],
        one_errors: dict[cirq.GridQubit, tuple[float, float]],
    ) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
        """Create a Cirq circuit that produces a Dicke state.

        Args:
            num_qubits: The number, N, of qubits in the Dicke state.
            num_excitations: The number, n, of 1s in the state.
            cz_pauli_error_rate: Measured CZ Pauli error rates and their statistical uncertainties.
            sq_pauli_error_rate: Measured single-qubit Pauli error rates and their statistical uncertainties.
            zero_errors: Measured parallel 0 -> 1 readout error rates and their statistical uncertainties.
            one_errors: Measured 1 -> 0 readout error rates and their statistical uncertainties.

        Returns:
            A Cirq circuit to produce Dicke state and a set indicating which qubits the resulting state lives on.

        Raises:
            NotImplementedError: The requested (N, n) combination is not implemented.
        """
        return build(
            num_qubits, num_excitations, cz_pauli_error_rate, sq_pauli_error_rate,
            zero_errors, one_errors,
        )


    def build(
        num_qubits, num_excitations, cz_pauli_error_rate, sq_pauli_error_rate,
        zero_errors, one_errors, *, w_tree: bool = True, synth: bool = True,
        scs: bool = True, tiny: bool = True, alap: bool = False, dd: bool = True,
    ) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
        """`create_dicke_state` with switches for the experiments (defaults = submission)."""
        circuit, qubits = _build_core(
            num_qubits, num_excitations, cz_pauli_error_rate, sq_pauli_error_rate,
            zero_errors, one_errors, w_tree=w_tree, synth=synth, scs=scs, tiny=tiny,
        )
        if alap:
            circuit = _alap(circuit)
        return (_add_dd(circuit) if dd else circuit), qubits


    def _alap(circuit: cirq.Circuit) -> cirq.Circuit:
        """Schedule every operation as late as possible, so qubits stay in |0> (immune to
        T1 and dephasing) until they are needed: pack the time-reversed circuit earliest-first,
        then reverse it back. Order on each qubit is unchanged."""
        rev = cirq.Circuit(op for m in reversed(circuit) for op in m)
        c = cirq.Circuit(reversed(list(rev)))
        if any(any(op.gate == cirq.CZ for op in m) and any(len(op.qubits) == 1 for op in m) for m in c):
            c = cirq.stratified_circuit(c, categories=[lambda op: op.gate == cirq.CZ])
        return c


    def _build_core(
        num_qubits, num_excitations, cz_pauli_error_rate, sq_pauli_error_rate,
        zero_errors, one_errors, *, w_tree: bool, synth: bool, scs: bool, tiny: bool,
    ) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
        N, n = int(num_qubits), int(num_excitations)
        if N < 1 or not 0 <= n <= N:
            raise NotImplementedError(f"invalid (N, n) = ({N}, {n})")
        flip = n > N - n  # D^N_n = X^N D^N_{N-n}: prepare the lighter one
        k = N - n if flip else n
        node_cost = {
            q: _qubit_cost(q, sq_pauli_error_rate, zero_errors, one_errors, n / N)
            for q in sq_pauli_error_rate
        }

        if k == 0:  # product state: |0...0> or |1...1>
            qubits = sorted(node_cost, key=node_cost.get)[:N]
            if len(qubits) < N:
                raise NotImplementedError("not enough calibrated qubits")
            ops = [cirq.PhasedXZGate(x_exponent=1, z_exponent=0, axis_phase_exponent=0).on(q) for q in qubits] if flip else []
            return cirq.Circuit(ops), set(qubits)

        g = _usable_graph(cz_pauli_error_rate, sq_pauli_error_rate)

        # Measured (experiments/results/tiny.csv): for W2 this raises E[score] 60.2 -> 65.6 while
        # the median dips 4.37 -> 4.16; for W3 it gains nothing, so only N = 2 uses it.
        if k == 1 and w_tree and tiny and N == 2:
            picked = _best_tiny_w(g, N, n, flip, cz_pauli_error_rate, sq_pauli_error_rate,
                                  zero_errors, one_errors)
            if picked is not None:
                return picked

        if k == 1 and w_tree:
            tree_cost = {  # a tree node receives ~4 non-virtual single-qubit gates
                q: _qubit_cost(q, sq_pauli_error_rate, zero_errors, one_errors, n / N, n_sq=4)
                for q in sq_pauli_error_rate
            }
            tree = _best_w_tree(g, N, tree_cost, zero_errors, one_errors)
            if tree is not None:
                root, rounds = tree
                logical = _w_tree_circuit(root, rounds)
                qubits = {root} | {c for r in rounds for _, c in r}
                if flip:
                    logical.append(cirq.X.on_each(*sorted(qubits)))
                return _to_cz_phxz(logical), qubits

        scs_cz = _scs_cz_count(N, k) if scs else None
        synth_entry = _SYNTH.get((N, k)) if synth else None
        if synth_entry is not None and (scs_cz is None or sum(map(len, synth_entry["cz_layers"])) <= scs_cz):
            qs = _place_synth(g, synth_entry, n, sq_pauli_error_rate, zero_errors, one_errors)
            if qs is not None:
                logical = _synth_circuit(synth_entry, qs)
                if flip:
                    logical.append(cirq.X.on_each(*qs))
                return _to_cz_phxz(logical), set(qs)

        if scs_cz is not None:
            paths = _best_path(g, N, node_cost, edge_weight=scs_cz / (N - 1), keep=16)
            if paths:
                K = math.comb(N, n)

                def total(cost_path):
                    cost, path = cost_path
                    lam = _readout_lambda(path, zero_errors, one_errors, n, math.exp(-cost))
                    return cost + math.log1p(lam / (K - 1))

                path = min(paths, key=total)[1]
                logical = _scs_circuit(N, k, path)
                if flip:
                    logical.append(cirq.X.on_each(*path))
                return _to_cz_phxz(logical), set(path)

        layout = _best_path(g, N, node_cost) or _connected_patch(g, N, node_cost)
        if layout is None:
            raise NotImplementedError("no connected layout of the requested size")

        logical, lq = _bartschi_eidenbenz(N, k)
        if flip:
            logical.append(cirq.X.on_each(*lq))

        # Routing is chaotic: routing the raw CNOT/CRy circuit or a CZ-compiled one wins on
        # different calibrations (by up to ~30% CZ). Try both while time allows; keep the
        # one with fewer CZs. The raw variant goes first because it is the faster one.
        device = g.subgraph(layout).copy()
        start = time.monotonic()
        best = None
        for precompile in (False, True):
            if best is not None and time.monotonic() - start > _SECOND_TRY_BUDGET_S:
                break
            router = cirq.RouteCQC(device)
            initial = cirq.HardCodedInitialMapper(dict(zip(lq, layout)))
            source = _to_cz_phxz(logical) if precompile else logical
            routed, initial_map, swap_map = router.route_circuit(source, initial_mapper=initial)
            circuit = _to_cz_phxz(routed)
            key = (sum(op.gate == cirq.CZ for op in circuit.all_operations()), len(circuit))
            if best is None or key < best[0]:
                best = (key, circuit, {swap_map[initial_map[q]] for q in lq})
        return best[1], best[2]


    # ---- bonus track ------------------------------------------------------------------------------


    def _uniform_calibration() -> dict:
        """The bonus provides no error rates: every Willow105 qubit and coupler, equal rates."""
        import cirq_google

        qubits = sorted(cirq_google.Willow105.metadata.qubit_set)
        pairs = [tuple(sorted(p)) for p in cirq_google.Willow105.metadata.qubit_pairs]
        return dict(
            cz_pauli_error_rate={p: (2.5e-3, 0.0) for p in pairs},
            sq_pauli_error_rate={q: (4e-4, 0.0) for q in qubits},
            zero_errors={q: (5e-3, 0.0) for q in qubits},
            one_errors={q: (1e-2, 0.0) for q in qubits},
        )


    def _pack(circuit: cirq.Circuit) -> cirq.Circuit:
        """Fewest moments: merge single-qubit runs, then re-insert every operation as early as
        possible (single-qubit gates may share a moment with CZs; depth is all that is scored)."""
        c = cirq.merge_single_qubit_gates_to_phxz(circuit)
        c = cirq.drop_empty_moments(cirq.drop_negligible_operations(c))
        return cirq.Circuit(c.all_operations())


    def create_dicke_state_adaptive(
        num_qubits: int, num_excitations: int
    ) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
        """Create a Cirq circuit that produces a Dicke state, allowing for adaptive circuits.

        This entry is not adaptive yet: it returns our shallowest exact unitary construction
        (allowed by the rules), built for a uniform calibration and packed for depth. Every
        measurement branch is trivially exact because there are no measurements.

        Args:
            num_qubits: The number, N, of qubits in the Dicke state.
            num_excitations: The number, n, of 1s in the state.

        Returns:
            A Cirq circuit to produce Dicke state and a set indicating which qubits the resulting state lives on.

        Raises:
            NotImplementedError: The requested (N, n) combination is not implemented.
        """
        circuit, qubits = build(num_qubits, num_excitations, **_uniform_calibration(), dd=False)
        return _pack(circuit), qubits
    return create_dicke_state

# aanha source SHA256: 6e78e83ab62d574ca34dfd9253898a391125f0ef2b7b4ddf0ddaf75800e16703
def _load_aanha():
    __name__ = '_embedded_aanha'
    import itertools
    import math
    import time

    import cirq
    import cirq_google
    import networkx as nx
    import numpy as np

    # Dicke state preparation on Google Willow.
    # For each (N, n) several exact candidate circuits are built (stored short templates
    # for small cases, splitting trees for n = 1, a two row split for larger n, a single
    # chain fallback), each is placed on low error qubits from the calibration data,
    # scheduled as late as possible, given dynamical decoupling pulses, and the candidate
    # with the highest predicted success is returned.




    # ---------------------------------------------------------------------------
    # Error dictionary helpers
    # ---------------------------------------------------------------------------

    def _pair_key(a, b):
        return frozenset((a, b))


    def _usable_graph(cz_pauli_error_rate, sq_pauli_error_rate):
        """Adjacency of qubits that have working single qubit gates and couplers."""
        good = set(sq_pauli_error_rate)
        adj = {q: {} for q in good}
        for pair, (err, unc) in cz_pauli_error_rate.items():
            a, b = tuple(pair)
            if a in good and b in good:
                # pessimistic estimate: a coupler that only looks good by chance is not trusted
                adj[a][b] = err + unc
                adj[b][a] = err + unc
        return adj


    def _qubit_cost(q, zero_errors, one_errors, sq_pauli_error_rate, frac_one):
        z = sum(zero_errors.get(q, (0.05, 0)))
        o = sum(one_errors.get(q, (0.05, 0)))
        s = sum(sq_pauli_error_rate.get(q, (0.01, 0)))
        return (1 - frac_one) * z + frac_one * o + 4 * s


    # ---------------------------------------------------------------------------
    # Choosing a path of qubits
    # ---------------------------------------------------------------------------

    def find_path(num_qubits, cz_pauli_error_rate, sq_pauli_error_rate,
                  zero_errors, one_errors, frac_one=0.5, time_budget=10.0):
        """Find a low error simple path of num_qubits qubits on the usable graph."""
        import time

        adj = _usable_graph(cz_pauli_error_rate, sq_pauli_error_rate)
        qcost = {q: _qubit_cost(q, zero_errors, one_errors, sq_pauli_error_rate, frac_one)
                 for q in adj}
        if num_qubits == 1:
            best = min(adj, key=lambda q: qcost[q])
            return [best]

        deadline = time.time() + time_budget
        best_path, best_cost = None, float("inf")

        def path_cost(p):
            c = sum(qcost[q] for q in p)
            c += sum(2.5 * adj[p[i]][p[i + 1]] for i in range(len(p) - 1))
            return c

        # starts ordered by local quality
        starts = sorted(adj, key=lambda q: qcost[q] + min(adj[q].values(), default=1))
        for s in starts:
            if time.time() > deadline:
                break
            # randomized greedy depth first search with Warnsdorff tie breaking
            for attempt in range(3):
                rng = np.random.default_rng(hash((s, attempt)) % (2 ** 32))
                path = [s]
                used = {s}
                stack = [iter(())]

                def options(q):
                    opts = [r for r in adj[q] if r not in used]
                    # prefer few onward options (keeps the path from boxing itself in),
                    # then low error
                    return sorted(opts, key=lambda r: (
                        sum(1 for t in adj[r] if t not in used),
                        qcost[r] + 2.5 * adj[q][r] + 1e-3 * rng.random()))

                stack = [iter(options(s))]
                steps = 0
                while stack and len(path) < num_qubits:
                    steps += 1
                    if steps > 400000 or time.time() > deadline:
                        break
                    nxt = next(stack[-1], None)
                    if nxt is None:
                        stack.pop()
                        used.discard(path.pop())
                        continue
                    path.append(nxt)
                    used.add(nxt)
                    stack.append(iter(options(nxt)))
                if len(path) == num_qubits:
                    c = path_cost(path)
                    if c < best_cost:
                        best_cost, best_path = c, list(path)
        if best_path is None:
            raise NotImplementedError("No path of the requested length found.")
        return best_path


    # ---------------------------------------------------------------------------
    # Logical gates (later compiled to CZ + PhasedXZ)
    # ---------------------------------------------------------------------------

    def _cry(control, target, theta):
        """Exact controlled Ry(theta) with 2 CZs."""
        return [
            cirq.ry(theta / 2).on(target),
            cirq.CZ(control, target),
            cirq.ry(-theta / 2).on(target),
            cirq.CZ(control, target),
        ]


    def _cnot(control, target):
        return [cirq.H(target), cirq.CZ(control, target), cirq.H(target)]


    def _mux_ry_two_controls(b, c, a, f00, f01, f10, f11):
        """Exact Ry on target a with angle f[b][c] (uniformly controlled rotation).

        a must neighbour both b and c. Uses 4 CZs.
        Sequence in time: Ry(t1) CX(c,a) Ry(t2) CX(b,a) Ry(t3) CX(c,a) Ry(t4) CX(b,a).
        """
        # parity of fired controls before each rotation: p1=0, p2=c, p3=c^b, p4=b
        rows = []
        rhs = []
        for (bb, cc, f) in [(0, 0, f00), (0, 1, f01), (1, 0, f10), (1, 1, f11)]:
            par = [0, cc, cc ^ bb, bb]
            rows.append([(-1) ** p for p in par])
            rhs.append(f)
        t = np.linalg.solve(np.array(rows, float), np.array(rhs, float))
        ops = []
        ops.append(cirq.ry(t[0]).on(a))
        ops += _cnot(c, a)
        ops.append(cirq.ry(t[1]).on(a))
        ops += _cnot(b, a)
        ops.append(cirq.ry(t[2]).on(a))
        ops += _cnot(c, a)
        ops.append(cirq.ry(t[3]).on(a))
        ops += _cnot(b, a)
        return ops


    def _split_angle(c, j):
        """Angle so the wall pair becomes sqrt(c/j)|01> + sqrt((j-c)/j)|10>."""
        alpha = math.sqrt(c / j)
        beta = math.sqrt((j - c) / j)
        return 2 * math.atan2(beta, alpha)


    def _plain_unit(a, b, phi):
        """Pair (a, b) holding 00, 01 or 11. 01 -> cos|01> + sin|10>."""
        ops = []
        ops += _cnot(a, b)
        ops += _cry(b, a, phi)
        ops += _cnot(a, b)
        return ops


    def _controlled_unit(c, a, b, phi):
        """Triple (c, a, b) on a line. If c is 0 this acts as the plain unit;
        if c is 1 (a hole is being carried through), 01 on (a, b) becomes 10."""
        # 5 CZ total: CNOT, a 3 CZ multiplexed rotation on a, CNOT.
        # In the frame after the first CNOT, a must get Ry(phi) when (b, c) = (1, 0),
        # flip from |0> to |1> when (b, c) = (1, 1), and stay put when b = 0.
        u = -(phi + math.pi) / 4
        v = (math.pi - phi) / 4
        ops = []
        ops += _cnot(a, b)
        ops += [cirq.ry(u).on(a), cirq.CZ(c, a), cirq.ry(v).on(a), cirq.CZ(a, b),
                cirq.ry(-v).on(a), cirq.CZ(c, a), cirq.ry(-u).on(a)]
        ops += _cnot(a, b)
        return ops


    def dicke_unitary_ops(chain, max_count):
        """Bartschi Eidenbenz Dicke unitary on a nearest neighbour chain.

        chain[0..m-1] are BE positions 1..m. Maps the unary state with l ones at the
        END of the chain (positions m-l+1..m) to D(m, l), for all l <= max_count.
        """
        m = len(chain)
        pos = lambda p: chain[p - 1]  # BE position p -> qubit
        ops = []
        for j in range(m, 1, -1):
            k = min(max_count, j - 1)
            for p in range(j - k, j):
                c = j - p  # number of ones to the right of the wall if the wall sits at p
                phi = _split_angle(c, j)
                if p == j - k or p == 1:
                    ops += _plain_unit(pos(p), pos(p + 1), phi)
                else:
                    ops += _controlled_unit(pos(p - 1), pos(p), pos(p + 1), phi)
        return ops


    # ---------------------------------------------------------------------------
    # Compilation to the allowed gate set
    # ---------------------------------------------------------------------------

    def compile_to_native(circuit):
        """Rewrite into CZ + PhasedXZ only, merge single qubit gates."""
        c = cirq.optimize_for_target_gateset(
            circuit, gateset=cirq.CZTargetGateset(allow_partial_czs=False))
        c = cirq.merge_single_qubit_gates_to_phxz(c)
        c = cirq.drop_negligible_operations(c)
        c = cirq.drop_empty_moments(c)
        # make sure every single qubit op is a PhasedXZGate
        out = []
        for op in c.all_operations():
            if len(op.qubits) == 1:
                out.append(cirq.PhasedXZGate.from_matrix(cirq.unitary(op)).on(*op.qubits))
            else:
                out.append(op)
        return cirq.Circuit(out)


    # ---------------------------------------------------------------------------
    # Main function
    # ---------------------------------------------------------------------------

    def _baseline_create_dicke_state(
        num_qubits,
        num_excitations,
        cz_pauli_error_rate,
        sq_pauli_error_rate,
        zero_errors,
        one_errors,
    ):
        N, n = num_qubits, num_excitations
        if N < 1 or n < 0 or n > N:
            raise NotImplementedError("Invalid (N, n).")

        flip = n > N - n          # build the smaller case, flip all qubits at the end
        k = N - n if flip else n

        if k == 0:
            # product state, no entangling gates and no path needed: pick the best N qubits
            good = sorted(sq_pauli_error_rate, key=lambda q: _qubit_cost(
                q, zero_errors, one_errors, sq_pauli_error_rate, n / N))
            if len(good) < N:
                raise NotImplementedError("Not enough working qubits.")
            chosen = good[:N]
            x = 1 if n == N else 0
            circuit = cirq.Circuit(cirq.Moment(cirq.PhasedXZGate(
                x_exponent=x, z_exponent=0, axis_phase_exponent=0).on(q) for q in chosen))
            return circuit, set(chosen)

        chain = find_path(N, cz_pauli_error_rate, sq_pauli_error_rate,
                          zero_errors, one_errors, frac_one=n / N,
                          time_budget=10.0 if N <= 60 else 30.0)

        ops = []
        # start: k ones packed at the end of the chain
        for q in chain[N - k:]:
            ops.append(cirq.X(q))
        if 0 < k < N:
            ops += dicke_unitary_ops(chain, k)
        if flip:
            ops += [cirq.X(q) for q in chain]
        if not ops:
            ops = [cirq.I(q) for q in chain]

        circuit = compile_to_native(cirq.Circuit(ops))
        # keep every output qubit in the circuit
        missing = [q for q in chain if q not in circuit.all_qubits()]
        if missing:
            circuit.append(cirq.PhasedXZGate(x_exponent=0, z_exponent=0,
                                             axis_phase_exponent=0).on_each(*missing))
        if len(circuit) > 1750:
            raise NotImplementedError("Circuit too deep.")
        return circuit, set(chain)






    def _grow_tree(root, N, adj, qcost):
        """Broadcast style growth. Returns rounds: list of [(parent, child), ...]."""
        used = {root}
        holders = [root]
        rounds = []
        while len(used) < N:
            G = nx.Graph()
            for h in holders:
                for r in adj[h]:
                    if r not in used:
                        # higher weight is better: prefer good couplers and good qubits
                        w = 1.0 - (2.5 * adj[h][r] + qcost[r])
                        G.add_edge(("h", h), ("f", r), weight=w)
            if G.number_of_edges() == 0:
                return None
            m = nx.max_weight_matching(G, maxcardinality=True)
            pairs = []
            for a, b in m:
                if a[0] == "f":
                    a, b = b, a
                pairs.append((a[1], b[1]))
            need = N - len(used)
            if len(pairs) > need:
                pairs.sort(key=lambda p: 2.5 * adj[p[0]][p[1]] + qcost[p[1]])
                pairs = pairs[:need]
            for _, c in pairs:
                used.add(c)
            holders += [c for _, c in pairs]
            rounds.append(pairs)
        return rounds


    def _tree_cost(root, rounds, adj, qcost):
        c = qcost[root]
        for rnd in rounds:
            for p, ch in rnd:
                c += qcost[ch] + 2 * 2.5 * adj[p][ch]
        c += 0.004 * len(rounds) * sum(1 + len(r) for r in rounds) ** 0  # depth penalty
        return c + 0.01 * len(rounds)


    def build_w_tree(N, cz_pauli_error_rate, sq_pauli_error_rate, zero_errors,
                     one_errors, num_roots=25, keep=1):
        trees = _choose_trees(N, cz_pauli_error_rate, sq_pauli_error_rate, zero_errors,
                              one_errors, num_roots, keep)
        return [_tree_circuit(root, rounds) for _, root, rounds in trees]


    def _choose_trees(N, cz_pauli_error_rate, sq_pauli_error_rate, zero_errors,
                      one_errors, num_roots, keep):
        adj = _usable_graph(cz_pauli_error_rate, sq_pauli_error_rate)
        qcost = {q: _qubit_cost(q, zero_errors, one_errors, sq_pauli_error_rate, 1.0 / N)
                 for q in adj}
        roots = sorted((q for q in adj if adj[q]),
                       key=lambda q: qcost[q] + min(adj[q].values()))[:num_roots]
        found = []
        for r in roots:
            rounds = _grow_tree(r, N, adj, qcost)
            if rounds is None:
                continue
            found.append((_tree_cost(r, rounds, adj, qcost), r, rounds))
        if not found:
            raise NotImplementedError("No tree found.")
        found.sort(key=lambda t: t[0])
        return found[:keep]


    def _tree_circuit(root, rounds):

        # subtree sizes
        children = {root: []}
        for rnd in rounds:
            for p, c in rnd:
                children.setdefault(p, []).append(c)
                children.setdefault(c, [])

        size = {}

        def sz(v):
            if v not in size:
                size[v] = 1 + sum(sz(c) for c in children[v])
            return size[v]

        sz(root)
        # remaining share owned by each node (itself plus children not yet split off)
        remaining = dict(size)

        moments = []
        first = True
        for rnd in rounds:
            m1, m2, m3, m4, m5 = [], [], [], [], []
            for p, c in rnd:
                frac = size[c] / remaining[p]          # share handed to the child
                theta = 2 * math.asin(math.sqrt(frac))
                if first:
                    # root and first child both start in |0>:
                    # child = Ry(theta)|0>, then root = NOT child
                    m1.append(cirq.ry(theta).on(c))
                    m2.append(cirq.H(p))
                    m3.append(cirq.CZ(c, p))
                    m4.append(cirq.PhasedXZGate.from_matrix(
                        cirq.unitary(cirq.X) @ cirq.unitary(cirq.H)).on(p))
                else:
                    # controlled Ry on a fresh |0> target (1 CZ), then CNOT child -> parent
                    m1.append(cirq.ry(-theta / 2).on(c))
                    m2.append(cirq.CZ(p, c))
                    m3.append(cirq.ry(theta / 2).on(c))
                    m3.append(cirq.H(p))
                    m4.append(cirq.CZ(c, p))
                    m5.append(cirq.H(p))
                remaining[p] -= size[c]
            if first:
                moments += [cirq.Moment(m1 + m2), cirq.Moment(m3), cirq.Moment(m4)]
                first = False
            else:
                moments += [cirq.Moment(m1), cirq.Moment(m2), cirq.Moment(m3),
                            cirq.Moment(m4), cirq.Moment(m5)]
        qubits = [root] + [c for rnd in rounds for _, c in rnd]
        return cirq.Circuit(moments), qubits






    # ---------------------------------------------------------------------------
    # Layout search
    # ---------------------------------------------------------------------------

    def _straight_ladders(adj, n):
        """All placements of two neighbouring straight rows of n qubits with working rungs.

        Returns (top, bottom) where top[i] is adjacent to bottom[i] (same column).
        """
        out = []
        qs = set(adj)
        for q in qs:
            for d, perp in (((0, 1), (1, 0)), ((1, 0), (0, 1)), ((0, 1), (-1, 0)), ((1, 0), (0, -1))):
                top = [q + d_scaled for d_scaled in []]  # placeholder, rebuilt below
                top = []
                bot = []
                ok = True
                for i in range(n):
                    t = q.__class__(q.row + d[0] * i, q.col + d[1] * i)
                    b = q.__class__(t.row + perp[0], t.col + perp[1])
                    if t not in qs or b not in qs or b not in adj[t]:
                        ok = False
                        break
                    if i > 0 and (t not in adj[top[-1]] or b not in adj[bot[-1]]):
                        ok = False
                        break
                    top.append(t)
                    bot.append(b)
                if ok:
                    out.append((top, bot))
        return out


    def _extend(start, length, adj, qcost, used, deadline):
        """Self avoiding path of `length` new qubits continuing from `start`."""
        if length == 0:
            return []
        best = None
        for attempt in range(4):
            rng = np.random.default_rng(attempt)
            path = []
            local_used = set(used)

            def options(q):
                opts = [r for r in adj[q] if r not in local_used]
                return sorted(opts, key=lambda r: (
                    sum(1 for t in adj[r] if t not in local_used),
                    qcost[r] + 2.5 * adj[q][r] + 1e-3 * rng.random()))

            stack = [iter(options(start))]
            prev = [start]
            steps = 0
            while stack and len(path) < length:
                steps += 1
                if steps > 5000 or time.time() > deadline:
                    break
                nxt = next(stack[-1], None)
                if nxt is None:
                    stack.pop()
                    if path:
                        local_used.discard(path.pop())
                        prev.pop()
                    continue
                path.append(nxt)
                local_used.add(nxt)
                prev.append(nxt)
                stack.append(iter(options(nxt)))
            if len(path) == length:
                c = sum(qcost[q] for q in path)
                if best is None or c < best[0]:
                    best = (c, list(path))
        return None if best is None else best[1]


    def find_two_rows(N, n, mA, mB, adj, qcost, time_budget=15.0, keep=1):
        deadline = time.time() + time_budget
        ladders = _straight_ladders(adj, n)

        def lcost(top, bot):
            c = sum(qcost[q] for q in top + bot)
            c += sum(2.5 * adj[t][b] for t, b in zip(top, bot))
            c += sum(2.5 * (adj[top[i]][top[i + 1]] + adj[bot[i]][bot[i + 1]]) for i in range(n - 1))
            return c

        ladders.sort(key=lambda tb: lcost(*tb))
        found = []
        for top, bot in ladders[:60]:
            if time.time() > deadline:
                break
            # A: a1 = top[0] ... an = top[n-1], continues from top[n-1]
            # B: b1 = bot[n-1] ... bn = bot[0],  continues from bot[0]
            A = list(top)
            B = list(reversed(bot))
            used = set(A) | set(B)
            extA = _extend(A[-1], mA - n, adj, qcost, used, deadline)
            if extA is None:
                continue
            used |= set(extA)
            extB = _extend(B[-1], mB - n, adj, qcost, used, deadline)
            if extB is None:
                continue
            A += extA
            B += extB
            cost = lcost(top, bot) + sum(qcost[q] for q in extA + extB)
            found.append((cost, A, B))
            if len(found) >= max(8, 3 * keep):
                break
        if not found:
            return None
        found.sort(key=lambda t: t[0])
        return [(A, B) for _, A, B in found[:keep]]


    # ---------------------------------------------------------------------------
    # Circuit
    # ---------------------------------------------------------------------------

    def _split_ops(A, B, n, mA, mB):
        """Thermometer on a1..an plus rungs. Column i (1 based): a_i and b_(n+1-i)."""
        N = mA + mB
        P = [math.comb(mA, j) * math.comb(mB, n - j) / math.comb(N, n) for j in range(n + 1)]
        ge = [sum(P[j:]) for j in range(n + 2)]   # P(j >= i)
        le = [sum(P[:i + 1]) for i in range(n + 1)]  # P(j <= i)

        def ang(p):
            p = min(max(p, 0.0), 1.0)
            return 2 * math.asin(math.sqrt(p))

        s = (n + 1) // 2  # start column, generate outward both ways
        ops = []
        a = lambda i: A[i - 1]
        # start column: marginal
        ops.append(cirq.ry(ang(ge[s])).on(a(s)))
        # right of start: x_i = 0 if x_(i-1) = 0, else 1 with P(j >= i | j >= i-1)
        right = []
        for i in range(s + 1, n + 1):
            th = ang(ge[i] / ge[i - 1]) if ge[i - 1] > 0 else 0.0
            right.append((i, i - 1, -th / 2, th / 2))
        # left of start: x_i = 1 if x_(i+1) = 1, else 1 with P(j = i | j <= i)
        left = []
        for i in range(s - 1, 0, -1):
            th = ang(P[i] / le[i]) if le[i] > 0 else 0.0
            left.append((i, i + 1, (th - math.pi) / 2, (th + math.pi) / 2))
        for seq in (right, left):
            for (i, ctrl, alpha, beta) in seq:
                ops.append(cirq.ry(alpha).on(a(i)))
                ops.append(cirq.CZ(a(ctrl), a(i)))
                ops.append(cirq.ry(beta).on(a(i)))
        # rungs: b_(n+1-i) = NOT x_i
        for i in range(1, n + 1):
            b = B[n - i]
            ops.append(cirq.X(b))
            ops += _cnot(a(i), b)
        return ops


    def build_two_rows(N, n, cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors,
                       keep=1):
        """Requires 1 <= n <= N/2. Returns (circuit, qubit list) or None if no layout fits."""
        mB = N // 2
        mA = N - mB
        if n > mB:
            return None
        adj = _usable_graph(cz_pauli_error_rate, sq_pauli_error_rate)
        qcost = {q: _qubit_cost(q, zero_errors, one_errors, sq_pauli_error_rate, n / N)
                 for q in adj}
        layouts = find_two_rows(N, n, mA, mB, adj, qcost, keep=keep)
        if layouts is None:
            return None
        out = []
        for A, B in layouts:
            ops = _split_ops(A, B, n, mA, mB)
            # finish each row: baseline expects the ones at the END of the chain list
            ops += dicke_unitary_ops(list(reversed(A)), min(n, mA))
            ops += dicke_unitary_ops(list(reversed(B)), min(n, mB))
            out.append((cirq.Circuit(ops), A + B))
        return out



    _X = cirq.PhasedXZGate(x_exponent=1, z_exponent=0, axis_phase_exponent=0)
    _Y = cirq.PhasedXZGate(x_exponent=1, z_exponent=0, axis_phase_exponent=0.5)


    def add_decoupling(circuit, min_gap=2):
        """Fill idle stretches of every qubit (after its first gate) with X X or X Y X Y.

        Idle stretches run until the end of the circuit, since the measurement comes
        right after.
        """
        moments = [list(m.operations) for m in circuit]
        busy = {}
        for t, mom in enumerate(moments):
            for op in mom:
                for q in op.qubits:
                    busy.setdefault(q, []).append(t)
        T = len(moments)
        for q, times in busy.items():
            times = sorted(set(times))
            # gaps after each busy moment
            bounds = list(zip(times, times[1:] + [T]))
            for start, end in bounds:
                gap = end - start - 1          # free moments strictly between
                if gap < min_gap:
                    continue
                seq = [_X, _Y, _X, _Y] if gap >= 4 else [_X, _X]
                L = len(seq)
                # evenly spaced pulse positions inside the free window
                slots = [start + 1 + int((i + 0.5) * gap / L) for i in range(L)]
                for s, g in zip(slots, seq):
                    moments[s].append(g.on(q))
        return cirq.Circuit(cirq.Moment(m) for m in moments)




    T1_US = 68.0
    T2_US = 100.0
    DT_CZ_US = 0.042
    DT_SQ_US = 0.025


    def alap(circuit):
        ops = list(circuit.all_operations())
        rev = cirq.Circuit(reversed(ops))      # earliest placement of the reversed list
        return cirq.Circuit(reversed(list(rev)))


    def _pair_err(cz, a, b):
        v = cz.get((a, b)) or cz.get((b, a))
        return v[0] + v[1]


    def predict_success(circuit, qubits, n, cz, sq, z0, o1):
        N = len(qubits)
        logp = 0.0
        dur = []
        for mom in circuit:
            has2 = False
            for op in mom:
                if len(op.qubits) == 2:
                    has2 = True
                    logp += math.log(max(1e-9, 1 - 1.25 * _pair_err(cz, *op.qubits)))
                else:
                    q = op.qubits[0]
                    logp += math.log(max(1e-9, 1 - sum(sq[q])))
            dur.append(DT_CZ_US if has2 else DT_SQ_US)
        # time each qubit spends after its first gate
        first = {}
        for t, mom in enumerate(circuit):
            for op in mom:
                for q in op.qubits:
                    first.setdefault(q, t)
        total = sum(dur)
        suffix = [0.0] * (len(dur) + 1)
        for t in range(len(dur) - 1, -1, -1):
            suffix[t] = suffix[t + 1] + dur[t]
        f = n / N if N else 0
        rate = f / T1_US + 1 / (2 * T2_US)     # decay only hurts the part that is 1
        for q, t in first.items():
            logp -= suffix[t] * rate
        for q in qubits:
            e = (1 - f) * sum(z0.get(q, (0.05, 0))) + f * sum(o1.get(q, (0.05, 0)))
            logp += math.log(max(1e-9, 1 - e))
        return math.exp(logp)


    TEMPLATES = {
        "4,2": {
            "m": 5,
            "cells": [(0, 0), (0, 1), (1, 1), (1, 0)],
            "edges": [(0, 1), (0, 3), (1, 2), (3, 2)],
            "seq": [3, 2, 1, 0, 3],
            "params": [0.9152251869199917, -1.570796325403217, 2.0900648210551114, 1.8692011249470026, -1.570796325562261, 2.3655201874146266, -2.4179334405501467, -1.6246094335940167, -2.197358915225356, -0.5760806525818848, 2.410567616738059, -1.6924249148248318, -1.7956944685060041, -1.06279538834646, -2.158352099813202, -2.2140202840636007, 1.2166626717544191, -0.9091641032693085, 1.0167743919391317, -1.6722008125477743, 2.8431878576638736, -0.651597662858857, -1.5707963267137959, 1.1685646075017446, 2.652700642515031, 8.791651371615145e-10, 2.997493840136398, -0.7369791770867095, -1.5707963258859357, 1.7743621060038888, 4.144244618119247, 1.570796328026762, -0.2822343628593521, 1.4364014618324719, -1.3113045784793899e-09, 0.12027243612363181, 1.0026519639928766, -1.5707963241025313, 0.736979177858213, 1.2691374818911962, 3.1415926554193554, -2.2235094747305855],
        },
        "5,2": {
            "m": 10,
            "cells": [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1)],
            "edges": [(0, 1), (0, 3), (1, 2), (1, 4), (3, 4)],
            "seq": [3, 2, 4, 0, 1, 0, 4, 3, 1, 4],
            "params": [1.0978818961939096, -0.7853981242917613, 1.4727555223511677, 1.3757814194126186, -0.9170366410619162, 1.7773964788258096, 2.926994429124498, -1.5707963040032937, -0.9655221139379317, 0.796969491675068, 0.6154798365710585, 1.9088611740202805, 0.9749007888954939, 2.4764731195617986, 2.086792667558297, 0.16814061795965682, 1.102983753098459, -0.8222705778608492, 1.5183442695097957, 2.9331039340755223, 0.9950968113067814, -2.2225822412902643, 1.055003266210752e-07, -0.8266343538038219, 0.3460466163492173, 1.5707962439153007, 0.21459822409394888, 1.0233899020030446, 0.6154797435120007, -0.7962892193437947, -3.1563444034644546, 1.5707960378431718, -0.007921372872869046, 3.3783045745121156, 1.570894713468651, -2.6685800860476085, 0.9629152381633073, 1.57079631071484, 1.6631556468854818, 1.0249551738824412, -1.5709348523232949, 0.5486862494919814, -0.6822053057212604, -1.5707963153536546, 0.5472672091202356, -1.7330199705327285, -1.5707963208246396, -1.0246777971383083, 2.9713176656497753, -3.1415926639658607, 1.778396019855926, -0.10054050223156384, 2.3561944349949893, 0.681813355783217, 0.47724778407631113, -0.7853981431521005, -3.1276252747892106, 1.1666465249312237, 1.5707963109177905, 2.9115862150569454, -2.2698559679921844, 6.705351474039285e-09, -0.0016836599857636219, -0.40442749188590676, 1.5707963006124381, 1.7330200043981783, 1.4971098712803268, 3.141592638313141, 2.368445670371039, 0.09929133328881226, 3.1415926979055717, 1.2750682106928366, -3.517720427914767, -3.358185736798283e-08, 0.19547327980386328],
        },
        "6,2": {
            "m": 15,
            "cells": [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)],
            "edges": [(0, 1), (0, 3), (1, 2), (1, 4), (2, 5), (3, 4), (4, 5)],
            "seq": [5, 1, 5, 4, 6, 4, 5, 2, 3, 0, 6, 1, 3, 4, 1],
            "params": [1.2278237895533397, 1.401178952441533, -2.7971731079931152, -2.5808665155802815, 1.5707963344531495, 2.3075858676623557, -2.302523838510664, 1.5707962901614496, -0.6545982908253885, -2.118287085663263, -0.45244323044809076, -0.9086714810104131, -4.0774646534769685, 1.4859614921461801, -0.8289020056393221, 2.4829013281105987, 1.5707963599748902, -1.5320151967703777, 1.453504434531185, -4.552213382295043, 2.44602334321934, 2.241999021287189, 1.1051961447881224, 1.4155261252380038, 1.942191141590358, -2.475483813008775, 1.9963999541010988, -1.8153612762882163, -1.6325335641088896, -0.8354025860910187, 2.888566821858411, 0.859519513675752, 2.519530923746934, -2.6918006043651137, -2.6348527828796633, -0.530288420449832, -0.031998291676036614, -1.8888729826046289, -0.3096573741564422, -1.7932385610057182, -1.963318048625143, 0.6028245962491279, -0.4446398191320723, -0.7853981786596648, 0.2868477619827955, -0.1544448782912649, -2.186276035512955, -2.7892455073449574, -1.7593069259144232, 1.5707964014514053, 2.092779927097489, 0.49633270849886335, 1.5707963807323797, 2.5106393259099633, -1.4188917551422757, -1.57079631237123, -1.1777392583960136, -2.605196610913039, 4.440195390281019, 2.2983149472950113, 2.5995270063708524, -1.5707963210396247, -3.7023188065502763, 1.51198075264584, 1.570796337621507, -1.693658105011778, -2.2114944561961374, 1.5707963178935218, -0.7173583394512172, 0.16912117937826204, -1.5707963482373573, -1.2828003532518621, -1.0348867239002373, -1.5707963511524992, -1.3674305728941436, -0.38974245103793564, 2.9467459649812683e-08, -2.424364869282272, 1.278496800532173, -3.141592613922145, -0.9952732781983203, 3.3468985622889664, 1.5707963506896416, -2.0671290550595534, -1.0856447003477343, -2.3561944888228528, 2.605683015157323, 0.7623685122886664, 1.570796350559726, 0.6334935986189051, -0.34739482161901575, 1.570796321655559, -1.2575835317253838, -0.7541170489891756, 1.1361168153401097e-08, 0.2628866377054098, -1.52817417510431, 1.5707963148155355, 1.6296119139426695, -1.8458725245988963, 3.141592669244836, 2.618588390205919, 3.3219912440709796, -1.5707963287100164, 1.0856446449391086, 0.1803986058817658, -1.5707962967666567, -2.3331648546130555],
        },
        "6,3": {
            "m": 14,
            "cells": [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)],
            "edges": [(0, 1), (0, 3), (1, 2), (1, 4), (2, 5), (3, 4), (4, 5)],
            "seq": [1, 2, 3, 6, 4, 5, 6, 4, 5, 6, 0, 3, 2, 5],
            "params": [-0.004358709043469499, -1.5707963280527075, 2.1646975265774966, 0.69052433826127, -1.4870179470153946, 2.1129187488917247, -2.6894721547872655, -0.7687799642392928, 2.672643051737504, 1.843160338226393, -1.5707963250015535, 0.12061533467884525, -0.7203065711589474, -2.255515554279712, -1.1523598426037729, -0.07536876045467923, -1.5707963847870503, 2.7631927091716424, -0.921409959511693, 1.3787311015441353, 2.437941252853558, 1.9003876674110678, 1.3597030391806944, 2.071534049549365, -1.3687913033373285, 1.948614278091, 1.8131899942308005, 1.5068387806050083, -0.5419279901627104, -1.43341971289844, 1.8552564157112017, 0.7853981799774187, 0.1034326672828162, 0.3245717925172843, -1.5707965112445277, -0.8504896831131051, -1.7553704098846912, 2.186276085949012, -0.6052804699646529, -1.3689612566630083, 1.5707963538630667, 0.07536880918777478, -0.5990504118756445, 1.8925468830959664, 3.0654892348700193, -1.0810624751546987, 0.9553166780067274, -1.4919227247967852, -3.096535663616525, -1.570796324778838, -2.390107791807479, 1.027401578436607, -1.5707963319509268, 0.9699723486586119, -2.208853910024467, -1.570796346732418, -2.598197908409128, 2.2746113343957597, -1.5707963310662383, -2.8459282677185844, 1.9951728548161158, -1.5707963322841036, 0.5990503619565999, -1.2534375342732498, 3.1415926276652546, -0.573961686242868, 1.586520023435268, 1.5707960164674422, -1.6158533118724603, -3.1116907476219002, -1.5707963378760128, -0.9327387570582413, -1.1124263210456944, 0.758673164741977, -0.02990197929149771, 1.4480070100233413, 1.5707963341829758, -1.7583017947930724, -1.6935856734954495, 2.329469431042431, 1.5109236723221127, 3.7033178073888386, 1.2844876246135925e-08, 3.5335844608605527, -1.1421604475949083, 1.5707963278762889, 3.474212003097007, -1.394733530546847, -1.5707963744861881, -0.45836987289150216, -2.4964677985562322, 5.508105106444252e-08, -1.196550057693644, 1.4480070083985028, -1.5707963644129843, 1.1464198044195317, 3.2388748403708822, -1.611814724257924e-08, 2.9057974451167095, 3.0188033320109473, 1.570796311367528, 3.7242030349228847],
        },
    }





    def _su2(a, b, c):
        return np.array([[np.exp(-0.5j * (a + c)) * np.cos(b / 2), -np.exp(-0.5j * (a - c)) * np.sin(b / 2)],
                         [np.exp(0.5j * (a - c)) * np.sin(b / 2), np.exp(0.5j * (a + c)) * np.cos(b / 2)]])


    def _symmetries(cells):
        out = []
        for flip in (False, True):
            for rot in range(4):
                pts = []
                for (r, c) in cells:
                    if flip:
                        c = -c
                    for _ in range(rot):
                        r, c = c, -r
                    pts.append((r, c))
                out.append(pts)
        return out


    def _template_candidates(N, k, cz, sq, z0, o1, keep=3):
        key = f"{N},{k}"
        if key not in TEMPLATES:
            return []
        t = TEMPLATES[key]
        cells, edges, seq, params = t["cells"], t["edges"], t["seq"], t["params"]
        used = {}
        for e in seq:
            used[e] = used.get(e, 0) + 1
        adj = _usable_graph(cz, sq)
        qcost = {q: _qubit_cost(q, z0, o1, sq, k / N) for q in adj}
        places = []
        seen = set()
        for pts in _symmetries(cells):
            for anchor in adj:
                r0, c0 = anchor.row - pts[0][0], anchor.col - pts[0][1]
                qs = [cirq.GridQubit(r0 + r, c0 + c) for (r, c) in pts]
                if any(q not in adj for q in qs):
                    continue
                ok = True
                cost = sum(qcost[q] for q in qs)
                for e, cnt in used.items():
                    a, b = qs[edges[e][0]], qs[edges[e][1]]
                    if b not in adj[a]:
                        ok = False
                        break
                    cost += 1.25 * cnt * adj[a][b]
                if not ok:
                    continue
                sig = (frozenset(qs), tuple(qs))
                if sig in seen:
                    continue
                seen.add(sig)
                places.append((cost, qs))
        places.sort(key=lambda x: x[0])
        out = []
        for _, qs in places[:keep]:
            ops = []
            p = 0
            for i in range(N):
                ops.append(cirq.MatrixGate(_su2(*params[p:p + 3])).on(qs[i]))
                p += 3
            for e in seq:
                i, j = edges[e]
                ops.append(cirq.CZ(qs[i], qs[j]))
                ops.append(cirq.MatrixGate(_su2(*params[p:p + 3])).on(qs[i]))
                ops.append(cirq.MatrixGate(_su2(*params[p + 3:p + 6])).on(qs[j]))
                p += 6
            out.append((cirq.Circuit(ops), qs))
        return out






    USE_DECOUPLING = True
    _FLIP = cirq.PhasedXZGate(x_exponent=1, z_exponent=0, axis_phase_exponent=0)


    def _finish(raw_circuit, qubits, flip):
        c = raw_circuit
        if flip:
            c = c + cirq.Circuit(cirq.Moment(_FLIP.on(q) for q in qubits))
        c = compile_to_native(c)
        c = alap(c)
        if USE_DECOUPLING:
            c = add_decoupling(c)
        return c


    def create_dicke_state(
        num_qubits,
        num_excitations,
        cz_pauli_error_rate,
        sq_pauli_error_rate,
        zero_errors,
        one_errors,
    ):
        N, n = num_qubits, num_excitations
        if N < 1 or n < 0 or n > N:
            raise NotImplementedError("Invalid (N, n).")
        errs = (cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors)
        start = time.time()

        k = min(n, N - n)
        flip = n != k
        if k == 0:
            return _baseline_create_dicke_state(N, n, *errs)

        raw = []   # (circuit before compiling, qubit list)
        if True:
            try:
                raw += _template_candidates(N, k, *errs)
            except Exception:
                pass
        if k == 1:
            try:
                for c in build_w_tree(N, *errs, keep=4 if N <= 30 else 1):
                    raw.append(c)
            except NotImplementedError:
                pass
        else:
            res = build_two_rows(N, k, *errs, keep=4 if N <= 16 else 1)
            if res:
                raw += res
        if N <= 8 or not raw:
            try:
                chain = find_path(N, *errs, frac_one=n / N,
                                           time_budget=10.0 if N <= 60 else 30.0)
                ops = [cirq.X(q) for q in chain[N - k:]]
                if N > 1:
                    ops += dicke_unitary_ops(chain, k)
                raw.append((cirq.Circuit(ops), chain))
            except NotImplementedError:
                pass
        if not raw:
            raise NotImplementedError("No layout found.")

        best = None
        for c, qs in raw:
            fc = _finish(c, qs, flip)
            if len(fc) > 1750:
                continue
            score = predict_success(fc, qs, n, *errs)
            if best is None or score > best[0]:
                best = (score, fc, qs)
            if time.time() - start > 45:
                break
        if best is None:
            raise NotImplementedError("Circuit too deep.")
        return best[1], set(best[2])


    # ---------------------------------------------------------------------------
    # Bonus. Reuses the main construction on uniform calibration data.
    # ---------------------------------------------------------------------------

    def _ideal_errors():
        qs = list(cirq_google.Willow105.metadata.qubit_set)
        pairs = [tuple(p) for p in cirq_google.Willow105.metadata.qubit_pairs]
        cz = {p: (2.5e-3, 0.0) for p in pairs}
        sq = {q: (4e-4, 0.0) for q in qs}
        z0 = {q: (5e-3, 0.0) for q in qs}
        o1 = {q: (1e-2, 0.0) for q in qs}
        return cz, sq, z0, o1


    def create_dicke_state_adaptive(num_qubits, num_excitations):
        global USE_DECOUPLING
        saved = USE_DECOUPLING
        USE_DECOUPLING = False
        try:
            circuit, qubits = create_dicke_state(num_qubits, num_excitations, *_ideal_errors())
        finally:
            USE_DECOUPLING = saved
        return circuit, qubits
    return create_dicke_state


_BUILDERS = {}


def _builder(name):
    if name not in _BUILDERS:
        _BUILDERS[name] = {
            "ours": _load_ours,
            "richard": _load_richard,
            "aanha": _load_aanha,
        }[name]()
    return _BUILDERS[name]


def _valid(circuit, targets, N, cz_rates, sq_rates, z0_rates, z1_rates):
    if not isinstance(circuit, cirq.Circuit) or not isinstance(targets, set):
        return False
    if len(targets) != N or len(circuit) > 1750:
        return False
    working = set(sq_rates) & set(z0_rates) & set(z1_rates)
    if not targets <= working:
        return False
    edges = {frozenset(edge) for edge in cz_rates}
    for op in circuit.all_operations():
        if not set(op.qubits) <= working:
            return False
        gate = op.gate
        if isinstance(gate, cirq.PhasedXZGate) and len(op.qubits) == 1:
            continue
        if (isinstance(gate, cirq.CZPowGate) and gate.exponent == 1
                and frozenset(op.qubits) in edges):
            continue
        return False
    return True


def create_dicke_state(num_qubits, num_excitations, cz_pauli_error_rate,
                       sq_pauli_error_rate, zero_errors, one_errors):
    """Return an exact, native-gate Dicke preparation on calibrated qubits."""
    N, n = num_qubits, num_excitations
    if N < 1 or not 0 <= n <= N:
        raise NotImplementedError(f"Invalid Dicke state ({N}, {n})")
    rates = (cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors)
    if n in (0, N) or N <= 3:
        order = ("ours",)
    elif N <= 8:
        order = ("richard", "ours")
    else:
        order = ("aanha", "ours")
    last_error = None
    for name in order:
        try:
            result = _builder(name)(N, n, *rates)
            if _valid(*result, N, *rates):
                return result
            last_error = ValueError(f"{name} produced an invalid circuit")
        except Exception as exc:
            last_error = exc
    raise NotImplementedError(f"No valid preparation for ({N}, {n}): {last_error}")


For small $N$, you may check that your circuit produces the correct state using the following function:

In [3]:
def create_dicke_statevector(num_qubits: int, num_excitations: int) -> np.ndarray:
    psi = np.zeros(2**num_qubits)
    normalization = 1 / np.sqrt(math.comb(num_qubits, num_excitations))
    qubits = range(num_qubits)
    for excited_qubits in itertools.combinations(qubits, num_excitations):
        idx = sum(2**qubit_number for qubit_number in excited_qubits)
        psi[idx] = normalization
    return psi


def test_circuit(circuit: cirq.Circuit, num_excitations: int, qubits: set[cirq.GridQubit], sampler = cirq.Simulator()):
    all_qubits = circuit.all_qubits()
    ancilla_qubits = all_qubits.difference(qubits)
    ordered_qubits = sorted(qubits) + sorted(ancilla_qubits)
    keep_indices = range(len(qubits))
    num_qubits = len(qubits)
    psi_circuit_full = sampler.simulate(circuit, qubit_order=ordered_qubits).final_state_vector
    psi_circuit = cirq.sub_state_vector(
        state_vector=psi_circuit_full,
        keep_indices=keep_indices
    ) if len(all_qubits) > len(qubits) else psi_circuit_full
    assert np.isclose(np.abs(psi_circuit @ create_dicke_statevector(num_qubits, num_excitations)) ** 2, 1.0)

### Requirements:
* The resulting circuit is allowed to contain only
  * CZ gates between pairs of qubits included in `cz_pauli_error_rate`, which will be a subset (and most) of the connected pairs in the [spec sheet](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf).
  * Arbitrary single-qubit unitaries (`cirq.PhasedXZGate`) on qubits included in `sq_pauli_error_rate`, which will be most of the qubits in the spec sheet.
  * Optionally, mid-circuit reset gates.
* Classically-conditioned gates are not supported.
* In the absence of noise, the circuit must either produce perfect Dicke states of the requested $N$ and $n$ or throw a `NotImplementedError`.
* If your circuit construction code takes longer than a minute to return a circuit, that is counted the same as returning a `NotImplementedError`.
* Any circuits containing more than 1750 moments will not be run.

# Scoring

We prepare the Dicke state on actual quantum hardware and measure it in both the Z basis and the X basis. We use the following `overall_score` function to compare against the expected distributions.

In [4]:
def check_z_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int
) -> float:
    """Measure how close the observed Z-basis distribution of bitstrings is to
    the ideal one. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots Z-basis bitstrings.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how close the distribution is to the ideal one.
    """
    # first, let's see how many bitstrings have one excitation
    n_shots, n_qubits = bitstrings.shape
    measured_excitations = np.sum(bitstrings, axis=1)
    fraction_in_subspace = np.mean(measured_excitations == num_excitations)

    # now, let's check how uniform the distribution is over states with n excitations
    chi2 = 0
    ideal_prob = 1 / math.comb(n_qubits, num_excitations)
    for indices in itertools.combinations(range(n_qubits), num_excitations):
        ideal_state = np.zeros(n_qubits)
        ideal_state[np.array(indices)] = 1
        match_count = np.sum(np.all(bitstrings == ideal_state, axis=1))
        expected_count = n_shots * fraction_in_subspace * ideal_prob
        chi2 += (match_count - expected_count) ** 2 / expected_count

    return ideal_prob * chi2 / fraction_in_subspace


def get_correlator_average(bitstrings: np.ndarray, correlator_size: int) -> float:
    """Get the average of all weight-correlator_size correlators in `bitstrings`.

    Args:
        bitstrings: The input bitstrings, shape (n_shots, n_qubits).
        correlator_size: Compute all correlators of this weight.

    Returns:
        The average of all weight-k correlators.
    """
    correlator_sum = 0.0
    n_shots, n_qubits = bitstrings.shape
    for indices in itertools.combinations(range(n_qubits), correlator_size):
        correlator_sum += 1 - 2 * np.mean(np.sum(bitstrings[:, indices], axis=1) % 2)
    return correlator_sum / math.comb(n_qubits, correlator_size)


def check_x_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int, max_correlator_size: int
) -> float:
    """Check expected correlations in the X basis. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots X-basis bitstrings.
        max_correlator_size: The maximum weight of correlators to consider.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how the correlations in the X-basis measurements compare to expectation.
    """
    comb = lambda n, k: math.comb(n, k) if (k >= 0 and n >= k) else 0
    n_shots, n_qubits = bitstrings.shape
    score = 0.0
    for correlator_size in range(1, max_correlator_size + 1):
        expected_correlator = (
            comb(correlator_size, correlator_size // 2)
            * comb(n_qubits - correlator_size, num_excitations - correlator_size // 2)
            / comb(n_qubits, num_excitations)
            if correlator_size % 2 == 0
            else 0.0
        )
        measured_correlator = get_correlator_average(bitstrings, correlator_size)
        score += (
            (expected_correlator - measured_correlator) ** 2 / expected_correlator**2
            if expected_correlator != 0
            else (expected_correlator - measured_correlator) ** 2
        )

    return score


def overall_score(
    z_bitstrings: np.ndarray,
    x_bitstrings: np.ndarray,
    num_excitations: int,
    max_correlator_size: int,
) -> float:
    """An overall score that checks how the Z and X basis bitstrings compare to their ideal
    distributions. The higher the score, the better.

    Args:
        z_bitstrings: The bitstrings measured in the Z basis
        x_bitstrings: The bitstrings measured in the X basis
        num_excitations: The weight, n, of the Dicke state.
        max_correlator_size: The maximum weight of correlators to consider

    Returns:
        A measure of how similar the bitstrings are to their ideal distributions.
    """
    return 1 / (
        check_z_bitstring_distribution(z_bitstrings, num_excitations)
        + check_x_bitstring_distribution(
            x_bitstrings, num_excitations, max_correlator_size
        )
    )


### Judging criteria:
* We will run circuits generated by your code for various choices of `num_qubits` and `num_excitations` on real quantum hardware, measuring in the Z basis as well as in the X basis, each for 10,000 shots.
* For each choice of `num_qubits` and `num_excitations`, we use the `overall_score` function to score the performance. We then sum this over the choices of `num_qubits`.
* The solution with the highest score wins.

### Things to consider:
* It is permitted to use AI to answer these questions, but please include any prompts you gave to the AI in your submission.
* This will be run on real quantum hardware, so techniques like dynamical decoupling are important.


You may use the following functions to generate fake error rates to test your algorithm.

In [5]:
def pick_single_qubit_dropouts(
    dropout_rate: float,
    rng: np.random.Generator = np.random.default_rng()
) -> set[cirq.GridQubit]:
    """Pick some qubits to be broken.

    Args:
        dropout_rate: The probability that a qubit is broken.
        rng: The pseudo-random number generator

    Returns:
        The broken qubits.
    """
    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_dropout = rng.random(len(all_qubits)) < dropout_rate
    return set(all_qubits[is_dropout])


def generate_qubit_error_rates(
    median: float,
    spread: float,
    dropouts: set[cirq.GridQubit],
    uncertainty: float,
    rng: np.random.Generator = np.random.default_rng(),
) -> dict[cirq.GridQubit, tuple[float, float]]:
    """Generate fake single-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropouts: Qubits to exclude
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """

    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_working = np.array([q not in dropouts for q in all_qubits])
    working_qubits = all_qubits[is_working]
    error_rate = rng.lognormal(np.log(median), spread, len(working_qubits))
    return dict(
        zip(working_qubits, zip(error_rate, uncertainty * np.ones(len(working_qubits))))
    )


def generate_two_qubit_pauli_error_rates(
    median: float = 2.5e-3,
    spread: float = 0.5,
    dropout_rate: float = 0.01,
    uncertainty: float = 5e-4,
    rng: np.random.Generator = np.random.default_rng(),
    single_qubit_dropouts: set[cirq.GridQubit] = None,
) -> dict[tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]]:
    """Generate fake two-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropout_rate: The probability of a broken coupler.
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator
        single_qubit_dropouts: The single-qubit dropouts to skip.

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """
    if single_qubit_dropouts is None:
        single_qubit_dropouts = set()

    all_pairs = np.array(list(cirq_google.Willow105.metadata.qubit_pairs))
    no_qubit_dropouts = np.array(
        [np.all([q not in single_qubit_dropouts for q in pair]) for pair in all_pairs]
    )
    is_working = (rng.random(len(all_pairs)) > dropout_rate) * no_qubit_dropouts
    working_pairs = [tuple(pair) for pair in all_pairs[is_working]]
    error_rate = rng.lognormal(np.log(median), spread, len(working_pairs))
    return dict(
        zip(working_pairs, zip(error_rate, uncertainty * np.ones(len(working_pairs))))
    )


In [6]:
single_qubit_dropouts = pick_single_qubit_dropouts(0.01)
sq_pauli_error_rate = generate_qubit_error_rates(4e-4, 0.5, single_qubit_dropouts, 2e-4)
cz_pauli_error_rate = generate_two_qubit_pauli_error_rates(
    single_qubit_dropouts=single_qubit_dropouts
)
zero_errors = generate_qubit_error_rates(5e-3, 0.5, single_qubit_dropouts, 2e-4)
one_errors = generate_qubit_error_rates(1e-2, 0.5, single_qubit_dropouts, 3e-4)

In [7]:
# Self-check on the fake calibration above: exact state, rule checks, and build time.
import time
for N, n in [(3, 1), (4, 2), (6, 3), (8, 2)]:
    start = time.time()
    circuit, qubits = create_dicke_state(N, n, cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors)
    test_circuit(circuit, n, qubits)
    cirq_google.Willow105.validate_circuit(circuit)
    print(f"D({N},{n}): ok, {len(circuit)} moments, "
          f"{sum(len(op.qubits) == 2 for op in circuit.all_operations())} CZ, {time.time() - start:.1f} s")


D(3,1): ok, 9 moments, 4 CZ, 0.2 s
D(4,2): ok, 7 moments, 5 CZ, 0.1 s
D(6,3): ok, 17 moments, 15 CZ, 0.1 s


D(8,2): ok, 23 moments, 26 CZ, 0.2 s


# Bonus question:
Now suppose we allow for adaptive circuits, i.e. those with mid-circuit measurements and subsequent gates conditioned on the outcomes of those measurements. See if you can construct shallower circuits to prepare Dicke states given this capability. We will not run these circuits on the hardware.

For all $(N, n)$, with $2 \leq N \leq 15$, and $1 \leq n \leq \lfloor N/2 \rfloor$, we will compare the depth of the circuits produced by the various submissions and assign a point to the one with the shallowest depth, measured by the number of moments in the circuit.

Specifically, your task is to complete the following function:

In [8]:
def create_dicke_state_adaptive(
    num_qubits: int, num_excitations: int
) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
    """Reuse the main (non-adaptive) construction on Willow105 with uniform error rates.

    The bonus allows reusing the main solution. Every construction above keeps its
    echo pulses inside existing moments, so they add no depth.
    """
    device = cirq_google.Willow105
    qubits = set(device.metadata.qubit_set)
    pairs = {tuple(pair) for pair in device.metadata.qubit_pairs}
    return create_dicke_state(
        num_qubits,
        num_excitations,
        {pair: (2.5e-3, 0.0) for pair in pairs},
        {q: (4e-4, 0.0) for q in qubits},
        {q: (5e-3, 0.0) for q in qubits},
        {q: (1e-2, 0.0) for q in qubits},
    )


## Scoring

We will use the following function to score submissions:

In [9]:
def score_solution() -> float:
    score = 0
    for num_qubits in range(2, 16):
        for num_excitations in range(1, num_qubits//2+1):
            try:
                circuit, qubits = create_dicke_state_adaptive(num_qubits, num_excitations)
            except NotImplementedError:
                continue
            validate_circuit(circuit, num_excitations, qubits) # checks that the circuit is valid
            score += num_qubits * num_excitations / len(circuit)
    return score

Notes and constraints:
* We will verify classically that the circuits produce the desired Dicke state up to a global phase.
* Using adaptive circuits is not required, so you may reuse your solution from the previous question.
* Although we do not provide error rates for this question, it is still necessary to respect the connectivity of the [Willow device](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf), i.e. you can only use qubits on that device and CZ gates between adjacent qubits on the device. You may use `cirq_google.Willow105.validate_circuit(circuit)` to validate that your circuits are correctly routed to such a device.
* On real hardware, qubits along a diagonal such that row # + column # = const share a readout line and measuring one qubit dephases the other qubits on that readout line. Therefore, mid-circuit measurements of some but not all of the qubits on a readout line are not allowed. (This constraint is not checked by `cirq_google.Willow105.validate_circuit(circuit)`.)
* As before, the code must execute in under a minute for each $(N, n)$.

# AI usage and prompts

The challenge rules ask for every prompt given to an AI. This solution was developed with Claude (Anthropic). The full conversation, including all prompts, is attached as `AI_prompts.md` with this submission.

<!-- TODO before final submission: export the Claude conversation and attach it. -->
